# Modelo Monte Carlo

## modelo 1

In [18]:
"""
=============================================================================
DESAFIO ESTRATÉGICO L.E.K. CONSULTING 2026 – FASE 1
Monte Carlo: Previsão de Demanda e Preço das 5 Culturas Agrícolas
=============================================================================
Culturas analisadas (basezona.xlsx – USDA PSD):
  Milho          → Corn
  Cana-de-açúcar → Sugar, Centrifugal
  Café (Arábica) → Coffee, Green
  Citros (OJ)    → Orange Juice
  Trigo          → Wheat

PREMISSAS EXTERNAS (⚠️):
  Todas as premissas não retiradas da base estão marcadas com
  "# ⚠️ PREMISSA EXTERNA" e agrupadas na Seção 1 para facilitar
  substituição futura por dados reais.

OUTPUTS GERADOS (abertos automaticamente ao final):
  fig1_demand_mc.png         – Fan charts demanda (GBM)
  fig2_price_mc.png          – Fan charts preço (Ornstein-Uhlenbeck)
  fig3_dashboard.png         – Dashboard estratégico completo
  fig4_top2_distribution.png – Histogramas do potencial de mercado (top 2)
  scoring_priorizacao.csv    – Scoring multicritério detalhado
  projecoes_mc.csv           – Projeções P10/Mediana/P90 por ano
  resumo_mc.json             – Parâmetros e resultado final
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")

import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # renderiza sem display – troca por "TkAgg" se quiser janela
import matplotlib.pyplot as plt
from pathlib import Path
import json

# ──────────────────────────────────────────────────────────────────────────────
# SECAO 0 - CONFIGURACOES GERAIS (sem premissas externas)
# ──────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

# ⚠️ PREMISSA EXTERNA [P-01] – Número de simulações
# Razão: padrão da literatura para Monte Carlo de commodities (10k–100k).
# Trocar por valor maior (ex: 50_000) para publicação formal; menor para testes.
N_SIMUL = 10_000

# ⚠️ PREMISSA EXTERNA [P-02] – Horizonte de projeção (anos)
# Razão: alinhado ao "médio/longo prazo" do desafio (módulo 1C).
# Altere para 3 (curto prazo) ou 10 (longo prazo) conforme necessário.
HORIZON = 5   # 2025-2029

# Último ano com dados históricos completos na base
PROJ_YEAR = 2023   # basezona.xlsx vai até 2023/2024; usamos 2023 como ancora

# Apontando para o diretório exato do desafio
INPUT_FILE  = r"C:\Users\lucas\Documents\Poli junior computador\Challenge LEK\Dados-Desafio-LEK\basezona.xlsx"
OUTPUT_DIR  = Path(r"C:\Users\lucas\Documents\Poli junior computador\Challenge LEK\Dados-Desafio-LEK\resultados monte carlo")

# Cria a pasta "resultados monte carlo" se ela ainda não existir
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Mapeamento nome USDA PSD → rótulo do desafio
CROP_MAP = {
    "Corn"              : "Milho",
    "Coffee, Green"     : "Cafe (Arabica)",
    "Sugar, Centrifugal": "Cana-de-acucar",
    "Orange Juice"      : "Citros (OJ)",
    "Wheat"             : "Trigo",
}

# Unidades de exibição (informativas, não afetam o modelo)
UNITS = {
    "Milho"          : {"demand": "1.000 MT",     "price": "US$/bu"},
    "Cafe (Arabica)" : {"demand": "M sacos 60kg", "price": "US$/lb"},
    "Cana-de-acucar" : {"demand": "1.000 MT",     "price": "US$/lb"},
    "Citros (OJ)"    : {"demand": "1.000 MT",     "price": "US$/lb"},
    "Trigo"          : {"demand": "1.000 MT",     "price": "US$/bu"},
}

# Cores L.E.K.
LEK_GREEN = "#00A651"
LEK_DKGRN = "#004D2C"
LEK_LGRAY  = "#D0D0D0"
LEK_GRAY   = "#555555"
BG_COLOR   = "#FAFAFA"
PALETTE    = ["#00A651", "#007D3F", "#33BF77", "#005C23", "#99DFB9"]


# ══════════════════════════════════════════════════════════════════════════════
# SECAO 1 – PREMISSAS EXTERNAS (TODAS AGRUPADAS AQUI)
# ══════════════════════════════════════════════════════════════════════════════
# Cada bloco indica: código da premissa, o que é, por que foi assumida,
# e como substituir por dado real.

# ─────────────────────────────────────────────────────────────────────────────
# ⚠️ PREMISSA EXTERNA [P-03] – Preços históricos anuais médios
# ─────────────────────────────────────────────────────────────────────────────
# O que   : Preços médios anuais de cada commodity (2005-2024).
# Por que : A basezona.xlsx contém apenas VOLUMES (USDA PSD). Preços não
#            estão na base fornecida.
# Fonte assumida: USDA ERS, ICE Futures, CBOT (compilação pública).
# Como substituir: Carregue um CSV/Excel com colunas
#   [Ano, Milho_USD_bu, Cafe_USD_lb, Acucar_USD_lb, Citros_USD_lb, Trigo_USD_bu]
#   e substitua o dicionário abaixo por:
#     df_p = pd.read_csv("precos_reais.csv", index_col="Ano")
#     PRICES_HIST_EXTERNAL = df_p.to_dict("index")
#
# Formato: Ano → (Milho US$/bu, Cafe US$/lb, Acucar US$/lb, Citros US$/lb, Trigo US$/bu)
PRICES_HIST_EXTERNAL = {
    2005: (2.00, 1.13, 0.098, 0.74, 3.42),
    2006: (3.04, 1.06, 0.122, 0.85, 3.56),
    2007: (3.40, 1.10, 0.098, 0.80, 4.26),
    2008: (5.74, 1.22, 0.120, 0.80, 7.57),
    2009: (3.55, 1.13, 0.189, 0.96, 5.00),
    2010: (3.69, 1.48, 0.228, 1.01, 5.70),
    2011: (6.01, 2.52, 0.264, 1.35, 7.77),
    2012: (6.98, 1.82, 0.196, 1.23, 7.77),
    2013: (4.62, 1.30, 0.170, 1.28, 6.87),
    2014: (3.61, 1.94, 0.155, 1.60, 5.99),
    2015: (3.61, 1.23, 0.120, 1.05, 4.89),
    2016: (3.36, 1.44, 0.188, 1.20, 4.64),
    2017: (3.36, 1.32, 0.148, 1.44, 4.72),
    2018: (3.61, 1.09, 0.125, 1.50, 5.16),
    2019: (3.56, 1.00, 0.126, 1.02, 4.65),
    2020: (3.56, 1.23, 0.128, 1.06, 5.05),
    2021: (5.63, 1.93, 0.175, 1.26, 7.77),
    2022: (6.70, 2.26, 0.195, 1.57, 8.86),
    2023: (4.82, 1.80, 0.232, 1.44, 6.08),
    2024: (4.35, 3.20, 0.218, 1.50, 5.75),
}
PRICE_COLS_ORDER = ["Milho", "Cafe (Arabica)", "Cana-de-acucar", "Citros (OJ)", "Trigo"]

# ─────────────────────────────────────────────────────────────────────────────
# ⚠️ PREMISSA EXTERNA [P-04] – Velocidade de reversão à média (kappa) – modelo OU
# ─────────────────────────────────────────────────────────────────────────────
# O que   : Parâmetro kappa do processo Ornstein-Uhlenbeck para preços.
#            kappa = 0 → caminho aleatório puro (GBM); kappa alto → reversão rápida.
# Por que : Commodities têm comportamento mean-reverting empírico.
#            kappa = 0.20 é valor central da literatura para graos/softs (Schwartz 1997).
# Como substituir: Estime kappa via regressão OLS de D(ln P) ~ b0 + b1*ln(P_{t-1})
#            onde kappa = -b1. Valores típicos: grãos 0.15-0.30, café 0.10-0.25.
KAPPA_OU_EXTERNAL = {
    "Milho"          : 0.20,   # grao, ciclo anual
    "Cafe (Arabica)" : 0.15,   # ciclo bienal, menos reversão
    "Cana-de-acucar" : 0.20,
    "Citros (OJ)"    : 0.25,   # estoque baixo, reversão mais rápida
    "Trigo"          : 0.20,
}

# ─────────────────────────────────────────────────────────────────────────────
# ⚠️ PREMISSA EXTERNA [P-05] – Pesos do scoring multicritério
# ─────────────────────────────────────────────────────────────────────────────
# O que   : Pesos de cada critério na nota final ponderada (soma = 1.0).
# Por que : Refletem a visão de um investidor estratégico genérico conforme
#            descrito no desafio (módulo 1D). Não há dados na base para calibrá-los.
# Como substituir: Discuta com a equipe e ajuste os pesos abaixo.
#            Critérios possíveis de adicionar: risco cambial, barreiras regulatórias,
#            capex de entrada, ciclo de maturação.
SCORING_WEIGHTS_EXTERNAL = {
    "tamanho_mercado"     : 0.25,  # potencial absoluto (Demanda x Preco)
    "crescimento_demanda" : 0.20,  # CAGR histórico da demanda mundial
    "posicao_brasil"      : 0.20,  # share do Brasil na producao mundial
    "tendencia_preco"     : 0.15,  # slope relativo de preco (positivo = atrativo)
    "volatilidade_preco"  : 0.10,  # menor volatilidade = menor risco
    "produtividade_brasil": 0.10,  # crescimento de yield do Brasil
}

# ─────────────────────────────────────────────────────────────────────────────
# ⚠️ PREMISSA EXTERNA [P-06] – Fallback de parâmetros GBM quando série < 5 obs
# ─────────────────────────────────────────────────────────────────────────────
# O que   : mu e sigma padrão usados se não há dados históricos suficientes.
# Por que : Evita crash em culturas com séries curtas; valores conservadores.
# Como substituir: Ajuste por cultura com base em literatura ou estimativa do time.
FALLBACK_MU_EXTERNAL    = 0.02   # crescimento anual de 2%
FALLBACK_SIGMA_EXTERNAL = 0.05   # volatilidade de 5%

# ─────────────────────────────────────────────────────────────────────────────
# ⚠️ PREMISSA EXTERNA [P-07] – Corte do histórico para calibração
# ─────────────────────────────────────────────────────────────────────────────
# O que   : Usamos dados a partir de 2005 para calibrar os modelos,
#            descartando 2000-2004.
# Por que : O período 2000-2004 tem valores inconsistentes na base
#            (ex: Citros com producao "978" vs "1.354.000" no ano seguinte –
#            possível erro de digitação no USDA PSD original).
#            Pós-2005 os mercados de commodities têm melhor integração global.
# Como substituir: Altere HIST_START_YEAR_EXTERNAL para 2000 para série mais longa.
HIST_START_YEAR_EXTERNAL = 2005


# ══════════════════════════════════════════════════════════════════════════════
# SECAO 2 – TRATAMENTO DA BASE (dados retirados de basezona.xlsx)
# ══════════════════════════════════════════════════════════════════════════════

def load_and_clean_base(path: str):
    df_raw = pd.read_excel(path, sheet_name="psd (3)")
    year_cols = [c for c in df_raw.columns if "/" in str(c)]

    for col in year_cols:
        df_raw[col] = (
            df_raw[col].astype(str)
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

    # Converte "YYYY/YYYY" → inteiro (primeiro ano do ciclo agrícola)
    year_map = {c: int(c.split("/")[0]) for c in year_cols}
    df_raw = df_raw.rename(columns=year_map)
    years = sorted(year_map.values())

    data = {}
    for commodity in df_raw["Commodity"].unique():
        data[commodity] = {}
        sub = df_raw[df_raw["Commodity"] == commodity]
        for country in sub["Country"].unique():
            sub2 = sub[sub["Country"] == country].copy()
            sub2 = sub2.set_index("Attribute")[years]
            data[commodity][country] = sub2

    return data, years


def get_series(data, commodity, country, attribute, years):
    try:
        s = data[commodity][country].loc[attribute, years]
        return pd.to_numeric(s, errors="coerce")
    except KeyError:
        return pd.Series(dtype=float)


def get_world_demand(data, commodity, years):
    # Para Cana-de-acucar a base usa "Total Disappearance" como proxy de demanda
    attr = "Total Disappearance" if commodity == "Sugar, Centrifugal" else "Domestic Consumption"
    return get_series(data, commodity, "World", attr, years)


def get_brazil_production(data, commodity, years):
    return get_series(data, commodity, "Brazil", "Production", years)


def get_brazil_yield(data, commodity, years):
    return get_series(data, commodity, "Brazil", "Yield", years)


def get_brazil_share(data, commodity, years):
    br  = get_brazil_production(data, commodity, years)
    wld = get_series(data, commodity, "World", "Production", years)
    return (br / wld * 100).replace([np.inf, -np.inf], np.nan)


def build_price_series(crop_lbl):
    """Monta série de preços a partir das premissas externas [P-03]."""
    idx = PRICE_COLS_ORDER.index(crop_lbl)
    return pd.Series(
        {yr: vals[idx] for yr, vals in PRICES_HIST_EXTERNAL.items()},
        name=crop_lbl,
    )


# ══════════════════════════════════════════════════════════════════════════════
# SECAO 3 – MODELOS DE SIMULACAO
# ══════════════════════════════════════════════════════════════════════════════

def fit_gbm(series: pd.Series):
    """
    Calibra GBM pela média e desvio-padrão dos log-retornos históricos.
    mu e sigma são DERIVADOS DA BASE (Secao 2), não premissas externas.
    Exceção: fallback [P-06] quando há menos de 5 observações.
    """
    s = series.dropna()
    if len(s) < 5:
        return FALLBACK_MU_EXTERNAL, FALLBACK_SIGMA_EXTERNAL  # ⚠️ [P-06]
    lr = np.log(s / s.shift(1)).dropna()
    return float(lr.mean()), float(lr.std())


def monte_carlo_demand(series: pd.Series, n_sim=N_SIMUL, horizon=HORIZON):
    """
    GBM (Geometric Brownian Motion) para demanda.
    S(t+1) = S(t) * exp[(mu - 0.5*sigma^2)*dt + sigma*sqrt(dt)*Z]

    mu e sigma são calibrados na base histórica (USDA PSD) – sem premissa externa.
    """
    mu, sigma = fit_gbm(series.dropna())
    last      = float(series.dropna().iloc[-1])
    dt        = 1.0
    Z         = np.random.normal(0, 1, (n_sim, horizon))
    log_ret   = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    paths     = last * np.exp(np.cumsum(log_ret, axis=1))
    return paths, mu, sigma


def monte_carlo_price(crop_lbl: str, n_sim=N_SIMUL, horizon=HORIZON):
    """
    Ornstein-Uhlenbeck (mean-reverting) para preço de commodity.
    DP = kappa*(theta - P)*dt + sigma*P*sqrt(dt)*Z

    mu e sigma calibrados nos preços históricos [P-03].
    kappa (velocidade reversão) é premissa externa [P-04].
    theta (long-run mean) calculado como média log-normal histórica dos preços [P-03].
    """
    ps      = build_price_series(crop_lbl)   # ⚠️ usa dados [P-03]
    lr      = np.log(ps / ps.shift(1)).dropna()
    mu_p    = float(lr.mean())               # ⚠️ derivado de [P-03]
    sigma_p = float(lr.std())               # ⚠️ derivado de [P-03]
    theta   = float(np.exp(np.log(ps).mean()))  # ⚠️ derivado de [P-03]

    kappa = KAPPA_OU_EXTERNAL[crop_lbl]     # ⚠️ PREMISSA EXTERNA [P-04]

    dt     = 1.0
    prices = np.full(n_sim, float(ps.iloc[-1]))   # ⚠️ valor inicial de [P-03]
    paths  = np.zeros((n_sim, horizon))
    for t in range(horizon):
        Z      = np.random.normal(0, 1, n_sim)
        prices = prices + kappa * (theta - prices) * dt + sigma_p * prices * np.sqrt(dt) * Z
        prices = np.clip(prices, 0.01, None)
        paths[:, t] = prices

    return paths, mu_p, sigma_p, theta


def calc_market_potential(demand_paths, price_paths):
    """Potencial = Demanda x Preco. Estatísticas por ano de projeção."""
    market = demand_paths * price_paths
    rows = []
    for t in range(demand_paths.shape[1]):
        a = market[:, t]
        rows.append({
            "year"  : PROJ_YEAR + t + 1,
            "mean"  : np.mean(a),
            "median": np.median(a),
            "p10"   : np.percentile(a, 10),
            "p90"   : np.percentile(a, 90),
            "std"   : np.std(a),
        })
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# SECAO 4 – SCORING MULTICRITERIO (módulo 1D do desafio)
# ══════════════════════════════════════════════════════════════════════════════

def normalize_minmax(raw: dict, invert=False):
    """Escala 1-10 por min-max. invert=True: menor valor = maior score."""
    vals = np.array(list(raw.values()), dtype=float)
    v_min, v_max = vals.min(), vals.max()
    if v_max == v_min:
        return {k: 5.0 for k in raw}
    if invert:
        return {k: 1 + 9 * (v_max - v) / (v_max - v_min) for k, v in raw.items()}
    return {k: 1 + 9 * (v - v_min) / (v_max - v_min) for k, v in raw.items()}


def build_scoring(data, years_hist, mc_results):
    crops = list(CROP_MAP.values())
    raw = {c: {} for c in SCORING_WEIGHTS_EXTERNAL}

    # Critério 1 – Tamanho de mercado: mediana do potencial no último ano projetado
    # (calculado inteiramente da base + simulacao)
    for lbl in crops:
        raw["tamanho_mercado"][lbl] = mc_results[lbl]["market_potential"]["median"].iloc[-1]

    # Critério 2 – Crescimento demanda: mu GBM (calibrado na base)
    for raw_name, lbl in CROP_MAP.items():
        s = get_world_demand(data, raw_name, years_hist).dropna()
        mu, _ = fit_gbm(s)
        raw["crescimento_demanda"][lbl] = mu

    # Critério 3 – Posicao do Brasil: share médio dos últimos 5 anos (base)
    for raw_name, lbl in CROP_MAP.items():
        share = get_brazil_share(data, raw_name, years_hist).dropna()
        raw["posicao_brasil"][lbl] = float(share.iloc[-5:].mean()) if len(share) >= 5 else 0.0

    # Critério 4 – Tendência de preco: slope linear relativo
    # (calculado nos dados externos [P-03] – portanto depende de [P-03])
    for lbl in crops:
        ps = build_price_series(lbl)   # ⚠️ usa [P-03]
        x  = np.arange(len(ps))
        slope, _ = np.polyfit(x, ps.values, 1)
        raw["tendencia_preco"][lbl] = slope / float(ps.mean())

    # Critério 5 – Volatilidade preco: sigma dos log-retornos
    # (menor volatilidade = menor risco = score maior, por isso invert=True)
    # ⚠️ depende de [P-03]
    for lbl in crops:
        ps = build_price_series(lbl)   # ⚠️ usa [P-03]
        raw["volatilidade_preco"][lbl] = float(np.log(ps / ps.shift(1)).dropna().std())

    # Critério 6 – Produtividade Brasil: slope relativo do yield (base USDA PSD)
    for raw_name, lbl in CROP_MAP.items():
        yld = get_brazil_yield(data, raw_name, years_hist).dropna()
        if len(yld) >= 5:
            x = np.arange(len(yld))
            slope, _ = np.polyfit(x, yld.values, 1)
            raw["produtividade_brasil"][lbl] = slope / float(yld.mean())
        else:
            raw["produtividade_brasil"][lbl] = 0.0

    # Normalização
    norm = {
        "tamanho_mercado"     : normalize_minmax(raw["tamanho_mercado"]),
        "crescimento_demanda" : normalize_minmax(raw["crescimento_demanda"]),
        "posicao_brasil"      : normalize_minmax(raw["posicao_brasil"]),
        "tendencia_preco"     : normalize_minmax(raw["tendencia_preco"]),
        "volatilidade_preco"  : normalize_minmax(raw["volatilidade_preco"], invert=True),
        "produtividade_brasil": normalize_minmax(raw["produtividade_brasil"]),
    }

    records = []
    for lbl in crops:
        # ⚠️ PREMISSA EXTERNA [P-05] – pesos do scoring
        score = sum(SCORING_WEIGHTS_EXTERNAL[c] * norm[c].get(lbl, 5.0)
                    for c in SCORING_WEIGHTS_EXTERNAL)
        row = {"Cultura": lbl}
        for c in SCORING_WEIGHTS_EXTERNAL:
            row[c] = round(norm[c].get(lbl, 5.0), 2)
            row[f"raw_{c}"] = round(raw[c].get(lbl, 0.0), 4)
        row["Score Final"] = round(score, 2)
        records.append(row)

    df = (pd.DataFrame(records)
          .sort_values("Score Final", ascending=False)
          .reset_index(drop=True))
    df["Ranking"] = df.index + 1
    return df


# ══════════════════════════════════════════════════════════════════════════════
# SECAO 5 – VISUALIZACOES
# ══════════════════════════════════════════════════════════════════════════════

def _fan(ax, hist_s, paths, color):
    proj_years = list(range(PROJ_YEAR + 1, PROJ_YEAR + HORIZON + 1))
    ax.plot(list(hist_s.index), hist_s.values, color=LEK_DKGRN, lw=2, label="Historico")
    ax.fill_between(proj_years,
                    np.percentile(paths, 10, axis=0),
                    np.percentile(paths, 90, axis=0),
                    alpha=0.18, color=color, label="P10-P90")
    ax.fill_between(proj_years,
                    np.percentile(paths, 25, axis=0),
                    np.percentile(paths, 75, axis=0),
                    alpha=0.35, color=color, label="P25-P75")
    ax.plot(proj_years, np.median(paths, axis=0),
            color=color, lw=2.5, ls="--", label="Mediana proj.")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_facecolor(BG_COLOR)


def fig1_demand(mc_results, save_path):
    """Fan charts de demanda mundial (GBM)."""
    crops = list(mc_results.keys())
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.patch.set_facecolor(BG_COLOR)
    fig.suptitle(
        f"Monte Carlo – Demanda Mundial por Cultura  |  {N_SIMUL:,} simulacoes  |  "   # ⚠️ N_SIMUL = [P-01]
        f"Horizonte: {PROJ_YEAR+1}-{PROJ_YEAR+HORIZON}  |  Modelo: GBM (mu/sigma da base USDA PSD)",
        fontsize=12, fontweight="bold", color=LEK_DKGRN
    )
    for i, lbl in enumerate(crops):
        ax = axes[i // 3][i % 3]
        res = mc_results[lbl]
        _fan(ax, res["demand_hist"], res["demand_paths"], LEK_GREEN)
        mu_d = res["mu_demand"]
        sigma_d = res["sigma_demand"]
        ax.set_title(
            f"{lbl}  |  {UNITS[lbl]['demand']}\n"
            f"mu={mu_d*100:+.1f}%  sigma={sigma_d*100:.1f}%  (da base)",
            fontsize=9, fontweight="bold", color=LEK_DKGRN
        )
        ax.set_xlabel("Ano", fontsize=9)
        if i == 0:
            ax.legend(fontsize=7, framealpha=0.5)
    axes[1][2].set_visible(False)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
    plt.close(fig)
    print(f"      Salvo: {save_path}")


def fig2_price(mc_results, save_path):
    """Fan charts de preco (Ornstein-Uhlenbeck)."""
    crops = list(mc_results.keys())
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.patch.set_facecolor(BG_COLOR)
    fig.suptitle(
        "Monte Carlo – Preco por Cultura  |  Modelo Ornstein-Uhlenbeck (mean reversion)\n"
        "⚠️ Precos historicos = [P-03] (externos)  |  ⚠️ kappa (velocidade reversao) = [P-04]",
        fontsize=11, fontweight="bold", color=LEK_DKGRN
    )
    for i, lbl in enumerate(crops):
        ax = axes[i // 3][i % 3]
        res = mc_results[lbl]
        _fan(ax, res["price_hist"], res["price_paths"], "#1565C0")
        theta = res["long_run_price"]
        kappa = KAPPA_OU_EXTERNAL[lbl]   # ⚠️ [P-04]
        ax.axhline(theta, color=LEK_GRAY, lw=1, ls=":", alpha=0.8,
                   label=f"theta={theta:.2f} ⚠️[P-03]")
        ax.set_title(
            f"{lbl}  |  {UNITS[lbl]['price']}\n"
            f"⚠️ kappa={kappa:.2f} [P-04]  |  theta={theta:.2f} [P-03]",
            fontsize=9, fontweight="bold", color=LEK_DKGRN
        )
        ax.set_xlabel("Ano", fontsize=9)
        if i == 0:
            ax.legend(fontsize=7, framealpha=0.5)
    axes[1][2].set_visible(False)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
    plt.close(fig)
    print(f"      Salvo: {save_path}")


def fig3_dashboard(data, years_hist, mc_results, df_score, save_path):
    """Dashboard com 4 paineis: scoring, potencial, share Brasil, tabela MC."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.patch.set_facecolor(BG_COLOR)
    fig.suptitle("L.E.K. Consulting 2026 – Dashboard de Analise Estrategica",
                 fontsize=15, fontweight="bold", color=LEK_DKGRN)

    # ── Painel A: Scoring ─────────────────────────────────────────────────
    ax = axes[0][0]
    colors = [LEK_GREEN if r <= 2 else LEK_LGRAY for r in df_score["Ranking"]]
    ax.barh(df_score["Cultura"], df_score["Score Final"], color=colors, edgecolor="white")
    ax.set_xlim(0, 10)
    ax.set_xlabel("Score ponderado (0-10)", fontsize=9)
    ax.set_title(
        "Priorizacao Estrategica\n"
        "⚠️ Pesos = premissa externa [P-05]  |  ⚠️ Criterios 4/5 usam precos [P-03]",
        fontsize=9, fontweight="bold", color=LEK_DKGRN
    )
    ax.invert_yaxis()
    for bar, val in zip(ax.patches, df_score["Score Final"]):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
                f"{val:.2f}", va="center", fontsize=9, color=LEK_GRAY)
    ax.text(0.98, 0.04, "Top 2 priorizadas", transform=ax.transAxes,
            ha="right", color=LEK_GREEN, fontsize=9, fontstyle="italic")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_facecolor(BG_COLOR)

    # ── Painel B: Potencial de mercado ───────────────────────────────────
    ax = axes[0][1]
    crops_order = df_score["Cultura"].tolist()
    meds = [mc_results[lbl]["market_potential"]["median"].iloc[-1] for lbl in crops_order]
    p10s = [mc_results[lbl]["market_potential"]["p10"].iloc[-1]    for lbl in crops_order]
    p90s = [mc_results[lbl]["market_potential"]["p90"].iloc[-1]    for lbl in crops_order]
    x = np.arange(len(crops_order))
    ax.bar(x, meds, color=LEK_GREEN, alpha=0.85, edgecolor="white")
    ax.errorbar(x, meds,
                yerr=[np.array(meds) - np.array(p10s), np.array(p90s) - np.array(meds)],
                fmt="none", color=LEK_DKGRN, capsize=5, lw=1.5)
    ax.set_xticks(x)
    ax.set_xticklabels(crops_order, rotation=20, ha="right", fontsize=8)
    ax.set_title(
        f"Potencial de Mercado – ano {PROJ_YEAR+HORIZON}  (mediana +/- P10/P90)\n"
        "Demanda (base USDA) x Preco  |  ⚠️ Preco depende de [P-03][P-04]",
        fontsize=9, fontweight="bold", color=LEK_DKGRN
    )
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_facecolor(BG_COLOR)

    # ── Painel C: Share do Brasil ────────────────────────────────────────
    ax = axes[1][0]
    for i, (raw_name, lbl) in enumerate(CROP_MAP.items()):
        share = get_brazil_share(data, raw_name, years_hist).dropna()
        ax.plot(share.index, share.values, lw=2, label=lbl,
                color=PALETTE[i % len(PALETTE)], marker="o", markersize=3)
    ax.set_title(
        "Share do Brasil na Producao Mundial (%)\n"
        "Fonte: USDA PSD (basezona.xlsx) – sem premissas externas",
        fontsize=9, fontweight="bold", color=LEK_DKGRN
    )
    ax.set_xlabel("Ano")
    ax.set_ylabel("%")
    ax.legend(fontsize=7, loc="upper left")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_facecolor(BG_COLOR)

    # ── Painel D: Tabela de parametros MC ───────────────────────────────
    ax = axes[1][1]
    ax.axis("off")
    tbl_rows = []
    for lbl in list(CROP_MAP.values()):
        res = mc_results[lbl]
        kap = KAPPA_OU_EXTERNAL[lbl]   # ⚠️ [P-04]
        tbl_rows.append([
            lbl,
            f"{res['mu_demand']*100:+.1f}%",       # da base
            f"{res['sigma_demand']*100:.1f}%",      # da base
            f"{res['mu_price']*100:+.1f}% ⚠️",     # derivado de [P-03]
            f"{res['sigma_price']*100:.1f}% ⚠️",   # derivado de [P-03]
            f"{kap:.2f} ⚠️",                         # [P-04]
        ])
    tbl = ax.table(
        cellText=tbl_rows,
        colLabels=["Cultura", "mu Dem.", "sig Dem.", "mu Preco", "sig Preco", "kappa"],
        loc="center", cellLoc="center"
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7.5)
    tbl.scale(1, 1.9)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor(LEK_DKGRN)
            cell.set_text_props(color="white", fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#E8F5E9")
        cell.set_edgecolor("#CCCCCC")
    ax.set_title(
        "Parametros Monte Carlo por Cultura\n"
        "mu/sig Dem. = calibrados na base USDA PSD (sem premissa)\n"
        "mu/sig Preco e kappa = colunas com ⚠️ = premissas externas [P-03][P-04]",
        fontsize=8, fontweight="bold", color=LEK_DKGRN, pad=14
    )

    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
    plt.close(fig)
    print(f"      Salvo: {save_path}")


def fig4_top2(mc_results, top2, save_path):
    """Histogramas do potencial de mercado para as 2 culturas priorizadas."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor(BG_COLOR)
    fig.suptitle(
        f"Distribuicao do Potencial de Mercado – Top 2 Culturas  |  ano {PROJ_YEAR+HORIZON}\n"
        f"({N_SIMUL:,} simulacoes  |  Demanda x Preco  |  ⚠️ componente Preco usa [P-03][P-04])",
        fontsize=12, fontweight="bold", color=LEK_DKGRN
    )
    for i, lbl in enumerate(top2):
        ax = axes[i]
        res = mc_results[lbl]
        final = res["demand_paths"][:, -1] * res["price_paths"][:, -1]
        mu_f  = np.mean(final)
        p10_f = np.percentile(final, 10)
        p90_f = np.percentile(final, 90)
        ax.hist(final, bins=80, color=PALETTE[i], edgecolor="white", alpha=0.85)
        ax.axvline(mu_f,  color=LEK_DKGRN, lw=2.5, label=f"Media: {mu_f:,.0f}")
        ax.axvline(p10_f, color=LEK_GRAY,  lw=1.5, ls="--", label=f"P10:  {p10_f:,.0f}")
        ax.axvline(p90_f, color=LEK_GRAY,  lw=1.5, ls="--", label=f"P90:  {p90_f:,.0f}")
        ci_width = (p90_f - p10_f) / mu_f * 100
        ax.text(0.97, 0.95,
                f"IC P10-P90: +/-{ci_width/2:.0f}% da media",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=8, color=LEK_GRAY,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=LEK_LGRAY))
        ax.set_title(f"#{i+1} – {lbl}", fontsize=11, fontweight="bold", color=LEK_DKGRN)
        ax.set_xlabel(f"Dem. ({UNITS[lbl]['demand']}) x Preco ({UNITS[lbl]['price']})", fontsize=9)
        ax.set_ylabel("Frequencia", fontsize=9)
        ax.legend(fontsize=8)
        ax.spines[["top", "right"]].set_visible(False)
        ax.set_facecolor(BG_COLOR)

    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight", facecolor=BG_COLOR)
    plt.close(fig)
    print(f"      Salvo: {save_path}")


# ══════════════════════════════════════════════════════════════════════════════
# SECAO 6 – PIPELINE PRINCIPAL
# ══════════════════════════════════════════════════════════════════════════════

def open_image(path: Path):
    """Tenta abrir a imagem no visualizador padrão do SO."""
    try:
        if sys.platform.startswith("darwin"):
            subprocess.Popen(["open", str(path)])
        elif sys.platform.startswith("win"):
            os.startfile(str(path))
        else:
            subprocess.Popen(["xdg-open", str(path)])
    except Exception as e:
        print(f"      (aviso: nao foi possivel abrir {path.name} automaticamente: {e})")


def run_pipeline():
    print("=" * 68)
    print("  DESAFIO L.E.K. 2026 – Monte Carlo: Demanda & Preco")
    print("=" * 68)

    # ── Carrega e trata a base ───────────────────────────────────────────────
    print("\n[1/6] Carregando e tratando basezona.xlsx…")
    data, years_all = load_and_clean_base(INPUT_FILE)

    # ⚠️ [P-07] – corte do historico
    years_hist = [y for y in years_all if HIST_START_YEAR_EXTERNAL <= y <= PROJ_YEAR]
    print(f"      Periodo: {years_hist[0]}-{years_hist[-1]}  "
          f"(⚠️ inicio em {HIST_START_YEAR_EXTERNAL} por premissa [P-07])")

    # ── Monte Carlo por cultura ──────────────────────────────────────────────
    print(f"\n[2/6] Monte Carlo ({N_SIMUL:,} simulacoes x {HORIZON} anos)…")  # ⚠️ [P-01][P-02]
    mc_results = {}
    for raw_name, lbl in CROP_MAP.items():
        print(f"      → {lbl}")
        demand_hist = get_world_demand(data, raw_name, years_hist).dropna()
        price_hist  = build_price_series(lbl)   # ⚠️ [P-03]

        d_paths, mu_d, sigma_d       = monte_carlo_demand(demand_hist)
        p_paths, mu_p, sigma_p, theta = monte_carlo_price(lbl)   # ⚠️ [P-03][P-04]
        mkt = calc_market_potential(d_paths, p_paths)

        mc_results[lbl] = {
            "demand_paths"    : d_paths,
            "price_paths"     : p_paths,
            "demand_hist"     : demand_hist,
            "price_hist"      : price_hist,
            "mu_demand"       : mu_d,
            "sigma_demand"    : sigma_d,
            "mu_price"        : mu_p,     # ⚠️ [P-03]
            "sigma_price"     : sigma_p,  # ⚠️ [P-03]
            "long_run_price"  : theta,    # ⚠️ [P-03]
            "market_potential": mkt,
        }

    # ── Scoring ──────────────────────────────────────────────────────────────
    print("\n[3/6] Scoring multicriterio…")   # ⚠️ [P-05]
    df_score = build_scoring(data, years_hist, mc_results)

    print("\n  +--- RESULTADO DO SCORING ----------------------------------------+")
    for _, row in df_score.iterrows():
        star = " *TOP*" if row["Ranking"] <= 2 else "      "
        print(f"  | #{int(row['Ranking'])} {row['Cultura']:<22} Score: {row['Score Final']:.2f}/10 {star}|")
    print("  +------------------------------------------------------------------+")
    top2 = df_score[df_score["Ranking"] <= 2]["Cultura"].tolist()
    print(f"\n  Priorizadas: {top2[0]}  e  {top2[1]}")

    # ── Gera e salva figuras ─────────────────────────────────────────────────
    print("\n[4/6] Gerando figuras…")
    paths_fig = {
        "fig1": OUTPUT_DIR / "fig1_demand_mc.png",
        "fig2": OUTPUT_DIR / "fig2_price_mc.png",
        "fig3": OUTPUT_DIR / "fig3_dashboard.png",
        "fig4": OUTPUT_DIR / "fig4_top2_distribution.png",
    }
    fig1_demand(mc_results, paths_fig["fig1"])
    fig2_price(mc_results, paths_fig["fig2"])
    fig3_dashboard(data, years_hist, mc_results, df_score, paths_fig["fig3"])
    fig4_top2(mc_results, top2, paths_fig["fig4"])

    # ── Exporta dados ────────────────────────────────────────────────────────
    print("\n[5/6] Exportando CSV / JSON…")

    score_cols = (["Ranking", "Cultura", "Score Final"]
                  + list(SCORING_WEIGHTS_EXTERNAL.keys()))
    df_score[score_cols].to_csv(
        OUTPUT_DIR / "scoring_priorizacao.csv", index=False, encoding="utf-8-sig"
    )

    rows = []
    for lbl, res in mc_results.items():
        for t in range(HORIZON):
            dp, pp = res["demand_paths"][:, t], res["price_paths"][:, t]
            rows.append({
                "Cultura"          : lbl,
                "Ano"              : PROJ_YEAR + t + 1,
                "Demanda_P10"      : np.percentile(dp, 10),   # da base
                "Demanda_Mediana"  : np.median(dp),            # da base
                "Demanda_P90"      : np.percentile(dp, 90),   # da base
                "Preco_P10"        : np.percentile(pp, 10),   # ⚠️ [P-03][P-04]
                "Preco_Mediana"    : np.median(pp),            # ⚠️ [P-03][P-04]
                "Preco_P90"        : np.percentile(pp, 90),   # ⚠️ [P-03][P-04]
                "Potencial_P10"    : res["market_potential"]["p10"].iloc[t],
                "Potencial_Mediana": res["market_potential"]["median"].iloc[t],
                "Potencial_P90"    : res["market_potential"]["p90"].iloc[t],
            })
    pd.DataFrame(rows).to_csv(
        OUTPUT_DIR / "projecoes_mc.csv", index=False,
        encoding="utf-8-sig", float_format="%.4f"
    )

    summary = {
        "top2"            : top2,
        "n_simulacoes"    : N_SIMUL,                      # ⚠️ [P-01]
        "horizonte_anos"  : HORIZON,                      # ⚠️ [P-02]
        "hist_start_year" : HIST_START_YEAR_EXTERNAL,    # ⚠️ [P-07]
        "premissas_externas": {
            "P-01": f"N_SIMUL = {N_SIMUL:,} (padrao literatura)",
            "P-02": f"HORIZON = {HORIZON} anos",
            "P-03": "Precos historicos USDA/ICE/CBOT – NAO estao na base fornecida",
            "P-04": "Kappa OU por cultura (literatura Schwartz 1997)",
            "P-05": "Pesos scoring multicriterio (decisao analitica da equipe)",
            "P-06": f"Fallback GBM mu={FALLBACK_MU_EXTERNAL*100:.0f}%, sigma={FALLBACK_SIGMA_EXTERNAL*100:.0f}% se serie < 5 obs",
            "P-07": f"Historico a partir de {HIST_START_YEAR_EXTERNAL} (descarta 2000-2004 – inconsistencias na base)",
        },
        "parametros_mc": {
            lbl: {
                "mu_demanda_pct"    : round(res["mu_demand"] * 100, 2),    # da base
                "sigma_demanda_pct" : round(res["sigma_demand"] * 100, 2), # da base
                "mu_preco_pct"      : round(res["mu_price"] * 100, 2),     # ⚠️ [P-03]
                "sigma_preco_pct"   : round(res["sigma_price"] * 100, 2),  # ⚠️ [P-03]
                "kappa_ou"          : KAPPA_OU_EXTERNAL[lbl],              # ⚠️ [P-04]
                "long_run_price"    : round(res["long_run_price"], 4),     # ⚠️ [P-03]
            }
            for lbl, res in mc_results.items()
        },
        "scoring": df_score[["Ranking", "Cultura", "Score Final"]].to_dict("records"),
    }
    with open(OUTPUT_DIR / "resumo_mc.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("      Arquivos salvos em:", OUTPUT_DIR)

    # ── Abre imagens automaticamente ─────────────────────────────────────────
    print("\n[6/6] Abrindo visualizacoes…")
    for fig_path in paths_fig.values():
        open_image(fig_path)

    # ── Sumario final ─────────────────────────────────────────────────────────
    print("\n" + "=" * 68)
    print("  PREMISSAS EXTERNAS UTILIZADAS")
    print("=" * 68)
    for code, desc in summary["premissas_externas"].items():
        print(f"  [{code}]  {desc}")
    print()
    print(f"  Priorizadas: {top2[0]}  e  {top2[1]}")
    print(f"  {N_SIMUL:,} simulacoes | {HORIZON} anos | GBM (demanda) + OU (preco)")
    print("=" * 68)

    return mc_results, df_score, pd.DataFrame(rows), summary


if __name__ == "__main__":
    mc_results, df_score, df_proj, summary = run_pipeline()

  DESAFIO L.E.K. 2026 – Monte Carlo: Demanda & Preco

[1/6] Carregando e tratando basezona.xlsx…
      Periodo: 2005-2023  (⚠️ inicio em 2005 por premissa [P-07])

[2/6] Monte Carlo (10,000 simulacoes x 5 anos)…
      → Milho
      → Cafe (Arabica)
      → Cana-de-acucar
      → Citros (OJ)
      → Trigo

[3/6] Scoring multicriterio…

  +--- RESULTADO DO SCORING ----------------------------------------+
  | #1 Citros (OJ)            Score: 6.70/10  *TOP*|
  | #2 Cafe (Arabica)         Score: 5.50/10  *TOP*|
  | #3 Milho                  Score: 4.26/10       |
  | #4 Cana-de-acucar         Score: 3.98/10       |
  | #5 Trigo                  Score: 3.52/10       |
  +------------------------------------------------------------------+

  Priorizadas: Citros (OJ)  e  Cafe (Arabica)

[4/6] Gerando figuras…
      Salvo: C:\Users\lucas\Documents\Poli junior computador\Challenge LEK\Dados-Desafio-LEK\resultados monte carlo\fig1_demand_mc.png
      Salvo: C:\Users\lucas\Documents\Poli junior c

## Modelo 2

In [21]:
"""
=============================================================================
MONTE_CARLO_LEK2026.PY – Modelo completo de simulação (Visual L.E.K.)
=============================================================================
Lê: base_commodities_LEK2026.xlsx (base construída do zero)
Features novas vs. versão anterior:
  - Identidade visual L.E.K. Consulting implementada nos gráficos.
  - Inflação US CPI integrada ao modelo de preço (deflator)
  - VIX como proxy de aversão a risco
  - Fed Funds Rate como variável de custo de capital
  - IPCA Brasil como fator de conversão BRL
  - Stocks-to-use ratio como driver de pressão de oferta
  - Yield trend do Brasil como driver de oferta futura
  - Market share de exportação como indicador de competitividade
  - Scoring com 8 critérios
=============================================================================
"""

import warnings
warnings.filterwarnings("ignore")
import os, sys, subprocess
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import json

# ─────────────────────────────────────────────────────────────────────────────
# 0. CAMINHOS
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH_XLSX = Path("base_commodities_LEK2026.xlsx")
BASE_PATH_CSV  = Path("base_commodities_LEK2026.xlsx - CONSOLIDADO.csv")
OUTPUT_DIR = Path("resultados_lek")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42)

# ─────────────────────────────────────────────────────────────────────────────
# 1. PREMISSAS EXTERNAS – TODAS AGRUPADAS AQUI
# ─────────────────────────────────────────────────────────────────────────────
N_SIMUL = 10_000
HORIZON = 5      # anos (2025–2029)
HIST_START = 2005

KAPPA = {
    "Milho"                   : 0.20,
    "Cafe (Arabica)"          : 0.15,
    "Cana-de-Acucar (Acucar)" : 0.20,
    "Citros (OJ_Laranja)"     : 0.25,
    "Trigo"                   : 0.20,
}

PESOS = {
    "tamanho_mercado_usd"     : 0.20,
    "crescimento_demanda"     : 0.18,
    "posicao_brasil_producao" : 0.15,
    "posicao_brasil_export"   : 0.12,
    "tendencia_preco"         : 0.12,
    "volatilidade_preco"      : 0.08,
    "yield_trend_brasil"      : 0.10,
    "stocks_to_use"           : 0.05,
}
assert abs(sum(PESOS.values()) - 1.0) < 1e-9, "Pesos do scoring não somam 1.0"

FALLBACK_MU    = 0.02
FALLBACK_SIGMA = 0.05

VIX_SENSITIVITY = {
    "Milho"                   : 0.010,
    "Cafe (Arabica)"          : 0.012,
    "Cana-de-Acucar (Acucar)" : 0.008,
    "Citros (OJ_Laranja)"     : 0.015,
    "Trigo"                   : 0.010,
}

FED_PRICE_IMPACT = -0.015
FED_FUNDS_PROJ = {2025: 4.25, 2026: 3.50, 2027: 3.00, 2028: 3.00, 2029: 3.00}
CPI_GROWTH_PROJ = 0.025
VIX_PROJ = 18.0

# ─────────────────────────────────────────────────────────────────────────────
# IDENTIDADE VISUAL L.E.K. CONSULTING
# ─────────────────────────────────────────────────────────────────────────────
# Cores L.E.K.
LEK_GREEN = "#00A651"
LEK_DKGRN = "#004D2C"
LEK_LGRAY  = "#D0D0D0"
LEK_GRAY   = "#555555"
BG_COLOR   = "#FAFAFA"
PALETTE    = ["#00A651", "#007D3F", "#33BF77", "#005C23", "#99DFB9"]

# Paleta para distinção de múltiplas culturas no mesmo gráfico
PALETTE  = [C_LEK_NAVY, C_LEK_GREEN, C_LEK_CYAN, "#6A1B9A", C_LEK_ACCENT]

PROJ_YEAR = 2024
PROJ_YEARS = list(range(PROJ_YEAR + 1, PROJ_YEAR + HORIZON + 1))

# ─────────────────────────────────────────────────────────────────────────────
# 2. LEITURA DA BASE   📊 DA BASE
# ─────────────────────────────────────────────────────────────────────────────

def load_base():
    if BASE_PATH_CSV.exists():
        df = pd.read_csv(BASE_PATH_CSV, header=1)
    elif BASE_PATH_XLSX.exists():
        df = pd.read_excel(BASE_PATH_XLSX, sheet_name="CONSOLIDADO", header=1)
    else:
        raise FileNotFoundError("Base de dados não encontrada.")
        
    df.columns = df.columns.str.lower().str.strip()
    col_map = {c: c.lower().replace(" ", "_") for c in df.columns}
    df = df.rename(columns=col_map)
    return df


def get_crop_series(df, cultura_pattern, col):
    mask = df["cultura"].str.contains(cultura_pattern, case=False, na=False)
    sub  = df[mask].sort_values("ano").set_index("ano")[col]
    return pd.to_numeric(sub, errors="coerce").dropna()


def get_macro_series(df, col):
    mask = df["cultura"].str.contains("Milho", case=False, na=False)
    sub  = df[mask].sort_values("ano").set_index("ano")[col]
    return pd.to_numeric(sub, errors="coerce").dropna()

# ─────────────────────────────────────────────────────────────────────────────
# 3. CALIBRAÇÃO DOS MODELOS
# ─────────────────────────────────────────────────────────────────────────────

def fit_gbm(series: pd.Series, start_year=HIST_START):
    s = series[series.index >= start_year].dropna()
    if len(s) < 5:
        return FALLBACK_MU, FALLBACK_SIGMA
    lr = np.log(s / s.shift(1)).dropna()
    return float(lr.mean()), float(lr.std())

def fit_ou_price(price_series: pd.Series, crop_label: str, start_year=HIST_START):
    s = price_series[price_series.index >= start_year].dropna()
    lr = np.log(s / s.shift(1)).dropna()
    mu_p    = float(lr.mean())
    sigma_p = float(lr.std())
    theta   = float(np.exp(np.log(s).mean()))
    kappa   = KAPPA[crop_label]
    last_p  = float(s.iloc[-1])
    return mu_p, sigma_p, theta, kappa, last_p

def fit_gbm_fx(fx_series: pd.Series, start_year=HIST_START):
    s = fx_series[fx_series.index >= start_year].dropna()
    lr = np.log(s / s.shift(1)).dropna()
    mu_fx    = float(lr.mean())
    sigma_fx = float(lr.std())
    last_fx  = float(s.iloc[-1])
    return mu_fx, sigma_fx, last_fx

# ─────────────────────────────────────────────────────────────────────────────
# 4. SIMULAÇÕES MONTE CARLO
# ─────────────────────────────────────────────────────────────────────────────

def sim_demand_gbm(series: pd.Series, n_sim=N_SIMUL, horizon=HORIZON):
    mu, sigma = fit_gbm(series)
    last = float(series.dropna().iloc[-1])
    dt   = 1.0
    Z    = np.random.normal(0, 1, (n_sim, horizon))
    log_ret = (mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z
    paths = last * np.exp(np.cumsum(log_ret, axis=1))
    return paths, mu, sigma

def sim_price_ou_macro(price_series, crop_label, vix_hist_last, fed_last, n_sim=N_SIMUL, horizon=HORIZON):
    mu_p, sigma_p, theta, kappa, last_p = fit_ou_price(price_series, crop_label)
    vix_proj = VIX_PROJ
    beta_vix = VIX_SENSITIVITY[crop_label]
    gamma    = FED_PRICE_IMPACT
    dt = 1.0
    prices = np.full(n_sim, last_p)
    paths  = np.zeros((n_sim, horizon))

    for t in range(horizon):
        yr = PROJ_YEAR + t + 1
        sigma_adj = sigma_p + beta_vix * vix_proj
        fed_proj_yr = FED_FUNDS_PROJ.get(yr, 3.0)
        macro_drift = gamma * (fed_proj_yr - fed_last)

        Z = np.random.normal(0, 1, n_sim)
        prices = (prices
                  + kappa * (theta - prices) * dt
                  + macro_drift * prices * dt
                  + sigma_adj * prices * np.sqrt(dt) * Z)
        prices = np.clip(prices, 0.01, None)
        paths[:, t] = prices
    return paths, mu_p, sigma_p, theta

def sim_fx_gbm(fx_series: pd.Series, n_sim=N_SIMUL, horizon=HORIZON):
    mu_fx, sigma_fx, last_fx = fit_gbm_fx(fx_series)
    dt  = 1.0
    Z   = np.random.normal(0, 1, (n_sim, horizon))
    log_ret = (mu_fx - 0.5*sigma_fx**2)*dt + sigma_fx*np.sqrt(dt)*Z
    paths = last_fx * np.exp(np.cumsum(log_ret, axis=1))
    return paths, mu_fx, sigma_fx

def sim_cpi_trend(last_cpi, n_sim=N_SIMUL, horizon=HORIZON):
    paths = np.zeros((n_sim, horizon))
    cpi = last_cpi
    for t in range(horizon):
        cpi = cpi * (1 + CPI_GROWTH_PROJ)
        paths[:, t] = cpi
    return paths

def calc_potencial(demand_paths, price_usd_paths, fx_paths, cpi_paths, cpi_base, convert_to_real=True):
    mkt_usd_nom = demand_paths * price_usd_paths
    mkt_brl_nom = mkt_usd_nom * fx_paths

    if convert_to_real:
        deflator = cpi_base / cpi_paths
        mkt_usd_real = mkt_usd_nom * deflator
    else:
        mkt_usd_real = mkt_usd_nom

    rows = []
    for t in range(demand_paths.shape[1]):
        for arr, label in [(mkt_usd_nom[:, t],  "usd_nominal"),
                           (mkt_brl_nom[:, t],  "brl_nominal"),
                           (mkt_usd_real[:, t], "usd_real_2024")]:
            rows.append({
                "year"    : PROJ_YEAR + t + 1,
                "metric"  : label,
                "mean"    : np.mean(arr),
                "median"  : np.median(arr),
                "p10"     : np.percentile(arr, 10),
                "p90"     : np.percentile(arr, 90),
                "std"     : np.std(arr),
            })
    return pd.DataFrame(rows)

# ─────────────────────────────────────────────────────────────────────────────
# 5. SCORING MULTICRITÉRIO (8 critérios)
# ─────────────────────────────────────────────────────────────────────────────

def normalize(d: dict, invert=False):
    vals = np.array(list(d.values()), dtype=float)
    mn, mx = vals.min(), vals.max()
    if mx == mn: return {k: 5.0 for k in d}
    if invert: return {k: 1 + 9*(mx - v)/(mx - mn) for k, v in d.items()}
    return {k: 1 + 9*(v - mn)/(mx - mn) for k, v in d.items()}

def build_scoring(df, mc_results):
    crops = list(mc_results.keys())
    raw = {c: {} for c in PESOS}

    for lbl in crops:
        mkt = mc_results[lbl]["potencial"]
        raw["tamanho_mercado_usd"][lbl] = mkt[(mkt.year == PROJ_YEARS[-1]) & (mkt.metric == "usd_nominal")]["median"].values[0]
        raw["crescimento_demanda"][lbl] = mc_results[lbl]["mu_demand"]

        pattern = lbl.split("(")[0].strip()
        s_prod = get_crop_series(df, pattern, "market_share_producao_pct")[lambda x: x.index >= HIST_START]
        raw["posicao_brasil_producao"][lbl] = float(s_prod.iloc[-3:].mean()) if len(s_prod) >= 3 else 0.0

        s_exp = get_crop_series(df, pattern, "market_share_exportacao_pct")[lambda x: x.index >= HIST_START]
        raw["posicao_brasil_export"][lbl] = float(s_exp.iloc[-3:].mean()) if len(s_exp) >= 3 else 0.0

        ps = mc_results[lbl]["price_hist_real"]
        if len(ps) >= 5:
            x = np.arange(len(ps))
            slope, _ = np.polyfit(x, ps.values, 1)
            raw["tendencia_preco"][lbl] = slope / float(ps.mean())
        else: raw["tendencia_preco"][lbl] = 0.0

        raw["volatilidade_preco"][lbl] = mc_results[lbl]["sigma_price"]

        yld = get_crop_series(df, pattern, "yield_brasil_mt_ha")[lambda x: x.index >= HIST_START].dropna()
        if len(yld) >= 5:
            x = np.arange(len(yld))
            slope, _ = np.polyfit(x, yld.values, 1)
            raw["yield_trend_brasil"][lbl] = slope / float(yld.mean())
        else: raw["yield_trend_brasil"][lbl] = 0.0

        s_stu = get_crop_series(df, pattern, "stocks_to_use_ratio_pct")[lambda x: x.index >= HIST_START]
        raw["stocks_to_use"][lbl] = float(s_stu.iloc[-3:].mean()) if len(s_stu) >= 3 else 50.0

    norm = {
        "tamanho_mercado_usd"     : normalize(raw["tamanho_mercado_usd"]),
        "crescimento_demanda"     : normalize(raw["crescimento_demanda"]),
        "posicao_brasil_producao" : normalize(raw["posicao_brasil_producao"]),
        "posicao_brasil_export"   : normalize(raw["posicao_brasil_export"]),
        "tendencia_preco"         : normalize(raw["tendencia_preco"]),
        "volatilidade_preco"      : normalize(raw["volatilidade_preco"], invert=True),
        "yield_trend_brasil"      : normalize(raw["yield_trend_brasil"]),
        "stocks_to_use"           : normalize(raw["stocks_to_use"], invert=True),
    }

    records = []
    for lbl in crops:
        score = sum(PESOS[c] * norm[c].get(lbl, 5.0) for c in PESOS)
        row = {"Cultura": lbl}
        for c in PESOS:
            row[f"score_{c}"] = round(norm[c].get(lbl, 5.0), 2)
            row[f"raw_{c}"]   = round(raw[c].get(lbl, 0.0), 4)
        row["Score_Final"] = round(score, 2)
        records.append(row)

    df_out = (pd.DataFrame(records)
              .sort_values("Score_Final", ascending=False)
              .reset_index(drop=True))
    df_out["Ranking"] = df_out.index + 1
    return df_out

# ─────────────────────────────────────────────────────────────────────────────
# 6. VISUALIZAÇÕES (ESTILO L.E.K.)
# ─────────────────────────────────────────────────────────────────────────────

def _setup_ax(ax, title, xlabel=None, ylabel=None):
    ax.set_title(title, fontsize=9, fontweight="bold", color=C_LEK_NAVY, pad=6)
    ax.set_facecolor(BG)
    ax.spines[["top","right"]].set_visible(False)
    ax.spines["bottom"].set_color(C_LEK_GRAY)
    ax.spines["left"].set_color(C_LEK_GRAY)
    if xlabel: ax.set_xlabel(xlabel, fontsize=8, color=C_LEK_NAVY)
    if ylabel: ax.set_ylabel(ylabel, fontsize=8, color=C_LEK_NAVY)
    ax.tick_params(labelsize=8, colors=C_LEK_NAVY)


def fan_chart(ax, hist_s, paths, color, label, unit=""):
    proj = PROJ_YEARS
    ax.plot(list(hist_s.index), hist_s.values, color=C_LEK_NAVY, lw=2, label="Histórico")
    ax.fill_between(proj, np.percentile(paths,10,axis=0), np.percentile(paths,90,axis=0),
                    alpha=0.18, color=color, label="P10–P90")
    ax.fill_between(proj, np.percentile(paths,25,axis=0), np.percentile(paths,75,axis=0),
                    alpha=0.35, color=color, label="P25–P75")
    ax.plot(proj, np.median(paths, axis=0), color=color, lw=2.5, ls="--", label="Mediana proj.")
    _setup_ax(ax, f"{label}\n{unit}")


def fig1_demand(mc, save):
    crops = list(mc.keys())
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.patch.set_facecolor(BG)
    fig.suptitle(
        f"Monte Carlo – Demanda Mundial  |  {N_SIMUL:,} simulações\n"
        f"Horizonte: {PROJ_YEARS[0]}–{PROJ_YEARS[-1]}",
        fontsize=12, fontweight="bold", color=C_LEK_NAVY
    )
    for i, lbl in enumerate(crops):
        ax = axes[i//3][i%3]
        r  = mc[lbl]
        # Aplica a cor da cultura vinda da paleta L.E.K.
        fan_chart(ax, r["demand_hist"], r["demand_paths"], PALETTE[i], lbl, "1.000 MT")
        mu_d, sig_d = r["mu_demand"], r["sigma_demand"]
        ax.set_title(f"{lbl}\nμ={mu_d*100:+.1f}%  σ={sig_d*100:.1f}%", fontsize=8, color=C_LEK_NAVY)
        if i == 0: ax.legend(fontsize=7, framealpha=0.5, facecolor=BG)
    axes[1][2].set_visible(False)
    plt.tight_layout()
    fig.savefig(save, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(fig)


def fig2_price(mc, save):
    crops = list(mc.keys())
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.patch.set_facecolor(BG)
    fig.suptitle(
        "Monte Carlo – Preço (Ornstein-Uhlenbeck + Macro Choques)\n"
        "Ajustado por VIX e Fed Funds",
        fontsize=12, fontweight="bold", color=C_LEK_NAVY
    )
    for i, lbl in enumerate(crops):
        ax = axes[i//3][i%3]
        r  = mc[lbl]
        fan_chart(ax, r["price_hist"], r["price_paths"], C_LEK_CYAN, lbl, r['price_unit'])
        theta = r["theta_price"]
        ax.axhline(theta, color=C_LEK_GRAY, lw=1, ls=":", alpha=0.7, label=f"θ={theta:.2f}")
        kap = KAPPA[lbl]
        ax.set_title(f"{lbl}\nθ={theta:.2f}  κ={kap:.2f}  σ_adj={r['sigma_adj']:.3f}",
                     fontsize=8, color=C_LEK_NAVY)
        if i == 0: ax.legend(fontsize=7, framealpha=0.5, facecolor=BG)
    axes[1][2].set_visible(False)
    plt.tight_layout()
    fig.savefig(save, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(fig)


def fig3_fx_cpi(mc, df_base, save):
    fx_hist = get_macro_series(df_base, "usd_brl")[lambda s: s.index >= HIST_START]
    cpi_hist = get_macro_series(df_base, "us_cpi_index")[lambda s: s.index >= HIST_START]
    any_key = list(mc.keys())[0]
    fx_paths  = mc[any_key]["fx_paths"]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor(BG)
    fig.suptitle(
        "Projeção de Variáveis Macro (Câmbio USD/BRL e US CPI)",
        fontsize=12, fontweight="bold", color=C_LEK_NAVY
    )

    ax = axes[0]
    fan_chart(ax, fx_hist, fx_paths, C_LEK_ACCENT, "Câmbio USD/BRL", "R$/US$")
    _setup_ax(ax, "Câmbio USD/BRL (Modelo GBM)", "Ano", "R$/US$")

    ax = axes[1]
    cpi_proj_med = [cpi_hist.iloc[-1] * (1 + CPI_GROWTH_PROJ)**(t+1) for t in range(HORIZON)]
    ax.plot(list(cpi_hist.index), cpi_hist.values, color=C_LEK_NAVY, lw=2, label="Histórico")
    ax.plot(PROJ_YEARS, cpi_proj_med, color=C_LEK_RED, lw=2.5, ls="--", label=f"Proj. {CPI_GROWTH_PROJ*100:.1f}% a.a.")
    lo = [cpi_hist.iloc[-1]*(1+CPI_GROWTH_PROJ-0.005)**(t+1) for t in range(HORIZON)]
    hi = [cpi_hist.iloc[-1]*(1+CPI_GROWTH_PROJ+0.005)**(t+1) for t in range(HORIZON)]
    ax.fill_between(PROJ_YEARS, lo, hi, alpha=0.15, color=C_LEK_RED, label="Margem ±0.5pp")
    ax.legend(fontsize=8, facecolor=BG)
    _setup_ax(ax, "US CPI Index", "Ano", "Índice")

    plt.tight_layout()
    fig.savefig(save, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(fig)


def fig4_dashboard(df_base, mc, df_score, save):
    fig = plt.figure(figsize=(20, 14))
    fig.patch.set_facecolor(BG)
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
    fig.suptitle("L.E.K. 2026 – Dashboard Estratégico de Culturas Alvo",
                 fontsize=15, fontweight="bold", color=C_LEK_NAVY)

    crops = list(mc.keys())

    # ── A: Scoring ────────────────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, :2])
    colors = [C_LEK_GREEN if r <= 2 else C_LEK_LT_GRY for r in df_score["Ranking"]]
    bars = ax.barh(df_score["Cultura"], df_score["Score_Final"], color=colors, edgecolor=BG)
    ax.set_xlim(0, 10)
    for b, v in zip(bars, df_score["Score_Final"]):
        ax.text(b.get_width()+0.1, b.get_y()+b.get_height()/2, f"{v:.2f}", va="center", fontsize=9, color=C_LEK_NAVY)
    ax.invert_yaxis()
    ax.text(0.99, 0.04, "★ Top 2 Priorizadas", transform=ax.transAxes,
            ha="right", color=C_LEK_GREEN, fontsize=10, fontweight="bold")
    _setup_ax(ax, "Scoring Multicritério Final (Escala 0–10)", "Score")

    # ── B: Tabela de parâmetros ───────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 2])
    ax.axis("off")
    tbl_data = []
    for lbl in crops:
        r = mc[lbl]
        tbl_data.append([
            lbl[:12],
            f"{r['mu_demand']*100:+.1f}%",
            f"{r['sigma_demand']*100:.1f}%",
            f"{r['sigma_adj']*100:.1f}%",
            f"{KAPPA[lbl]:.2f}",
        ])
    t = ax.table(cellText=tbl_data,
                 colLabels=["Cultura","μ Dem","σ Dem","σ Preço","κ OU"],
                 loc="center", cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(8); t.scale(1, 1.7)
    for (r,c), cell in t.get_celld().items():
        cell.set_edgecolor(C_LEK_LT_GRY)
        if r == 0:
            cell.set_facecolor(C_LEK_NAVY)
            cell.set_text_props(color=BG, fontweight="bold")
        elif r%2==0: cell.set_facecolor("#F4F6F8")
    ax.set_title("Resumo de Parâmetros Estocásticos", fontsize=9, color=C_LEK_NAVY, pad=10, fontweight="bold")

    # ── C: Potencial USD nominal ano 5 ───────────────────────────────────
    ax = fig.add_subplot(gs[1, :2])
    ord_crops = df_score["Cultura"].tolist()
    meds, p10s, p90s = [], [], []
    for lbl in ord_crops:
        mkt = mc[lbl]["potencial"]
        row = mkt[(mkt.year == PROJ_YEARS[-1]) & (mkt.metric == "usd_nominal")]
        meds.append(row["median"].values[0])
        p10s.append(row["p10"].values[0])
        p90s.append(row["p90"].values[0])
    x = np.arange(len(ord_crops))
    ax.bar(x, meds, color=C_LEK_GREEN, alpha=0.9, edgecolor=BG)
    ax.errorbar(x, meds, yerr=[np.array(meds)-np.array(p10s), np.array(p90s)-np.array(meds)],
                fmt="none", color=C_LEK_NAVY, capsize=5, lw=1.5)
    ax.set_xticks(x); ax.set_xticklabels(ord_crops, rotation=15, ha="right", fontsize=9)
    _setup_ax(ax, f"Potencial de Mercado Global em {PROJ_YEARS[-1]} (USD Nominal)\nLinhas indicam spread P10–P90")

    # ── D: Potencial BRL nominal ─────────────────────────────────────────
    ax = fig.add_subplot(gs[1, 2])
    meds_brl = []
    for lbl in ord_crops:
        mkt = mc[lbl]["potencial"]
        row = mkt[(mkt.year == PROJ_YEARS[-1]) & (mkt.metric == "brl_nominal")]
        meds_brl.append(row["median"].values[0])
    ax.barh(ord_crops, meds_brl, color=C_LEK_CYAN, alpha=0.9, edgecolor=BG)
    ax.invert_yaxis()
    _setup_ax(ax, f"Potencial de Mercado Convertido ({PROJ_YEARS[-1]} BRL Nominal)", "R$")

    # ── E: Share Produção ────────────────────────────────────────────────
    ax = fig.add_subplot(gs[2, 0])
    for i, lbl in enumerate(crops):
        pattern = lbl.split("(")[0].strip()
        s = get_crop_series(df_base, pattern, "market_share_producao_pct")[lambda x: x.index >= HIST_START]
        ax.plot(s.index, s.values, lw=2, label=lbl[:8], color=PALETTE[i], marker="o", markersize=3)
    ax.legend(fontsize=7, loc="upper left", framealpha=0.8, facecolor=BG)
    _setup_ax(ax, "Market Share Global – Produção Brasil (%)", "Ano", "%")

    # ── F: Share Exportação ──────────────────────────────────────────────
    ax = fig.add_subplot(gs[2, 1])
    for i, lbl in enumerate(crops):
        pattern = lbl.split("(")[0].strip()
        s = get_crop_series(df_base, pattern, "market_share_exportacao_pct")[lambda x: x.index >= HIST_START]
        ax.plot(s.index, s.values, lw=2, label=lbl[:8], color=PALETTE[i], marker="s", markersize=3)
    ax.legend(fontsize=7, loc="upper left", framealpha=0.8, facecolor=BG)
    _setup_ax(ax, "Market Share Global – Exportação Brasil (%)", "Ano", "%")

    # ── G: Stocks-to-use ratio ───────────────────────────────────────────
    ax = fig.add_subplot(gs[2, 2])
    for i, lbl in enumerate(crops):
        pattern = lbl.split("(")[0].strip()
        s = get_crop_series(df_base, pattern, "stocks_to_use_ratio_pct")[lambda x: x.index >= HIST_START]
        ax.plot(s.index, s.values, lw=2, label=lbl[:8], color=PALETTE[i], marker="^", markersize=3)
    ax.legend(fontsize=7, loc="upper right", framealpha=0.8, facecolor=BG)
    _setup_ax(ax, "Stocks-to-Use Ratio Mundial (%)", "Ano", "%")

    fig.savefig(save, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(fig)


def fig5_top2(mc, top2, save):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.patch.set_facecolor(BG)
    fig.suptitle(
        f"Distribuições Probabilísticas – Top 2 Priorizadas ({PROJ_YEARS[-1]})\n"
        f"Ativos Estratégicos: {top2[0]} e {top2[1]}",
        fontsize=13, fontweight="bold", color=C_LEK_NAVY
    )
    for col, lbl in enumerate(top2):
        r = mc[lbl]
        ax = axes[0][col]
        arr_usd = r["demand_paths"][:,-1] * r["price_paths"][:,-1]
        mu_u  = np.mean(arr_usd)
        p10_u = np.percentile(arr_usd, 10)
        p90_u = np.percentile(arr_usd, 90)
        ax.hist(arr_usd, bins=80, color=PALETTE[col], edgecolor=BG, alpha=0.9)
        ax.axvline(mu_u,  color=C_LEK_NAVY, lw=2.5, label=f"Média: {mu_u:,.0f}")
        ax.axvline(p10_u, color=C_LEK_GRAY,  lw=1.5, ls="--", label=f"P10: {p10_u:,.0f}")
        ax.axvline(p90_u, color=C_LEK_GRAY,  lw=1.5, ls="--", label=f"P90: {p90_u:,.0f}")
        ax.legend(fontsize=8, facecolor=BG)
        _setup_ax(ax, f"{lbl} – Potencial USD Nominal")

        ax = axes[1][col]
        arr_brl = arr_usd * r["fx_paths"][:,-1]
        mu_b  = np.mean(arr_brl)
        p10_b = np.percentile(arr_brl, 10)
        p90_b = np.percentile(arr_brl, 90)
        ax.hist(arr_brl, bins=80, color=C_LEK_GREEN, edgecolor=BG, alpha=0.9)
        ax.axvline(mu_b,  color=C_LEK_NAVY, lw=2.5, label=f"Média: {mu_b:,.0f}")
        ax.axvline(p10_b, color=C_LEK_GRAY,  lw=1.5, ls="--", label=f"P10: {p10_b:,.0f}")
        ax.axvline(p90_b, color=C_LEK_GRAY,  lw=1.5, ls="--", label=f"P90: {p90_b:,.0f}")
        ax.legend(fontsize=8, facecolor=BG)
        _setup_ax(ax, f"{lbl} – Potencial BRL Nominal (Câmbio Embutido)")

    plt.tight_layout()
    fig.savefig(save, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(fig)


def fig6_macro_sensitivity(mc, top2, save):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor(BG)
    fig.suptitle("Análise de Sensibilidade Macro (Fed Funds e VIX)", fontsize=12, fontweight="bold", color=C_LEK_NAVY)

    ax = axes[0]
    fed_range = np.linspace(1.0, 6.0, 20)
    for i, lbl in enumerate(top2):
        r   = mc[lbl]
        ps  = r["price_hist"]
        _, sigma_p, theta, kappa, last_p = fit_ou_price(ps, lbl)
        prices_fed = []
        for f in fed_range:
            macro_drift = FED_PRICE_IMPACT * (f - float(ps[ps.index >= HIST_START].iloc[-1]))
            p_eq = theta * np.exp(macro_drift * HORIZON)
            prices_fed.append(p_eq)
        ax.plot(fed_range, prices_fed, lw=2.5, label=lbl, color=PALETTE[i])
    ax.axvline(FED_FUNDS_PROJ[PROJ_YEARS[-1]], color=C_LEK_GRAY, lw=1.5, ls="--",
               label=f"Fed proj. {PROJ_YEARS[-1]}: {FED_FUNDS_PROJ[PROJ_YEARS[-1]]}%")
    ax.legend(fontsize=8, facecolor=BG)
    _setup_ax(ax, "Preço de equilíbrio (θ) vs. Fed Funds", "Fed Funds (%)", "Preço")

    ax = axes[1]
    vix_range = np.linspace(10, 45, 20)
    for i, lbl in enumerate(top2):
        r = mc[lbl]
        _, sigma_p, theta, kappa, last_p = fit_ou_price(r["price_hist"], lbl)
        beta = VIX_SENSITIVITY[lbl]
        sigma_adj_range = sigma_p + beta * vix_range
        spread_pct = sigma_adj_range * np.sqrt(HORIZON) * 200
        ax.plot(vix_range, spread_pct, lw=2.5, label=lbl, color=PALETTE[i])
    ax.axvline(VIX_PROJ, color=C_LEK_GRAY, lw=1.5, ls="--", label=f"VIX proj.: {VIX_PROJ}")
    ax.legend(fontsize=8, facecolor=BG)
    _setup_ax(ax, "Spread de Risco (P90-P10) vs. Índice VIX", "VIX Médio", "Spread % (~±1σ em 5 anos)")

    plt.tight_layout()
    fig.savefig(save, dpi=150, bbox_inches="tight", facecolor=BG)
    plt.close(fig)

# ─────────────────────────────────────────────────────────────────────────────
# 7. PIPELINE PRINCIPAL
# ─────────────────────────────────────────────────────────────────────────────

def run():
    print("=" * 70)
    print("  MONTE CARLO L.E.K. 2026 – Inicializando Engine...")
    print("=" * 70)

    df = load_base()
    fx_hist  = get_macro_series(df, "usd_brl")[lambda s: s.index >= HIST_START]
    cpi_hist = get_macro_series(df, "us_cpi_index")[lambda s: s.index >= HIST_START]
    vix_hist = get_macro_series(df, "vix_avg")[lambda s: s.index >= HIST_START]
    fed_hist = get_macro_series(df, "us_fed_funds_rate_pct")[lambda s: s.index >= HIST_START]

    vix_last, fed_last, cpi_last = float(vix_hist.iloc[-1]), float(fed_hist.iloc[-1]), float(cpi_hist.iloc[-1])

    fx_paths,  mu_fx, sigma_fx = sim_fx_gbm(fx_hist)
    cpi_paths                  = sim_cpi_trend(cpi_last)

    CULTURE_SEARCH = {"Milho": "Milho", "Cafe (Arabica)": "Cafe", "Cana-de-Acucar (Acucar)": "Cana", "Citros (OJ_Laranja)": "Citros", "Trigo": "Trigo"}

    mc = {}
    for lbl, pattern in CULTURE_SEARCH.items():
        demand_hist = get_crop_series(df, pattern, "consumo_mundial_1000mt")[lambda s: s.index >= HIST_START].dropna()
        price_hist  = get_crop_series(df, pattern, "preco_usd_unidade")[lambda s: s.index >= HIST_START].dropna()
        price_real  = get_crop_series(df, pattern, "preco_real_usd2024_unidade")[lambda s: s.index >= HIST_START].dropna()
        price_unit  = df[df["cultura"].str.contains(pattern, case=False, na=False)]["unidade_preco"].iloc[0]

        d_paths, mu_d, sig_d = sim_demand_gbm(demand_hist)
        p_paths, mu_p, sig_p, theta = sim_price_ou_macro(price_hist, lbl, vix_last, fed_last)
        sigma_adj = sig_p + VIX_SENSITIVITY[lbl] * VIX_PROJ
        potencial = calc_potencial(d_paths, p_paths, fx_paths, cpi_paths, cpi_base=cpi_last)

        mc[lbl] = {"demand_paths": d_paths, "price_paths": p_paths, "fx_paths": fx_paths, "cpi_paths": cpi_paths,
                   "demand_hist": demand_hist, "price_hist": price_hist, "price_hist_real": price_real,
                   "mu_demand": mu_d, "sigma_demand": sig_d, "mu_price": mu_p, "sigma_price": sig_p,
                   "sigma_adj": sigma_adj, "theta_price": theta, "price_unit": price_unit, "potencial": potencial}

    df_score = build_scoring(df, mc)
    top2 = df_score[df_score["Ranking"] <= 2]["Cultura"].tolist()

    figs = {
        "fig1": OUTPUT_DIR / "fig1_demand_mc.png",
        "fig2": OUTPUT_DIR / "fig2_price_mc.png",
        "fig3": OUTPUT_DIR / "fig3_fx_cpi.png",
        "fig4": OUTPUT_DIR / "fig4_dashboard.png",
        "fig5": OUTPUT_DIR / "fig5_top2_distributions.png",
        "fig6": OUTPUT_DIR / "fig6_macro_sensitivity.png",
    }
    fig1_demand(mc, figs["fig1"])
    fig2_price(mc, figs["fig2"])
    fig3_fx_cpi(mc, df, figs["fig3"])
    fig4_dashboard(df, mc, df_score, figs["fig4"])
    fig5_top2(mc, top2, figs["fig5"])
    fig6_macro_sensitivity(mc, top2, figs["fig6"])

    return mc, df_score

if __name__ == "__main__":
    mc, df_score = run()

  MONTE CARLO L.E.K. 2026 – Inicializando Engine...


## modelo 3


In [16]:
"""
=============================================================================
MONTE_CARLO_LEK2026.PY  –  Modelo Monte Carlo com identidade visual L.E.K.
=============================================================================
Lê:    base_commodities_LEK2026.xlsx
Gera:  6 figuras PNG  +  projecoes_mc.csv  +  scoring_mc.csv  +  resumo_mc.json
LEGENDA:  📊 DA BASE = valor lido/calibrado da base   |   ⚠️ PREMISSA = assumido externamente
=============================================================================
"""
import warnings; warnings.filterwarnings("ignore")
import os, sys, subprocess, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from pathlib import Path

BASE_PATH  = Path("base_commodities_LEK2026.xlsx")
OUTPUT_DIR = Path("resultados_lek")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42)

# ── IDENTIDADE VISUAL L.E.K. ─────────────────────────────────────────────────
LEK_GREEN       = "#00A651"
LEK_DARK_GREEN  = "#004D2C"
LEK_LIGHT_GREEN = "#C8E6C9"
LEK_NAVY        = "#1A2744"
LEK_GRAY        = "#6E7B8B"
LEK_LIGHT_GRAY  = "#E8ECEF"
LEK_WHITE       = "#FFFFFF"
LEK_BG          = "#F8F9FA"
LEK_PALETTE     = ["#00A651","#1A2744","#2196F3","#FF6F00","#6A1B9A"]
COLOR_HIST      = LEK_DARK_GREEN
COLOR_MACRO     = "#FF6F00"

plt.rcParams.update({
    "font.family":"DejaVu Sans","font.size":9,
    "axes.titlesize":10,"axes.labelsize":9,
    "xtick.labelsize":8,"ytick.labelsize":8,"legend.fontsize":8,
    "figure.facecolor":LEK_BG,"axes.facecolor":LEK_BG,
    "axes.edgecolor":LEK_LIGHT_GRAY,"axes.grid":True,
    "grid.color":LEK_LIGHT_GRAY,"grid.linewidth":0.6,
    "axes.spines.top":False,"axes.spines.right":False,
    "text.color":LEK_NAVY,"axes.labelcolor":LEK_NAVY,
    "xtick.color":LEK_GRAY,"ytick.color":LEK_GRAY,
})

PROJ_YEAR  = 2024
HORIZON    = 5                                                  # ⚠️ PREMISSA [P-02]
PROJ_YEARS = list(range(PROJ_YEAR+1, PROJ_YEAR+HORIZON+1))

# ── PREMISSAS EXTERNAS ───────────────────────────────────────────────────────
N_SIMUL    = 10_000   # ⚠️ [P-01]
HIST_START = 2005     # ⚠️ [P-03]
KAPPA = {"Milho":0.20,"Cafe (Arabica)":0.15,"Cana-de-Acucar (Acucar)":0.20,
         "Citros (OJ_Laranja)":0.25,"Trigo":0.20}              # ⚠️ [P-04]
PESOS = {"tamanho_mercado_usd":0.20,"crescimento_demanda":0.18,
         "posicao_brasil_producao":0.15,"posicao_brasil_export":0.12,
         "tendencia_preco":0.12,"volatilidade_preco":0.08,
         "yield_trend_brasil":0.10,"stocks_to_use":0.05}       # ⚠️ [P-05]
assert abs(sum(PESOS.values())-1.0)<1e-9
FALLBACK_MU=0.02; FALLBACK_SIGMA=0.05                          # ⚠️ [P-06]
VIX_SENSITIVITY={"Milho":0.010,"Cafe (Arabica)":0.012,
                 "Cana-de-Acucar (Acucar)":0.008,
                 "Citros (OJ_Laranja)":0.015,"Trigo":0.010}    # ⚠️ [P-07]
FED_PRICE_IMPACT=-0.015                                        # ⚠️ [P-08]
FED_FUNDS_PROJ={2025:4.25,2026:3.50,2027:3.00,2028:3.00,2029:3.00}  # ⚠️ [P-09]
CPI_GROWTH_PROJ=0.025                                          # ⚠️ [P-10]
VIX_PROJ=18.0                                                  # ⚠️ [P-11]

# ── BASE ──────────────────────────────────────────────────────────────────────
def load_base():
    df = pd.read_excel(BASE_PATH, sheet_name="CONSOLIDADO", header=1)
    df.columns = df.columns.str.lower().str.strip().str.replace(" ","_")
    return df

def get_crop_series(df, pat, col):
    mask=df["cultura"].str.contains(pat,case=False,na=False)
    return pd.to_numeric(df[mask].sort_values("ano").set_index("ano")[col],errors="coerce").dropna()

def get_macro_series(df, col):
    mask=df["cultura"].str.contains("Milho",case=False,na=False)
    return pd.to_numeric(df[mask].sort_values("ano").set_index("ano")[col],errors="coerce").dropna()

# ── CALIBRAÇÃO ────────────────────────────────────────────────────────────────
def fit_gbm(s):
    s=s[s.index>=HIST_START].dropna()
    if len(s)<5: return FALLBACK_MU,FALLBACK_SIGMA           # ⚠️ [P-06]
    lr=np.log(s/s.shift(1)).dropna()
    return float(lr.mean()),float(lr.std())

def fit_ou(ps,lbl):
    s=ps[ps.index>=HIST_START].dropna()
    lr=np.log(s/s.shift(1)).dropna()
    return (float(lr.mean()),float(lr.std()),
            float(np.exp(np.log(s).mean())),
            KAPPA[lbl],float(s.iloc[-1]))                    # κ ⚠️ [P-04]

def fit_fx(s):
    s=s[s.index>=HIST_START].dropna()
    lr=np.log(s/s.shift(1)).dropna()
    return float(lr.mean()),float(lr.std()),float(s.iloc[-1])

# ── SIMULAÇÕES ────────────────────────────────────────────────────────────────
def sim_demand(s):
    mu,sig=fit_gbm(s); last=float(s.dropna().iloc[-1])
    Z=np.random.normal(0,1,(N_SIMUL,HORIZON))               # ⚠️ [P-01][P-02]
    return last*np.exp(np.cumsum((mu-0.5*sig**2)+sig*Z,axis=1)),mu,sig

def sim_price(ps,lbl,fed_last):
    mu_p,sig_p,theta,kappa,last_p=fit_ou(ps,lbl)
    sig_adj=sig_p+VIX_SENSITIVITY[lbl]*VIX_PROJ             # ⚠️ [P-07][P-11]
    prices=np.full(N_SIMUL,last_p)
    paths=np.zeros((N_SIMUL,HORIZON))
    for t in range(HORIZON):
        yr=PROJ_YEAR+t+1
        mac=FED_PRICE_IMPACT*(FED_FUNDS_PROJ.get(yr,3.0)-fed_last)  # ⚠️ [P-08][P-09]
        Z=np.random.normal(0,1,N_SIMUL)
        prices=(prices+kappa*(theta-prices)+mac*prices+sig_adj*prices*Z)
        prices=np.clip(prices,0.01,None)
        paths[:,t]=prices
    return paths,mu_p,sig_p,theta,sig_adj

def sim_fx(s):
    mu,sig,last=fit_fx(s)
    Z=np.random.normal(0,1,(N_SIMUL,HORIZON))
    return last*np.exp(np.cumsum((mu-0.5*sig**2)+sig*Z,axis=1)),mu,sig

def sim_cpi(last_cpi):
    paths=np.zeros((N_SIMUL,HORIZON)); c=last_cpi
    for t in range(HORIZON):
        c*=(1+CPI_GROWTH_PROJ)                               # ⚠️ [P-10]
        paths[:,t]=c
    return paths

def calc_pot(d,p,fx,cpi,cpi0):
    mkt_u=d*p; mkt_b=mkt_u*fx; mkt_r=mkt_u*(cpi0/cpi)      # ⚠️ [P-10]
    rows=[]
    for t in range(HORIZON):
        for arr,lbl in [(mkt_u[:,t],"usd_nominal"),(mkt_b[:,t],"brl_nominal"),(mkt_r[:,t],"usd_real_2024")]:
            rows.append({"year":PROJ_YEAR+t+1,"metric":lbl,
                         "mean":np.mean(arr),"median":np.median(arr),
                         "p10":np.percentile(arr,10),"p90":np.percentile(arr,90),"std":np.std(arr)})
    return pd.DataFrame(rows)

# ── SCORING ───────────────────────────────────────────────────────────────────
def norm(d,inv=False):
    v=np.array(list(d.values()),dtype=float); mn,mx=v.min(),v.max()
    if mx==mn: return {k:5.0 for k in d}
    return ({k:1+9*(mx-v)/(mx-mn) for k,v in d.items()} if inv
            else {k:1+9*(v-mn)/(mx-mn) for k,v in d.items()})

def build_scoring(df,mc):
    crops=list(mc.keys()); raw={c:{} for c in PESOS}
    for lbl in crops:
        pat=lbl.split("(")[0].strip()
        mkt=mc[lbl]["potencial"]
        raw["tamanho_mercado_usd"][lbl]=mkt[(mkt.year==PROJ_YEARS[-1])&(mkt.metric=="usd_nominal")]["median"].values[0]
        raw["crescimento_demanda"][lbl]=mc[lbl]["mu_demand"]
        for col_key,raw_key in [("market_share_producao_pct","posicao_brasil_producao"),
                                 ("market_share_exportacao_pct","posicao_brasil_export"),
                                 ("stocks_to_use_ratio_pct","stocks_to_use")]:
            s=get_crop_series(df,pat,col_key); s=s[s.index>=HIST_START]
            raw[raw_key][lbl]=float(s.iloc[-3:].mean()) if len(s)>=3 else 0.0
        pr=mc[lbl]["price_hist_real"]
        if len(pr)>=5:
            sl,_=np.polyfit(np.arange(len(pr)),pr.values,1)
            raw["tendencia_preco"][lbl]=sl/float(pr.mean())
        else: raw["tendencia_preco"][lbl]=0.0
        raw["volatilidade_preco"][lbl]=mc[lbl]["sigma_price"]
        yld=get_crop_series(df,pat,"yield_brasil_mt_ha"); yld=yld[yld.index>=HIST_START].dropna()
        if len(yld)>=5:
            sl,_=np.polyfit(np.arange(len(yld)),yld.values,1)
            raw["yield_trend_brasil"][lbl]=sl/float(yld.mean())
        else: raw["yield_trend_brasil"][lbl]=0.0
    n2={
        "tamanho_mercado_usd":norm(raw["tamanho_mercado_usd"]),
        "crescimento_demanda":norm(raw["crescimento_demanda"]),
        "posicao_brasil_producao":norm(raw["posicao_brasil_producao"]),
        "posicao_brasil_export":norm(raw["posicao_brasil_export"]),
        "tendencia_preco":norm(raw["tendencia_preco"]),
        "volatilidade_preco":norm(raw["volatilidade_preco"],inv=True),
        "yield_trend_brasil":norm(raw["yield_trend_brasil"]),
        "stocks_to_use":norm(raw["stocks_to_use"],inv=True),
    }
    records=[]
    for lbl in crops:
        score=sum(PESOS[c]*n2[c].get(lbl,5.0) for c in PESOS)   # ⚠️ [P-05]
        row={"Cultura":lbl}
        for c in PESOS: row[f"score_{c}"]=round(n2[c].get(lbl,5.0),2); row[f"raw_{c}"]=round(raw[c].get(lbl,0.0),4)
        row["Score_Final"]=round(score,2); records.append(row)
    df_o=(pd.DataFrame(records).sort_values("Score_Final",ascending=False).reset_index(drop=True))
    df_o["Ranking"]=df_o.index+1
    return df_o

# ── HELPERS VISUAIS L.E.K. ────────────────────────────────────────────────────
def _lek_ax(ax,title,xlabel=None,ylabel=None,subtitle=None):
    full=title+(f"\n{subtitle}" if subtitle else "")
    ax.set_title(full,fontsize=9,fontweight="bold",color=LEK_NAVY,pad=6,loc="left")
    ax.set_facecolor(LEK_BG)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(LEK_LIGHT_GRAY); ax.spines["bottom"].set_color(LEK_LIGHT_GRAY)
    ax.grid(True,color=LEK_LIGHT_GRAY,linewidth=0.6,axis="y"); ax.set_axisbelow(True)
    if xlabel: ax.set_xlabel(xlabel,fontsize=8,color=LEK_GRAY)
    if ylabel: ax.set_ylabel(ylabel,fontsize=8,color=LEK_GRAY)

def _lek_fig(fig,title,subtitle=""):
    fig.patch.set_facecolor(LEK_BG)
    bar=fig.add_axes([0,0.975,1,0.015]); bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])
    fig.suptitle(title,fontsize=13,fontweight="bold",color=LEK_NAVY,y=0.968,x=0.02,ha="left")
    if subtitle: fig.text(0.02,0.952,subtitle,fontsize=8,color=LEK_GRAY,ha="left",va="top")

def _lek_leg(ax):
    return ax.legend(fontsize=7,framealpha=0.9,facecolor=LEK_WHITE,
                     edgecolor=LEK_LIGHT_GRAY,labelcolor=LEK_NAVY)

def _wm(fig):
    fig.text(0.98,0.01,"L.E.K. Consulting  |  Análise Estratégica 2026",
             fontsize=7,color=LEK_GRAY,ha="right",va="bottom",fontstyle="italic")

def fan(ax,hist_s,paths,color):
    proj=PROJ_YEARS
    ax.plot(list(hist_s.index),hist_s.values,color=COLOR_HIST,lw=2.2,label="Histórico",zorder=3)
    ax.fill_between(proj,np.percentile(paths,10,axis=0),np.percentile(paths,90,axis=0),alpha=0.15,color=color,label="P10–P90")
    ax.fill_between(proj,np.percentile(paths,25,axis=0),np.percentile(paths,75,axis=0),alpha=0.35,color=color,label="P25–P75")
    ax.plot(proj,np.median(paths,axis=0),color=color,lw=2.5,ls="--",label="Mediana",zorder=3)
    ax.axvline(PROJ_YEAR,color=LEK_GRAY,lw=0.8,ls=":",alpha=0.7)

# ── FIGURAS ───────────────────────────────────────────────────────────────────
def fig1_demand(mc,save):
    crops=list(mc.keys())
    fig,axes=plt.subplots(2,3,figsize=(19,11))
    _lek_fig(fig,"Monte Carlo – Demanda Mundial por Cultura",
             f"{N_SIMUL:,} simulações  |  GBM calibrado na base USDA PSD  |  Horizonte {PROJ_YEARS[0]}–{PROJ_YEARS[-1]}")
    for i,lbl in enumerate(crops):
        ax=axes[i//3][i%3]; r=mc[lbl]
        fan(ax,r["demand_hist"],r["demand_paths"],LEK_PALETTE[i])
        _lek_ax(ax,lbl,"Ano","1.000 MT",subtitle=f"μ={r['mu_demand']*100:+.1f}%  σ={r['sigma_demand']*100:.1f}%  (📊 USDA PSD)")
        if i==0: _lek_leg(ax)
    axes[1][2].set_visible(False); _wm(fig)
    plt.tight_layout(rect=[0,0.02,1,0.94]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"  ✓ {save.name}")

def fig2_price(mc,save):
    crops=list(mc.keys())
    fig,axes=plt.subplots(2,3,figsize=(19,11))
    _lek_fig(fig,"Monte Carlo – Preço por Cultura (Ornstein-Uhlenbeck + Macro)",
             "⚠️ κ=[P-04]  |  ⚠️ β_vix=[P-07]  |  ⚠️ VIX=[P-11]  |  ⚠️ Fed=[P-08][P-09]")
    for i,lbl in enumerate(crops):
        ax=axes[i//3][i%3]; r=mc[lbl]
        fan(ax,r["price_hist"],r["price_paths"],LEK_PALETTE[i])
        ax.axhline(r["theta_price"],color=LEK_GRAY,lw=1.2,ls=":",alpha=0.8,label=f"θ={r['theta_price']:.2f}")
        _lek_ax(ax,lbl,"Ano",r["price_unit"],
                subtitle=f"θ={r['theta_price']:.2f}  κ={KAPPA[lbl]:.2f}⚠️  σ_adj={r['sigma_adj']:.3f}⚠️")
        if i==0: _lek_leg(ax)
    axes[1][2].set_visible(False); _wm(fig)
    plt.tight_layout(rect=[0,0.02,1,0.94]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"  ✓ {save.name}")

def fig3_fx_cpi(mc,df_base,save):
    fx_hist=get_macro_series(df_base,"usd_brl")[lambda s:s.index>=HIST_START]
    cpi_hist=get_macro_series(df_base,"us_cpi_index")[lambda s:s.index>=HIST_START]
    fx_paths=mc[list(mc.keys())[0]]["fx_paths"]
    fig,axes=plt.subplots(1,2,figsize=(15,6))
    _lek_fig(fig,"Projeção de Variáveis Macroeconômicas",
             "📊 Câmbio GBM calibrado na base (BCB SGS)  |  ⚠️ CPI crescimento constante [P-10]")
    fan(axes[0],fx_hist,fx_paths,COLOR_MACRO)
    _lek_ax(axes[0],"Câmbio USD/BRL","Ano","R$/US$",subtitle="📊 Série BCB SGS – GBM calibrado"); _lek_leg(axes[0])
    last_cpi=float(cpi_hist.iloc[-1])
    proj=[last_cpi*(1+CPI_GROWTH_PROJ)**(t+1) for t in range(HORIZON)]
    lo=[last_cpi*(1+CPI_GROWTH_PROJ-0.005)**(t+1) for t in range(HORIZON)]
    hi=[last_cpi*(1+CPI_GROWTH_PROJ+0.005)**(t+1) for t in range(HORIZON)]
    axes[1].plot(list(cpi_hist.index),cpi_hist.values,color=COLOR_HIST,lw=2.2,label="Histórico (📊 BLS)")
    axes[1].plot(PROJ_YEARS,proj,color=LEK_GREEN,lw=2.5,ls="--",label=f"Proj. {CPI_GROWTH_PROJ*100:.1f}%a.a. ⚠️[P-10]")
    axes[1].fill_between(PROJ_YEARS,lo,hi,alpha=0.2,color=LEK_GREEN,label="±0.5pp ⚠️[P-10]")
    axes[1].axvline(PROJ_YEAR,color=LEK_GRAY,lw=0.8,ls=":",alpha=0.7)
    _lek_ax(axes[1],"US CPI Index (BLS CPI-U)","Ano","Índice",subtitle="⚠️ Crescimento projetado = premissa [P-10]"); _lek_leg(axes[1])
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.93]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"  ✓ {save.name}")

def fig4_dashboard(df_base,mc,df_score,save):
    fig=plt.figure(figsize=(22,16)); fig.patch.set_facecolor(LEK_BG)
    gs=gridspec.GridSpec(3,3,figure=fig,hspace=0.50,wspace=0.38,top=0.90,bottom=0.06,left=0.06,right=0.97)
    bar=fig.add_axes([0,0.955,1,0.015]); bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])
    fig.suptitle("L.E.K. Consulting  |  Dashboard Estratégico de Culturas  |  2026",
                 fontsize=14,fontweight="bold",color=LEK_NAVY,y=0.975,x=0.02,ha="left")
    fig.text(0.02,0.952,"📊=base calibrada  |  ⚠️=premissa externa  |  GBM + Ornstein-Uhlenbeck + Macro",
             fontsize=8,color=LEK_GRAY,ha="left")
    crops=list(mc.keys())

    ax=fig.add_subplot(gs[0,:2])
    colors=[LEK_GREEN if r<=2 else LEK_LIGHT_GRAY for r in df_score["Ranking"]]
    bars=ax.barh(df_score["Cultura"],df_score["Score_Final"],color=colors,edgecolor=LEK_BG,height=0.6)
    ax.set_xlim(0,10.5)
    for b,v in zip(bars,df_score["Score_Final"]):
        ax.text(b.get_width()+0.12,b.get_y()+b.get_height()/2,f"{v:.2f}",va="center",fontsize=9,color=LEK_NAVY,fontweight="bold")
    ax.invert_yaxis(); ax.axvline(5,color=LEK_LIGHT_GRAY,lw=1,ls="--")
    ax.text(10.3,-0.5,"★ Top 2",color=LEK_GREEN,fontsize=10,fontweight="bold",ha="right")
    _lek_ax(ax,"Scoring Multicritério  (0–10)",subtitle="⚠️ Pesos [P-05]  |  8 critérios  |  Normalização min-max")

    ax=fig.add_subplot(gs[0,2]); ax.axis("off")
    tdata=[[lbl[:14],f"{mc[lbl]['mu_demand']*100:+.1f}%",f"{mc[lbl]['sigma_demand']*100:.1f}%",
            f"{mc[lbl]['sigma_adj']*100:.1f}%⚠️",f"{KAPPA[lbl]:.2f}⚠️"] for lbl in crops]
    t=ax.table(cellText=tdata,colLabels=["Cultura","μ Dem📊","σ Dem📊","σ P⚠️","κ⚠️"],loc="center",cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(8); t.scale(1,1.75)
    for (r_i,c_i),cell in t.get_celld().items():
        cell.set_edgecolor(LEK_LIGHT_GRAY)
        if r_i==0: cell.set_facecolor(LEK_DARK_GREEN); cell.set_text_props(color=LEK_WHITE,fontweight="bold")
        elif r_i%2==0: cell.set_facecolor(LEK_LIGHT_GREEN)
        else: cell.set_facecolor(LEK_BG)
    ax.set_title("Parâmetros Estocásticos",fontsize=9,color=LEK_NAVY,fontweight="bold",pad=12,loc="left")

    ax=fig.add_subplot(gs[1,:2])
    ord_crops=df_score["Cultura"].tolist(); meds,p10s,p90s=[],[],[]
    for lbl in ord_crops:
        mkt=mc[lbl]["potencial"]; row=mkt[(mkt.year==PROJ_YEARS[-1])&(mkt.metric=="usd_nominal")]
        meds.append(row["median"].values[0]); p10s.append(row["p10"].values[0]); p90s.append(row["p90"].values[0])
    x=np.arange(len(ord_crops))
    bcolors=[LEK_GREEN if i<2 else LEK_LIGHT_GRAY for i in range(len(ord_crops))]
    ax.bar(x,meds,color=bcolors,edgecolor=LEK_BG,width=0.6)
    ax.errorbar(x,meds,yerr=[np.array(meds)-np.array(p10s),np.array(p90s)-np.array(meds)],
                fmt="none",color=LEK_DARK_GREEN,capsize=6,lw=1.8)
    ax.set_xticks(x); ax.set_xticklabels(ord_crops,rotation=15,ha="right",fontsize=8)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v,_:f"{v/1e6:.1f}M" if v>=1e6 else f"{v:,.0f}"))
    _lek_ax(ax,f"Potencial de Mercado Global – {PROJ_YEARS[-1]} (USD Nominal)",subtitle="Mediana ± P10–P90  |  📊 Demanda × ⚠️ Preço OU+Macro")

    ax=fig.add_subplot(gs[1,2]); meds_brl=[]
    for lbl in ord_crops:
        mkt=mc[lbl]["potencial"]; row=mkt[(mkt.year==PROJ_YEARS[-1])&(mkt.metric=="brl_nominal")]
        meds_brl.append(row["median"].values[0])
    bc2=[LEK_DARK_GREEN if i<2 else "#90A4AE" for i in range(len(ord_crops))]
    ax.barh(ord_crops,meds_brl,color=bc2,edgecolor=LEK_BG,height=0.6); ax.invert_yaxis()
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v,_:f"{v/1e6:.1f}M"))
    _lek_ax(ax,f"Potencial BRL {PROJ_YEARS[-1]}",subtitle="📊 USD × ⚠️ câmbio GBM")

    for col_i,(col_key,title,marker) in enumerate([
        ("market_share_producao_pct","Market Share – Produção (%)","o"),
        ("market_share_exportacao_pct","Market Share – Exportação (%)","s"),
        ("stocks_to_use_ratio_pct","Stocks-to-Use Ratio (%)","^")]):
        ax=fig.add_subplot(gs[2,col_i])
        for i,lbl in enumerate(crops):
            s=get_crop_series(df_base,lbl.split("(")[0].strip(),col_key)
            s=s[s.index>=HIST_START]
            ax.plot(s.index,s.values,lw=2,label=lbl[:8],color=LEK_PALETTE[i],marker=marker,markersize=3)
        _lek_leg(ax); _lek_ax(ax,title,"Ano","%",subtitle="📊 USDA PSD + MDIC – base")

    _wm(fig); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"  ✓ {save.name}")

def fig5_top2(mc,top2,save):
    fig,axes=plt.subplots(2,2,figsize=(15,11))
    _lek_fig(fig,f"Distribuições Probabilísticas – Top 2 Priorizadas",
             f"{top2[0]}  e  {top2[1]}  |  ano {PROJ_YEARS[-1]}  |  {N_SIMUL:,} simulações")
    for col,lbl in enumerate(top2):
        r=mc[lbl]; color=LEK_PALETTE[col]
        for row_i,(arr,title,c) in enumerate([
            (r["demand_paths"][:,-1]*r["price_paths"][:,-1],f"{lbl} – USD Nominal",color),
            (r["demand_paths"][:,-1]*r["price_paths"][:,-1]*r["fx_paths"][:,-1],f"{lbl} – BRL Nominal",LEK_DARK_GREEN)]):
            ax=axes[row_i][col]
            mu_v,p10_v,p90_v=np.mean(arr),np.percentile(arr,10),np.percentile(arr,90)
            ax.hist(arr,bins=80,color=c,edgecolor=LEK_BG,alpha=0.85)
            ax.axvline(mu_v,color=LEK_DARK_GREEN if row_i==0 else LEK_GREEN,lw=2.5,label=f"Média: {mu_v:,.0f}")
            ax.axvline(p10_v,color=LEK_GRAY,lw=1.5,ls="--",label=f"P10: {p10_v:,.0f}")
            ax.axvline(p90_v,color=LEK_GRAY,lw=1.5,ls="--",label=f"P90: {p90_v:,.0f}")
            ci=(p90_v-p10_v)/mu_v*100
            ax.text(0.97,0.95,f"IC P10–P90\n±{ci/2:.0f}% da média",transform=ax.transAxes,ha="right",va="top",
                    fontsize=8,color=LEK_GRAY,bbox=dict(boxstyle="round,pad=0.3",fc=LEK_WHITE,ec=LEK_LIGHT_GRAY,alpha=0.9))
            _lek_leg(ax); _lek_ax(ax,title,subtitle="📊 Demanda × ⚠️ Preço × ⚠️ Câmbio")
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.93]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"  ✓ {save.name}")

def fig6_sensitivity(mc,top2,save):
    fig,axes=plt.subplots(1,2,figsize=(15,6))
    _lek_fig(fig,"Análise de Sensibilidade Macro – Top 2 Culturas",
             "⚠️ [P-07][P-08][P-09][P-11]  |  Impacto de Fed Funds e VIX no preço projetado")
    fed_range=np.linspace(1.0,6.5,25)
    for i,lbl in enumerate(top2):
        _,_,theta,_,last_p=fit_ou(mc[lbl]["price_hist"],lbl)
        prices=[theta*np.exp(FED_PRICE_IMPACT*(f-last_p)*HORIZON) for f in fed_range]
        axes[0].plot(fed_range,prices,lw=2.5,label=lbl,color=LEK_PALETTE[i])
    axes[0].axvline(FED_FUNDS_PROJ[PROJ_YEARS[-1]],color=LEK_GRAY,lw=1.5,ls="--",
                    label=f"Fed proj.{PROJ_YEARS[-1]}: {FED_FUNDS_PROJ[PROJ_YEARS[-1]]}% ⚠️[P-09]")
    _lek_leg(axes[0]); _lek_ax(axes[0],"Preço de Equilíbrio (θ) vs. Fed Funds","Fed Funds (%)","Preço",
                                subtitle="⚠️ Impacto: -1.5% por 1pp Fed [P-08]")
    vix_range=np.linspace(8,50,25)
    for i,lbl in enumerate(top2):
        _,sig_p,_,_,_=fit_ou(mc[lbl]["price_hist"],lbl)
        spread=(sig_p+VIX_SENSITIVITY[lbl]*vix_range)*np.sqrt(HORIZON)*200
        axes[1].plot(vix_range,spread,lw=2.5,label=lbl,color=LEK_PALETTE[i])
    axes[1].axvline(VIX_PROJ,color=LEK_GRAY,lw=1.5,ls="--",label=f"VIX proj.: {VIX_PROJ} ⚠️[P-11]")
    _lek_leg(axes[1]); _lek_ax(axes[1],"Spread P90–P10 do Preço vs. VIX","VIX Médio","Spread % (~±1σ)",
                                subtitle="⚠️ β_vix por cultura [P-07]")
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.93]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"  ✓ {save.name}")

# ── PIPELINE ──────────────────────────────────────────────────────────────────
def open_image(p):
    try:
        if sys.platform.startswith("darwin"): subprocess.Popen(["open",str(p)])
        elif sys.platform.startswith("win"):  os.startfile(str(p))
        else: subprocess.Popen(["xdg-open",str(p)])
    except: pass

def run():
    print("="*70); print("  L.E.K. CONSULTING 2026 – Monte Carlo Engine"); print("="*70)
    df=load_base()
    print(f"\n[1/7] Base: {len(df)} linhas  |  {df['cultura'].nunique()} culturas")
    fx_hist=get_macro_series(df,"usd_brl")[lambda s:s.index>=HIST_START]
    cpi_hist=get_macro_series(df,"us_cpi_index")[lambda s:s.index>=HIST_START]
    vix_hist=get_macro_series(df,"vix_avg")[lambda s:s.index>=HIST_START]
    fed_hist=get_macro_series(df,"us_fed_funds_rate_pct")[lambda s:s.index>=HIST_START]
    vix_last=float(vix_hist.iloc[-1]); fed_last=float(fed_hist.iloc[-1]); cpi_last=float(cpi_hist.iloc[-1])
    print(f"      Macro 2024: BRL={fx_hist.iloc[-1]:.3f}  CPI={cpi_last:.0f}  VIX={vix_last:.1f}  Fed={fed_last:.2f}%")
    print(f"\n[2/7] Projetando macro ({N_SIMUL:,} sim)…")
    fx_paths,mu_fx,sig_fx=sim_fx(fx_hist); cpi_paths=sim_cpi(cpi_last)
    print("\n[3/7] Simulando culturas…")
    CSEARCH={"Milho":"Milho","Cafe (Arabica)":"Cafe","Cana-de-Acucar (Acucar)":"Cana",
             "Citros (OJ_Laranja)":"Citros","Trigo":"Trigo"}
    mc={}
    for lbl,pat in CSEARCH.items():
        print(f"  → {lbl}")
        dh=get_crop_series(df,pat,"consumo_mundial_1000mt"); dh=dh[dh.index>=HIST_START].dropna()
        ph=get_crop_series(df,pat,"preco_usd_unidade"); ph=ph[ph.index>=HIST_START].dropna()
        pr=get_crop_series(df,pat,"preco_real_usd2024_unidade"); pr=pr[pr.index>=HIST_START].dropna()
        pu=df[df["cultura"].str.contains(pat,case=False,na=False)]["unidade_preco"].iloc[0]
        d_p,mu_d,sig_d=sim_demand(dh)
        p_p,mu_p,sig_p,theta,sig_adj=sim_price(ph,lbl,fed_last)
        pot=calc_pot(d_p,p_p,fx_paths,cpi_paths,cpi_last)
        mc[lbl]={"demand_paths":d_p,"price_paths":p_p,"fx_paths":fx_paths,"cpi_paths":cpi_paths,
                 "demand_hist":dh,"price_hist":ph,"price_hist_real":pr,
                 "mu_demand":mu_d,"sigma_demand":sig_d,"mu_price":mu_p,"sigma_price":sig_p,
                 "sigma_adj":sig_adj,"theta_price":theta,"price_unit":pu,"potencial":pot}
    print("\n[4/7] Scoring multicritério…")
    df_score=build_scoring(df,mc)
    top2=df_score[df_score["Ranking"]<=2]["Cultura"].tolist()
    print("\n  RANKING:")
    for _,row in df_score.iterrows():
        star="  ★" if row["Ranking"]<=2 else "   "
        print(f"  #{int(row['Ranking'])} {row['Cultura']:<30} {row['Score_Final']:.2f}/10{star}")
    print(f"\n  Priorizadas: {top2[0]}  e  {top2[1]}")
    print("\n[5/7] Gerando figuras (estilo L.E.K.)…")
    figs={"fig1":OUTPUT_DIR/"fig1_demand_mc.png","fig2":OUTPUT_DIR/"fig2_price_mc.png",
          "fig3":OUTPUT_DIR/"fig3_fx_cpi.png","fig4":OUTPUT_DIR/"fig4_dashboard.png",
          "fig5":OUTPUT_DIR/"fig5_top2_distributions.png","fig6":OUTPUT_DIR/"fig6_macro_sensitivity.png"}
    fig1_demand(mc,figs["fig1"]); fig2_price(mc,figs["fig2"]); fig3_fx_cpi(mc,df,figs["fig3"])
    fig4_dashboard(df,mc,df_score,figs["fig4"]); fig5_top2(mc,top2,figs["fig5"]); fig6_sensitivity(mc,top2,figs["fig6"])
    print("\n[6/7] Exportando dados…")
    rows=[]
    for lbl,r in mc.items():
        for t in range(HORIZON):
            dp,pp,fp=r["demand_paths"][:,t],r["price_paths"][:,t],r["fx_paths"][:,t]
            mu=dp*pp; mb=mu*fp
            rows.append({"Cultura":lbl,"Ano":PROJ_YEARS[t],
                         "Dem_P10":np.percentile(dp,10),"Dem_Med":np.median(dp),"Dem_P90":np.percentile(dp,90),
                         "Preco_P10":np.percentile(pp,10),"Preco_Med":np.median(pp),"Preco_P90":np.percentile(pp,90),
                         "FX_Med":np.median(fp),"Pot_USD_Med":np.median(mu),"Pot_BRL_Med":np.median(mb)})
    pd.DataFrame(rows).to_csv(OUTPUT_DIR/"projecoes_mc.csv",index=False,encoding="utf-8-sig",float_format="%.3f")
    score_cols=["Ranking","Cultura","Score_Final"]+[f"score_{c}" for c in PESOS]
    df_score[score_cols].to_csv(OUTPUT_DIR/"scoring_mc.csv",index=False,encoding="utf-8-sig")
    summary={"top2":top2,"premissas":{"P-01":f"N_SIMUL={N_SIMUL}","P-02":f"HORIZON={HORIZON}",
             "P-03":f"HIST_START={HIST_START}","P-04":str(KAPPA),"P-05":str(PESOS),
             "P-06":f"fallback mu={FALLBACK_MU} sigma={FALLBACK_SIGMA}","P-07":str(VIX_SENSITIVITY),
             "P-08":f"FED_PRICE_IMPACT={FED_PRICE_IMPACT}","P-09":str(FED_FUNDS_PROJ),
             "P-10":f"CPI_GROWTH={CPI_GROWTH_PROJ}","P-11":f"VIX_PROJ={VIX_PROJ}"},
             "scoring":df_score[["Ranking","Cultura","Score_Final"]].to_dict("records")}
    with open(OUTPUT_DIR/"resumo_mc.json","w",encoding="utf-8") as f: json.dump(summary,f,ensure_ascii=False,indent=2)
    print("      ✓ projecoes_mc.csv  |  scoring_mc.csv  |  resumo_mc.json")
    print("\n[7/7] Abrindo imagens…")
    for p in figs.values(): open_image(p)
    print("\n"+"="*70); print(f"  Top 2: {top2[0]}  e  {top2[1]}"); print("="*70)
    return mc,df_score,summary

if __name__=="__main__":
    mc,df_score,summary=run()

  L.E.K. CONSULTING 2026 – Monte Carlo Engine

[1/7] Base: 100 linhas  |  5 culturas
      Macro 2024: BRL=5.100  CPI=312  VIX=15.8  Fed=5.25%

[2/7] Projetando macro (10,000 sim)…

[3/7] Simulando culturas…
  → Milho
  → Cafe (Arabica)
  → Cana-de-Acucar (Acucar)
  → Citros (OJ_Laranja)
  → Trigo

[4/7] Scoring multicritério…

  RANKING:
  #1 Milho                          5.79/10  ★
  #2 Cana-de-Acucar (Acucar)        4.31/10  ★
  #3 Trigo                          4.26/10   
  #4 Cafe (Arabica)                 4.25/10   
  #5 Citros (OJ_Laranja)            2.96/10   

  Priorizadas: Milho  e  Cana-de-Acucar (Acucar)

[5/7] Gerando figuras (estilo L.E.K.)…
  ✓ fig1_demand_mc.png
  ✓ fig2_price_mc.png
  ✓ fig3_fx_cpi.png
  ✓ fig4_dashboard.png
  ✓ fig5_top2_distributions.png
  ✓ fig6_macro_sensitivity.png

[6/7] Exportando dados…
      ✓ projecoes_mc.csv  |  scoring_mc.csv  |  resumo_mc.json

[7/7] Abrindo imagens…

  Top 2: Milho  e  Cana-de-Acucar (Acucar)


## Modelo 4


In [ ]:
"""
=============================================================================
MONTE_CARLO_LEK2026.PY  –  Modelo Monte Carlo com identidade visual L.E.K.
=============================================================================
Lê:    base_commodities_LEK2026.xlsx
Gera:  6 figuras PNG  +  projecoes_mc.csv  +  scoring_mc.csv  +  resumo_mc.json
LEGENDA:  📊 DA BASE = valor lido/calibrado da base   |   ⚠️ PREMISSA = assumido externamente
=============================================================================
"""
import warnings; warnings.filterwarnings("ignore")
import os, sys, subprocess, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from pathlib import Path

BASE_PATH  = Path("base_commodities_LEK2026.xlsx")
OUTPUT_DIR = Path("resultados_lek")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42)

# ── IDENTIDADE VISUAL L.E.K. ─────────────────────────────────────────────────
LEK_GREEN       = "#00A651"
LEK_DARK_GREEN  = "#004D2C"
LEK_LIGHT_GREEN = "#C8E6C9"
LEK_NAVY        = "#1A2744"
LEK_GRAY        = "#6E7B8B"
LEK_LIGHT_GRAY  = "#E8ECEF"
LEK_WHITE       = "#FFFFFF"
LEK_BG          = "#F8F9FA"
LEK_PALETTE     = ["#00A651","#1A2744","#2196F3","#FF6F00","#6A1B9A"]
COLOR_HIST      = LEK_DARK_GREEN
COLOR_MACRO     = "#FF6F00"

plt.rcParams.update({
    "font.family":"DejaVu Sans","font.size":9,
    "axes.titlesize":10,"axes.labelsize":9,
    "xtick.labelsize":8,"ytick.labelsize":8,"legend.fontsize":8,
    "figure.facecolor":LEK_BG,"axes.facecolor":LEK_BG,
    "axes.edgecolor":LEK_LIGHT_GRAY,"axes.grid":True,
    "grid.color":LEK_LIGHT_GRAY,"grid.linewidth":0.6,
    "axes.spines.top":False,"axes.spines.right":False,
    "text.color":LEK_NAVY,"axes.labelcolor":LEK_NAVY,
    "xtick.color":LEK_GRAY,"ytick.color":LEK_GRAY,
})

PROJ_YEAR  = 2024
HORIZON    = 5                                                  # ⚠️ PREMISSA [P-02]
PROJ_YEARS = list(range(PROJ_YEAR+1, PROJ_YEAR+HORIZON+1))

# ── PREMISSAS EXTERNAS ───────────────────────────────────────────────────────
N_SIMUL    = 10_000   # ⚠️ [P-01]
HIST_START = 2005     # ⚠️ [P-03]
KAPPA = {"Milho":0.20,"Cafe (Arabica)":0.15,"Cana-de-Acucar (Acucar)":0.20,
         "Citros (OJ_Laranja)":0.25,"Trigo":0.20}              # ⚠️ [P-04]
PESOS = {"tamanho_mercado_usd":0.20,"crescimento_demanda":0.18,
         "posicao_brasil_producao":0.15,"posicao_brasil_export":0.12,
         "tendencia_preco":0.12,"volatilidade_preco":0.08,
         "yield_trend_brasil":0.10,"stocks_to_use":0.05}       # ⚠️ [P-05]
assert abs(sum(PESOS.values())-1.0)<1e-9
FALLBACK_MU=0.02; FALLBACK_SIGMA=0.05                          # ⚠️ [P-06]
VIX_SENSITIVITY={"Milho":0.010,"Cafe (Arabica)":0.012,
                 "Cana-de-Acucar (Acucar)":0.008,
                 "Citros (OJ_Laranja)":0.015,"Trigo":0.010}    # ⚠️ [P-07]
FED_PRICE_IMPACT=-0.015                                        # ⚠️ [P-08]
FED_FUNDS_PROJ={2025:4.25,2026:3.50,2027:3.00,2028:3.00,2029:3.00}  # ⚠️ [P-09]
CPI_GROWTH_PROJ=0.025                                          # ⚠️ [P-10]
VIX_PROJ=18.0                                                  # ⚠️ [P-11]

# ── BASE ──────────────────────────────────────────────────────────────────────
def load_base():
    df = pd.read_excel(BASE_PATH, sheet_name="CONSOLIDADO", header=1)
    df.columns = df.columns.str.lower().str.strip().str.replace(" ","_")
    return df

def get_crop_series(df, pat, col):
    mask=df["cultura"].str.contains(pat,case=False,na=False)
    return pd.to_numeric(df[mask].sort_values("ano").set_index("ano")[col],errors="coerce").dropna()

def get_macro_series(df, col):
    mask=df["cultura"].str.contains("Milho",case=False,na=False)
    return pd.to_numeric(df[mask].sort_values("ano").set_index("ano")[col],errors="coerce").dropna()

# ── CALIBRAÇÃO ────────────────────────────────────────────────────────────────
def fit_gbm(s):
    s=s[s.index>=HIST_START].dropna()
    if len(s)<5: return FALLBACK_MU,FALLBACK_SIGMA           # ⚠️ [P-06]
    lr=np.log(s/s.shift(1)).dropna()
    return float(lr.mean()),float(lr.std())

def fit_ou(ps,lbl):
    s=ps[ps.index>=HIST_START].dropna()
    lr=np.log(s/s.shift(1)).dropna()
    return (float(lr.mean()),float(lr.std()),
            float(np.exp(np.log(s).mean())),
            KAPPA[lbl],float(s.iloc[-1]))                    # κ ⚠️ [P-04]

def fit_fx(s):
    s=s[s.index>=HIST_START].dropna()
    lr=np.log(s/s.shift(1)).dropna()
    return float(lr.mean()),float(lr.std()),float(s.iloc[-1])

# ── SIMULAÇÕES ────────────────────────────────────────────────────────────────
def sim_demand(s):
    mu,sig=fit_gbm(s); last=float(s.dropna().iloc[-1])
    Z=np.random.normal(0,1,(N_SIMUL,HORIZON))               # ⚠️ [P-01][P-02]
    return last*np.exp(np.cumsum((mu-0.5*sig**2)+sig*Z,axis=1)),mu,sig

def sim_price(ps,lbl,fed_last):
    mu_p,sig_p,theta,kappa,last_p=fit_ou(ps,lbl)
    sig_adj=sig_p+VIX_SENSITIVITY[lbl]*VIX_PROJ             # ⚠️ [P-07][P-11]
    prices=np.full(N_SIMUL,last_p)
    paths=np.zeros((N_SIMUL,HORIZON))
    for t in range(HORIZON):
        yr=PROJ_YEAR+t+1
        mac=FED_PRICE_IMPACT*(FED_FUNDS_PROJ.get(yr,3.0)-fed_last)  # ⚠️ [P-08][P-09]
        Z=np.random.normal(0,1,N_SIMUL)
        prices=(prices+kappa*(theta-prices)+mac*prices+sig_adj*prices*Z)
        prices=np.clip(prices,0.01,None)
        paths[:,t]=prices
    return paths,mu_p,sig_p,theta,sig_adj

def sim_fx(s):
    mu,sig,last=fit_fx(s)
    Z=np.random.normal(0,1,(N_SIMUL,HORIZON))
    return last*np.exp(np.cumsum((mu-0.5*sig**2)+sig*Z,axis=1)),mu,sig

def sim_cpi(last_cpi):
    paths=np.zeros((N_SIMUL,HORIZON)); c=last_cpi
    for t in range(HORIZON):
        c*=(1+CPI_GROWTH_PROJ)                               # ⚠️ [P-10]
        paths[:,t]=c
    return paths

def calc_pot(d,p,fx,cpi,cpi0):
    mkt_u=d*p; mkt_b=mkt_u*fx; mkt_r=mkt_u*(cpi0/cpi)      # ⚠️ [P-10]
    rows=[]
    for t in range(HORIZON):
        for arr,lbl in [(mkt_u[:,t],"usd_nominal"),(mkt_b[:,t],"brl_nominal"),(mkt_r[:,t],"usd_real_2024")]:
            rows.append({"year":PROJ_YEAR+t+1,"metric":lbl,
                         "mean":np.mean(arr),"median":np.median(arr),
                         "p10":np.percentile(arr,10),"p90":np.percentile(arr,90),"std":np.std(arr)})
    return pd.DataFrame(rows)

# ── SCORING ───────────────────────────────────────────────────────────────────
def norm(d,inv=False):
    v=np.array(list(d.values()),dtype=float); mn,mx=v.min(),v.max()
    if mx==mn: return {k:5.0 for k in d}
    return ({k:1+9*(mx-v)/(mx-mn) for k,v in d.items()} if inv
            else {k:1+9*(v-mn)/(mx-mn) for k,v in d.items()})

def build_scoring(df,mc):
    crops=list(mc.keys()); raw={c:{} for c in PESOS}
    for lbl in crops:
        pat=lbl.split("(")[0].strip()
        mkt=mc[lbl]["potencial"]
        raw["tamanho_mercado_usd"][lbl]=mkt[(mkt.year==PROJ_YEARS[-1])&(mkt.metric=="usd_nominal")]["median"].values[0]
        raw["crescimento_demanda"][lbl]=mc[lbl]["mu_demand"]
        for col_key,raw_key in [("market_share_producao_pct","posicao_brasil_producao"),
                                 ("market_share_exportacao_pct","posicao_brasil_export"),
                                 ("stocks_to_use_ratio_pct","stocks_to_use")]:
            s=get_crop_series(df,pat,col_key); s=s[s.index>=HIST_START]
            raw[raw_key][lbl]=float(s.iloc[-3:].mean()) if len(s)>=3 else 0.0
        pr=mc[lbl]["price_hist_real"]
        if len(pr)>=5:
            sl,_=np.polyfit(np.arange(len(pr)),pr.values,1)
            raw["tendencia_preco"][lbl]=sl/float(pr.mean())
        else: raw["tendencia_preco"][lbl]=0.0
        raw["volatilidade_preco"][lbl]=mc[lbl]["sigma_price"]
        yld=get_crop_series(df,pat,"yield_brasil_mt_ha"); yld=yld[yld.index>=HIST_START].dropna()
        if len(yld)>=5:
            sl,_=np.polyfit(np.arange(len(yld)),yld.values,1)
            raw["yield_trend_brasil"][lbl]=sl/float(yld.mean())
        else: raw["yield_trend_brasil"][lbl]=0.0
    n2={
        "tamanho_mercado_usd":norm(raw["tamanho_mercado_usd"]),
        "crescimento_demanda":norm(raw["crescimento_demanda"]),
        "posicao_brasil_producao":norm(raw["posicao_brasil_producao"]),
        "posicao_brasil_export":norm(raw["posicao_brasil_export"]),
        "tendencia_preco":norm(raw["tendencia_preco"]),
        "volatilidade_preco":norm(raw["volatilidade_preco"],inv=True),
        "yield_trend_brasil":norm(raw["yield_trend_brasil"]),
        "stocks_to_use":norm(raw["stocks_to_use"],inv=True),
    }
    records=[]
    for lbl in crops:
        score=sum(PESOS[c]*n2[c].get(lbl,5.0) for c in PESOS)   # ⚠️ [P-05]
        row={"Cultura":lbl}
        for c in PESOS: row[f"score_{c}"]=round(n2[c].get(lbl,5.0),2); row[f"raw_{c}"]=round(raw[c].get(lbl,0.0),4)
        row["Score_Final"]=round(score,2); records.append(row)
    df_o=(pd.DataFrame(records).sort_values("Score_Final",ascending=False).reset_index(drop=True))
    df_o["Ranking"]=df_o.index+1
    return df_o

# ── HELPERS VISUAIS L.E.K. ────────────────────────────────────────────────────
def _lek_ax(ax,title,xlabel=None,ylabel=None,subtitle=None):
    full=title+(f"\n{subtitle}" if subtitle else "")
    ax.set_title(full,fontsize=9,fontweight="bold",color=LEK_NAVY,pad=6,loc="left")
    ax.set_facecolor(LEK_BG)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(LEK_LIGHT_GRAY); ax.spines["bottom"].set_color(LEK_LIGHT_GRAY)
    ax.grid(True,color=LEK_LIGHT_GRAY,linewidth=0.6,axis="y"); ax.set_axisbelow(True)
    if xlabel: ax.set_xlabel(xlabel,fontsize=8,color=LEK_GRAY)
    if ylabel: ax.set_ylabel(ylabel,fontsize=8,color=LEK_GRAY)

def _lek_fig(fig,title,subtitle=""):
    fig.patch.set_facecolor(LEK_BG)
    bar=fig.add_axes([0,0.975,1,0.015]); bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])
    fig.suptitle(title,fontsize=13,fontweight="bold",color=LEK_NAVY,y=0.968,x=0.02,ha="left")
    if subtitle: fig.text(0.02,0.952,subtitle,fontsize=8,color=LEK_GRAY,ha="left",va="top")

def _lek_leg(ax):
    return ax.legend(fontsize=7,framealpha=0.9,facecolor=LEK_WHITE,
                     edgecolor=LEK_LIGHT_GRAY,labelcolor=LEK_NAVY)

def _wm(fig):
    fig.text(0.98,0.01,"L.E.K. Consulting  |  Análise Estratégica 2026",
             fontsize=7,color=LEK_GRAY,ha="right",va="bottom",fontstyle="italic")

def fan(ax,hist_s,paths,color):
    proj=PROJ_YEARS
    
    # 1. Plota o Histórico
    ax.plot(list(hist_s.index),hist_s.values,color=COLOR_HIST,lw=2.2,label="Histórico",zorder=3)
    
    # 2. Amostragem de caminhos para o estilo "Spaghetti" (trajetórias individuais)
    # Puxa 150 caminhos aleatórios para não sobrecarregar a renderização
    n_amostras = min(150, paths.shape[0])
    amostra_idx = np.random.choice(paths.shape[0], n_amostras, replace=False)
    
    # Conecta o último ponto real com o primeiro projetado para o gráfico não ter "buraco"
    ultimo_ano = hist_s.index[-1]
    ultimo_valor = hist_s.values[-1]
    x_spaghetti = [ultimo_ano] + proj
    
    # Plota as linhas individuais usando a cor da L.E.K. correspondente à cultura
    for idx in amostra_idx:
        y_spaghetti = [ultimo_valor] + list(paths[idx, :])
        # alpha baixo (0.15) cria o efeito visual sobreposto sem esconder tudo
        ax.plot(x_spaghetti, y_spaghetti, color=color, alpha=0.15, lw=0.8, zorder=1)
        
    # 3. Plota a mediana em destaque (tracejado escuro por cima de tudo)
    ax.plot(proj,np.median(paths,axis=0),color=LEK_NAVY,lw=2.5,ls="--",label="Mediana",zorder=4)
    ax.axvline(PROJ_YEAR,color=LEK_GRAY,lw=0.8,ls=":",alpha=0.7)
# ── FIGURAS ───────────────────────────────────────────────────────────────────
def fig1_demand(mc,save):
    crops=list(mc.keys())
    fig,axes=plt.subplots(2,3,figsize=(19,11))
    _lek_fig(fig,"Monte Carlo – Demanda Mundial por Cultura",
             f"{N_SIMUL:,} simulações  |  GBM calibrado na base USDA PSD  |  Horizonte {PROJ_YEARS[0]}–{PROJ_YEARS[-1]}")
    for i,lbl in enumerate(crops):
        ax=axes[i//3][i%3]; r=mc[lbl]
        fan(ax,r["demand_hist"],r["demand_paths"],LEK_PALETTE[i])
        _lek_ax(ax,lbl,"Ano","1.000 MT",subtitle=f"μ={r['mu_demand']*100:+.1f}%  σ={r['sigma_demand']*100:.1f}%  (📊 USDA PSD)")
        if i==0: _lek_leg(ax)
    axes[1][2].set_visible(False); _wm(fig)
    plt.tight_layout(rect=[0,0.02,1,0.94]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"  ✓ {save.name}")

def fig2_price(mc,save):
    crops=list(mc.keys())
    fig,axes=plt.subplots(2,3,figsize=(19,11))
    _lek_fig(fig,"Monte Carlo – Preço por Cultura (Ornstein-Uhlenbeck + Macro)",
             "⚠️ κ=[P-04]  |  ⚠️ β_vix=[P-07]  |  ⚠️ VIX=[P-11]  |  ⚠️ Fed=[P-08][P-09]")
    for i,lbl in enumerate(crops):
        ax=axes[i//3][i%3]; r=mc[lbl]
        fan(ax,r["price_hist"],r["price_paths"],LEK_PALETTE[i])
        ax.axhline(r["theta_price"],color=LEK_GRAY,lw=1.2,ls=":",alpha=0.8,label=f"θ={r['theta_price']:.2f}")
        _lek_ax(ax,lbl,"Ano",r["price_unit"],
                subtitle=f"θ={r['theta_price']:.2f}  κ={KAPPA[lbl]:.2f}⚠️  σ_adj={r['sigma_adj']:.3f}⚠️")
        if i==0: _lek_leg(ax)
    axes[1][2].set_visible(False); _wm(fig)
    plt.tight_layout(rect=[0,0.02,1,0.94]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"  ✓ {save.name}")

def fig3_fx_cpi(mc,df_base,save):
    fx_hist=get_macro_series(df_base,"usd_brl")[lambda s:s.index>=HIST_START]
    cpi_hist=get_macro_series(df_base,"us_cpi_index")[lambda s:s.index>=HIST_START]
    fx_paths=mc[list(mc.keys())[0]]["fx_paths"]
    fig,axes=plt.subplots(1,2,figsize=(15,6))
    _lek_fig(fig,"Projeção de Variáveis Macroeconômicas",
             "📊 Câmbio GBM calibrado na base (BCB SGS)  |  ⚠️ CPI crescimento constante [P-10]")
    fan(axes[0],fx_hist,fx_paths,COLOR_MACRO)
    _lek_ax(axes[0],"Câmbio USD/BRL","Ano","R$/US$",subtitle="📊 Série BCB SGS – GBM calibrado"); _lek_leg(axes[0])
    last_cpi=float(cpi_hist.iloc[-1])
    proj=[last_cpi*(1+CPI_GROWTH_PROJ)**(t+1) for t in range(HORIZON)]
    lo=[last_cpi*(1+CPI_GROWTH_PROJ-0.005)**(t+1) for t in range(HORIZON)]
    hi=[last_cpi*(1+CPI_GROWTH_PROJ+0.005)**(t+1) for t in range(HORIZON)]
    axes[1].plot(list(cpi_hist.index),cpi_hist.values,color=COLOR_HIST,lw=2.2,label="Histórico (📊 BLS)")
    axes[1].plot(PROJ_YEARS,proj,color=LEK_GREEN,lw=2.5,ls="--",label=f"Proj. {CPI_GROWTH_PROJ*100:.1f}%a.a. ⚠️[P-10]")
    axes[1].fill_between(PROJ_YEARS,lo,hi,alpha=0.2,color=LEK_GREEN,label="±0.5pp ⚠️[P-10]")
    axes[1].axvline(PROJ_YEAR,color=LEK_GRAY,lw=0.8,ls=":",alpha=0.7)
    _lek_ax(axes[1],"US CPI Index (BLS CPI-U)","Ano","Índice",subtitle="⚠️ Crescimento projetado = premissa [P-10]"); _lek_leg(axes[1])
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.93]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"  ✓ {save.name}")

def fig4_dashboard(df_base,mc,df_score,save):
    fig=plt.figure(figsize=(22,16)); fig.patch.set_facecolor(LEK_BG)
    gs=gridspec.GridSpec(3,3,figure=fig,hspace=0.50,wspace=0.38,top=0.90,bottom=0.06,left=0.06,right=0.97)
    bar=fig.add_axes([0,0.955,1,0.015]); bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])
    fig.suptitle("L.E.K. Consulting  |  Dashboard Estratégico de Culturas  |  2026",
                 fontsize=14,fontweight="bold",color=LEK_NAVY,y=0.975,x=0.02,ha="left")
    fig.text(0.02,0.952,"📊=base calibrada  |  ⚠️=premissa externa  |  GBM + Ornstein-Uhlenbeck + Macro",
             fontsize=8,color=LEK_GRAY,ha="left")
    crops=list(mc.keys())

    ax=fig.add_subplot(gs[0,:2])
    colors=[LEK_GREEN if r<=2 else LEK_LIGHT_GRAY for r in df_score["Ranking"]]
    bars=ax.barh(df_score["Cultura"],df_score["Score_Final"],color=colors,edgecolor=LEK_BG,height=0.6)
    ax.set_xlim(0,10.5)
    for b,v in zip(bars,df_score["Score_Final"]):
        ax.text(b.get_width()+0.12,b.get_y()+b.get_height()/2,f"{v:.2f}",va="center",fontsize=9,color=LEK_NAVY,fontweight="bold")
    ax.invert_yaxis(); ax.axvline(5,color=LEK_LIGHT_GRAY,lw=1,ls="--")
    ax.text(10.3,-0.5,"★ Top 2",color=LEK_GREEN,fontsize=10,fontweight="bold",ha="right")
    _lek_ax(ax,"Scoring Multicritério  (0–10)",subtitle="⚠️ Pesos [P-05]  |  8 critérios  |  Normalização min-max")

    ax=fig.add_subplot(gs[0,2]); ax.axis("off")
    tdata=[[lbl[:14],f"{mc[lbl]['mu_demand']*100:+.1f}%",f"{mc[lbl]['sigma_demand']*100:.1f}%",
            f"{mc[lbl]['sigma_adj']*100:.1f}%⚠️",f"{KAPPA[lbl]:.2f}⚠️"] for lbl in crops]
    t=ax.table(cellText=tdata,colLabels=["Cultura","μ Dem📊","σ Dem📊","σ P⚠️","κ⚠️"],loc="center",cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(8); t.scale(1,1.75)
    for (r_i,c_i),cell in t.get_celld().items():
        cell.set_edgecolor(LEK_LIGHT_GRAY)
        if r_i==0: cell.set_facecolor(LEK_DARK_GREEN); cell.set_text_props(color=LEK_WHITE,fontweight="bold")
        elif r_i%2==0: cell.set_facecolor(LEK_LIGHT_GREEN)
        else: cell.set_facecolor(LEK_BG)
    ax.set_title("Parâmetros Estocásticos",fontsize=9,color=LEK_NAVY,fontweight="bold",pad=12,loc="left")

    ax=fig.add_subplot(gs[1,:2])
    ord_crops=df_score["Cultura"].tolist(); meds,p10s,p90s=[],[],[]
    for lbl in ord_crops:
        mkt=mc[lbl]["potencial"]; row=mkt[(mkt.year==PROJ_YEARS[-1])&(mkt.metric=="usd_nominal")]
        meds.append(row["median"].values[0]); p10s.append(row["p10"].values[0]); p90s.append(row["p90"].values[0])
    x=np.arange(len(ord_crops))
    bcolors=[LEK_GREEN if i<2 else LEK_LIGHT_GRAY for i in range(len(ord_crops))]
    ax.bar(x,meds,color=bcolors,edgecolor=LEK_BG,width=0.6)
    ax.errorbar(x,meds,yerr=[np.array(meds)-np.array(p10s),np.array(p90s)-np.array(meds)],
                fmt="none",color=LEK_DARK_GREEN,capsize=6,lw=1.8)
    ax.set_xticks(x); ax.set_xticklabels(ord_crops,rotation=15,ha="right",fontsize=8)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v,_:f"{v/1e6:.1f}M" if v>=1e6 else f"{v:,.0f}"))
    _lek_ax(ax,f"Potencial de Mercado Global – {PROJ_YEARS[-1]} (USD Nominal)",subtitle="Mediana ± P10–P90  |  📊 Demanda × ⚠️ Preço OU+Macro")

    ax=fig.add_subplot(gs[1,2]); meds_brl=[]
    for lbl in ord_crops:
        mkt=mc[lbl]["potencial"]; row=mkt[(mkt.year==PROJ_YEARS[-1])&(mkt.metric=="brl_nominal")]
        meds_brl.append(row["median"].values[0])
    bc2=[LEK_DARK_GREEN if i<2 else "#90A4AE" for i in range(len(ord_crops))]
    ax.barh(ord_crops,meds_brl,color=bc2,edgecolor=LEK_BG,height=0.6); ax.invert_yaxis()
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v,_:f"{v/1e6:.1f}M"))
    _lek_ax(ax,f"Potencial BRL {PROJ_YEARS[-1]}",subtitle="📊 USD × ⚠️ câmbio GBM")

    for col_i,(col_key,title,marker) in enumerate([
        ("market_share_producao_pct","Market Share – Produção (%)","o"),
        ("market_share_exportacao_pct","Market Share – Exportação (%)","s"),
        ("stocks_to_use_ratio_pct","Stocks-to-Use Ratio (%)","^")]):
        ax=fig.add_subplot(gs[2,col_i])
        for i,lbl in enumerate(crops):
            s=get_crop_series(df_base,lbl.split("(")[0].strip(),col_key)
            s=s[s.index>=HIST_START]
            ax.plot(s.index,s.values,lw=2,label=lbl[:8],color=LEK_PALETTE[i],marker=marker,markersize=3)
        _lek_leg(ax); _lek_ax(ax,title,"Ano","%",subtitle="📊 USDA PSD + MDIC – base")

    _wm(fig); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"  ✓ {save.name}")

def fig5_top2(mc,top2,save):
    fig,axes=plt.subplots(2,2,figsize=(15,11))
    _lek_fig(fig,f"Distribuições Probabilísticas – Top 2 Priorizadas",
             f"{top2[0]}  e  {top2[1]}  |  ano {PROJ_YEARS[-1]}  |  {N_SIMUL:,} simulações")
    for col,lbl in enumerate(top2):
        r=mc[lbl]; color=LEK_PALETTE[col]
        for row_i,(arr,title,c) in enumerate([
            (r["demand_paths"][:,-1]*r["price_paths"][:,-1],f"{lbl} – USD Nominal",color),
            (r["demand_paths"][:,-1]*r["price_paths"][:,-1]*r["fx_paths"][:,-1],f"{lbl} – BRL Nominal",LEK_DARK_GREEN)]):
            ax=axes[row_i][col]
            mu_v,p10_v,p90_v=np.mean(arr),np.percentile(arr,10),np.percentile(arr,90)
            ax.hist(arr,bins=80,color=c,edgecolor=LEK_BG,alpha=0.85)
            ax.axvline(mu_v,color=LEK_DARK_GREEN if row_i==0 else LEK_GREEN,lw=2.5,label=f"Média: {mu_v:,.0f}")
            ax.axvline(p10_v,color=LEK_GRAY,lw=1.5,ls="--",label=f"P10: {p10_v:,.0f}")
            ax.axvline(p90_v,color=LEK_GRAY,lw=1.5,ls="--",label=f"P90: {p90_v:,.0f}")
            ci=(p90_v-p10_v)/mu_v*100
            ax.text(0.97,0.95,f"IC P10–P90\n±{ci/2:.0f}% da média",transform=ax.transAxes,ha="right",va="top",
                    fontsize=8,color=LEK_GRAY,bbox=dict(boxstyle="round,pad=0.3",fc=LEK_WHITE,ec=LEK_LIGHT_GRAY,alpha=0.9))
            _lek_leg(ax); _lek_ax(ax,title,subtitle="📊 Demanda × ⚠️ Preço × ⚠️ Câmbio")
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.93]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"  ✓ {save.name}")

def fig6_sensitivity(mc,top2,save):
    fig,axes=plt.subplots(1,2,figsize=(15,6))
    _lek_fig(fig,"Análise de Sensibilidade Macro – Top 2 Culturas",
             "⚠️ [P-07][P-08][P-09][P-11]  |  Impacto de Fed Funds e VIX no preço projetado")
    fed_range=np.linspace(1.0,6.5,25)
    for i,lbl in enumerate(top2):
        _,_,theta,_,last_p=fit_ou(mc[lbl]["price_hist"],lbl)
        prices=[theta*np.exp(FED_PRICE_IMPACT*(f-last_p)*HORIZON) for f in fed_range]
        axes[0].plot(fed_range,prices,lw=2.5,label=lbl,color=LEK_PALETTE[i])
    axes[0].axvline(FED_FUNDS_PROJ[PROJ_YEARS[-1]],color=LEK_GRAY,lw=1.5,ls="--",
                    label=f"Fed proj.{PROJ_YEARS[-1]}: {FED_FUNDS_PROJ[PROJ_YEARS[-1]]}% ⚠️[P-09]")
    _lek_leg(axes[0]); _lek_ax(axes[0],"Preço de Equilíbrio (θ) vs. Fed Funds","Fed Funds (%)","Preço",
                                subtitle="⚠️ Impacto: -1.5% por 1pp Fed [P-08]")
    vix_range=np.linspace(8,50,25)
    for i,lbl in enumerate(top2):
        _,sig_p,_,_,_=fit_ou(mc[lbl]["price_hist"],lbl)
        spread=(sig_p+VIX_SENSITIVITY[lbl]*vix_range)*np.sqrt(HORIZON)*200
        axes[1].plot(vix_range,spread,lw=2.5,label=lbl,color=LEK_PALETTE[i])
    axes[1].axvline(VIX_PROJ,color=LEK_GRAY,lw=1.5,ls="--",label=f"VIX proj.: {VIX_PROJ} ⚠️[P-11]")
    _lek_leg(axes[1]); _lek_ax(axes[1],"Spread P90–P10 do Preço vs. VIX","VIX Médio","Spread % (~±1σ)",
                                subtitle="⚠️ β_vix por cultura [P-07]")
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.93]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig); print(f"  ✓ {save.name}")

# ── PIPELINE ──────────────────────────────────────────────────────────────────
def open_image(p):
    try:
        if sys.platform.startswith("darwin"): subprocess.Popen(["open",str(p)])
        elif sys.platform.startswith("win"):  os.startfile(str(p))
        else: subprocess.Popen(["xdg-open",str(p)])
    except: pass

def run():
    print("="*70); print("  L.E.K. CONSULTING 2026 – Monte Carlo Engine"); print("="*70)
    df=load_base()
    print(f"\n[1/7] Base: {len(df)} linhas  |  {df['cultura'].nunique()} culturas")
    fx_hist=get_macro_series(df,"usd_brl")[lambda s:s.index>=HIST_START]
    cpi_hist=get_macro_series(df,"us_cpi_index")[lambda s:s.index>=HIST_START]
    vix_hist=get_macro_series(df,"vix_avg")[lambda s:s.index>=HIST_START]
    fed_hist=get_macro_series(df,"us_fed_funds_rate_pct")[lambda s:s.index>=HIST_START]
    vix_last=float(vix_hist.iloc[-1]); fed_last=float(fed_hist.iloc[-1]); cpi_last=float(cpi_hist.iloc[-1])
    print(f"      Macro 2024: BRL={fx_hist.iloc[-1]:.3f}  CPI={cpi_last:.0f}  VIX={vix_last:.1f}  Fed={fed_last:.2f}%")
    print(f"\n[2/7] Projetando macro ({N_SIMUL:,} sim)…")
    fx_paths,mu_fx,sig_fx=sim_fx(fx_hist); cpi_paths=sim_cpi(cpi_last)
    print("\n[3/7] Simulando culturas…")
    CSEARCH={"Milho":"Milho","Cafe (Arabica)":"Cafe","Cana-de-Acucar (Acucar)":"Cana",
             "Citros (OJ_Laranja)":"Citros","Trigo":"Trigo"}
    mc={}
    for lbl,pat in CSEARCH.items():
        print(f"  → {lbl}")
        dh=get_crop_series(df,pat,"consumo_mundial_1000mt"); dh=dh[dh.index>=HIST_START].dropna()
        ph=get_crop_series(df,pat,"preco_usd_unidade"); ph=ph[ph.index>=HIST_START].dropna()
        pr=get_crop_series(df,pat,"preco_real_usd2024_unidade"); pr=pr[pr.index>=HIST_START].dropna()
        pu=df[df["cultura"].str.contains(pat,case=False,na=False)]["unidade_preco"].iloc[0]
        d_p,mu_d,sig_d=sim_demand(dh)
        p_p,mu_p,sig_p,theta,sig_adj=sim_price(ph,lbl,fed_last)
        pot=calc_pot(d_p,p_p,fx_paths,cpi_paths,cpi_last)
        mc[lbl]={"demand_paths":d_p,"price_paths":p_p,"fx_paths":fx_paths,"cpi_paths":cpi_paths,
                 "demand_hist":dh,"price_hist":ph,"price_hist_real":pr,
                 "mu_demand":mu_d,"sigma_demand":sig_d,"mu_price":mu_p,"sigma_price":sig_p,
                 "sigma_adj":sig_adj,"theta_price":theta,"price_unit":pu,"potencial":pot}
    print("\n[4/7] Scoring multicritério…")
    df_score=build_scoring(df,mc)
    top2=df_score[df_score["Ranking"]<=2]["Cultura"].tolist()
    print("\n  RANKING:")
    for _,row in df_score.iterrows():
        star="  ★" if row["Ranking"]<=2 else "   "
        print(f"  #{int(row['Ranking'])} {row['Cultura']:<30} {row['Score_Final']:.2f}/10{star}")
    print(f"\n  Priorizadas: {top2[0]}  e  {top2[1]}")
    print("\n[5/7] Gerando figuras (estilo L.E.K.)…")
    figs={"fig1":OUTPUT_DIR/"fig1_demand_mc.png","fig2":OUTPUT_DIR/"fig2_price_mc.png",
          "fig3":OUTPUT_DIR/"fig3_fx_cpi.png","fig4":OUTPUT_DIR/"fig4_dashboard.png",
          "fig5":OUTPUT_DIR/"fig5_top2_distributions.png","fig6":OUTPUT_DIR/"fig6_macro_sensitivity.png"}
    fig1_demand(mc,figs["fig1"]); fig2_price(mc,figs["fig2"]); fig3_fx_cpi(mc,df,figs["fig3"])
    fig4_dashboard(df,mc,df_score,figs["fig4"]); fig5_top2(mc,top2,figs["fig5"]); fig6_sensitivity(mc,top2,figs["fig6"])
    print("\n[6/7] Exportando dados…")
    rows=[]
    for lbl,r in mc.items():
        for t in range(HORIZON):
            dp,pp,fp=r["demand_paths"][:,t],r["price_paths"][:,t],r["fx_paths"][:,t]
            mu=dp*pp; mb=mu*fp
            rows.append({"Cultura":lbl,"Ano":PROJ_YEARS[t],
                         "Dem_P10":np.percentile(dp,10),"Dem_Med":np.median(dp),"Dem_P90":np.percentile(dp,90),
                         "Preco_P10":np.percentile(pp,10),"Preco_Med":np.median(pp),"Preco_P90":np.percentile(pp,90),
                         "FX_Med":np.median(fp),"Pot_USD_Med":np.median(mu),"Pot_BRL_Med":np.median(mb)})
    pd.DataFrame(rows).to_csv(OUTPUT_DIR/"projecoes_mc.csv",index=False,encoding="utf-8-sig",float_format="%.3f")
    score_cols=["Ranking","Cultura","Score_Final"]+[f"score_{c}" for c in PESOS]
    df_score[score_cols].to_csv(OUTPUT_DIR/"scoring_mc.csv",index=False,encoding="utf-8-sig")
    summary={"top2":top2,"premissas":{"P-01":f"N_SIMUL={N_SIMUL}","P-02":f"HORIZON={HORIZON}",
             "P-03":f"HIST_START={HIST_START}","P-04":str(KAPPA),"P-05":str(PESOS),
             "P-06":f"fallback mu={FALLBACK_MU} sigma={FALLBACK_SIGMA}","P-07":str(VIX_SENSITIVITY),
             "P-08":f"FED_PRICE_IMPACT={FED_PRICE_IMPACT}","P-09":str(FED_FUNDS_PROJ),
             "P-10":f"CPI_GROWTH={CPI_GROWTH_PROJ}","P-11":f"VIX_PROJ={VIX_PROJ}"},
             "scoring":df_score[["Ranking","Cultura","Score_Final"]].to_dict("records")}
    with open(OUTPUT_DIR/"resumo_mc.json","w",encoding="utf-8") as f: json.dump(summary,f,ensure_ascii=False,indent=2)
    print("      ✓ projecoes_mc.csv  |  scoring_mc.csv  |  resumo_mc.json")
    print("\n[7/7] Abrindo imagens…")
    for p in figs.values(): open_image(p)
    print("\n"+"="*70); print(f"  Top 2: {top2[0]}  e  {top2[1]}"); print("="*70)
    return mc,df_score,summary

if __name__=="__main__":
    mc,df_score,summary=run()

  L.E.K. CONSULTING 2026 – Monte Carlo Engine

[1/7] Base: 100 linhas  |  5 culturas
      Macro 2024: BRL=5.100  CPI=312  VIX=15.8  Fed=5.25%

[2/7] Projetando macro (10,000 sim)…

[3/7] Simulando culturas…
  → Milho
  → Cafe (Arabica)
  → Cana-de-Acucar (Acucar)
  → Citros (OJ_Laranja)
  → Trigo

[4/7] Scoring multicritério…

  RANKING:
  #1 Milho                          5.79/10  ★
  #2 Cana-de-Acucar (Acucar)        4.31/10  ★
  #3 Trigo                          4.26/10   
  #4 Cafe (Arabica)                 4.25/10   
  #5 Citros (OJ_Laranja)            2.96/10   

  Priorizadas: Milho  e  Cana-de-Acucar (Acucar)

[5/7] Gerando figuras (estilo L.E.K.)…
  ✓ fig1_demand_mc.png
  ✓ fig2_price_mc.png
  ✓ fig3_fx_cpi.png
  ✓ fig4_dashboard.png
  ✓ fig5_top2_distributions.png
  ✓ fig6_macro_sensitivity.png

[6/7] Exportando dados…
      ✓ projecoes_mc.csv  |  scoring_mc.csv  |  resumo_mc.json

[7/7] Abrindo imagens…

  Top 2: Milho  e  Cana-de-Acucar (Acucar)


## modelo com a base nova

In [21]:
"""
=============================================================================
MONTE_CARLO_LEK2026.PY  –  Modelo Estocástico com Cholesky e KAPPA Dinâmico
=============================================================================
Lê:    Base LEK.xlsx (Sheet1)
Gera:  6 figuras PNG  +  projecoes_mc.csv  +  resumo_mc.json
=============================================================================
"""
import warnings; warnings.filterwarnings("ignore")
import os, sys, subprocess, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from pathlib import Path

BASE_PATH  = Path("Base LEK.xlsx")
OUTPUT_DIR = Path("resultados_lek_modelo_novo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42)

# ── IDENTIDADE VISUAL L.E.K. ─────────────────────────────────────────────────
LEK_GREEN       = "#00A651"
LEK_DARK_GREEN  = "#004D2C"
LEK_LIGHT_GREEN = "#C8E6C9"
LEK_NAVY        = "#1A2744"
LEK_GRAY        = "#6E7B8B"
LEK_LIGHT_GRAY  = "#E8ECEF"
LEK_WHITE       = "#FFFFFF"
LEK_BG          = "#F8F9FA"
LEK_PALETTE     = ["#00A651","#1A2744","#2196F3","#FF6F00","#6A1B9A"]
COLOR_HIST      = LEK_DARK_GREEN
COLOR_MACRO     = "#FF6F00"

plt.rcParams.update({
    "font.family":"DejaVu Sans","font.size":9,
    "axes.titlesize":10,"axes.labelsize":9,
    "xtick.labelsize":8,"ytick.labelsize":8,"legend.fontsize":8,
    "figure.facecolor":LEK_BG,"axes.facecolor":LEK_BG,
    "axes.edgecolor":LEK_LIGHT_GRAY,"axes.grid":True,
    "grid.color":LEK_LIGHT_GRAY,"grid.linewidth":0.6,
    "axes.spines.top":False,"axes.spines.right":False,
    "text.color":LEK_NAVY,"axes.labelcolor":LEK_NAVY,
    "xtick.color":LEK_GRAY,"ytick.color":LEK_GRAY,
})

PROJ_YEAR  = 2025 # Último ano consolidado da base
HORIZON    = 5
PROJ_YEARS = list(range(PROJ_YEAR+1, PROJ_YEAR+HORIZON+1))

# ── PREMISSAS EXTERNAS ───────────────────────────────────────────────────────
N_SIMUL    = 10_000
HIST_START = 2018 
FALLBACK_MU=0.02; FALLBACK_SIGMA=0.05
VIX_SENSITIVITY={"Milho":0.010,"Cafe":0.012,"Cana":0.009,"Citros":0.015,"Trigo":0.010}
FED_PRICE_IMPACT=-0.015
FED_FUNDS_PROJ={2026:3.50,2027:3.00,2028:3.00,2029:3.00,2030:3.00}
CPI_GROWTH_PROJ=0.025
VIX_PROJ=18.0

# ── BASE ──────────────────────────────────────────────────────────────────────
def load_base():
    df = pd.read_excel(BASE_PATH, sheet_name=0)
    df['ano_int'] = df['Ano'].astype(str).str.split('/').str[0].apply(pd.to_numeric, errors='coerce')
    
    # ── CORREÇÃO DA CANA ──
    df.loc[df['Commodities'] == 'Cana', 'Demanda (Global)(1000 t)'] = df.loc[df['Commodities'] == 'Cana', 'Produção (mil t)']
    
    rename_dict = {
        'Commodities': 'cultura',
        'Preço de venda da tonelada - U$D/t': 'preco_usd_unidade',
        'Demanda (Global)(1000 t)': 'consumo_mundial_1000mt',
        'MarketShareProdução (%)': 'market_share_producao_pct',
        'MarketShareExportação (%)': 'market_share_exportacao_pct',
        'Produção / HE': 'yield_brasil_mt_ha'
    }
    df = df.rename(columns=rename_dict)
    df['unidade_preco'] = "USD/t"
    
    if 'preco_real_usd2024_unidade' not in df.columns:
        df['preco_real_usd2024_unidade'] = df['preco_usd_unidade']
    return df

def get_crop_series(df, pat, col):
    mask = df["cultura"].str.contains(pat, case=False, na=False)
    return pd.to_numeric(df[mask].sort_values("ano_int").set_index("ano_int")[col], errors="coerce").dropna()

def get_macro_series(df, col):
    anos = sorted(df['ano_int'].dropna().unique())
    if not anos: anos = list(range(HIST_START, PROJ_YEAR + 1))
    
    mock_data = {
        "usd_brl": np.linspace(3.5, 5.5, len(anos)), 
        "us_cpi_index": np.linspace(250, 320, len(anos)),
        "vix_avg": np.full(len(anos), 18.0),
        "us_fed_funds_rate_pct": np.full(len(anos), 3.0)
    }
    
    if col in mock_data:
        s = pd.Series(mock_data[col], index=anos)
        return s.loc[s.index >= HIST_START]
    return pd.Series(dtype=float)

# ── CALIBRAÇÃO (COM KAPPA DINÂMICO) ───────────────────────────────────────────
def fit_gbm(s):
    s=s[s.index>=HIST_START].dropna()
    if len(s)<3: return FALLBACK_MU,FALLBACK_SIGMA
    lr=np.log(s/s.shift(1)).dropna()
    return float(lr.mean()),float(lr.std())

def fit_ou_dynamic(ps):
    s = ps[ps.index >= HIST_START].dropna()
    if len(s) < 4: 
        return FALLBACK_MU, FALLBACK_SIGMA, float(s.iloc[-1]) if len(s)>0 else 100, 0.1, float(s.iloc[-1]) if len(s)>0 else 100
    
    y = s.diff().dropna().values
    X = s.shift(1).dropna().values
    b, a = np.polyfit(X, y, 1)
    
    kappa = -b
    if kappa <= 0:
        kappa = 0.05
        theta = float(s.mean())
    else:
        theta = a / kappa
        
    sigma = np.std(y - (a + b * X)) / float(s.mean())
    return float(s.pct_change().mean()), sigma, theta, max(kappa, 0.05), float(s.iloc[-1])

def fit_fx(s):
    s=s[s.index>=HIST_START].dropna()
    lr=np.log(s/s.shift(1)).dropna()
    return float(lr.mean()),float(lr.std()),float(s.iloc[-1])

# ── SIMULAÇÕES ────────────────────────────────────────────────────────────────
def sim_demand_correlated(s, Z_corr_slice):
    mu,sig=fit_gbm(s); last=float(s.dropna().iloc[-1])
    return last*np.exp(np.cumsum((mu-0.5*sig**2)+sig*Z_corr_slice,axis=1)),mu,sig

def sim_price(ps,lbl,fed_last):
    mu_p,sig_p,theta,kappa,last_p=fit_ou_dynamic(ps)
    sig_adj=sig_p+VIX_SENSITIVITY.get(lbl, 0.010)*VIX_PROJ
    prices=np.full(N_SIMUL,last_p)
    paths=np.zeros((N_SIMUL,HORIZON))
    for t in range(HORIZON):
        yr=PROJ_YEAR+t+1
        mac=FED_PRICE_IMPACT*(FED_FUNDS_PROJ.get(yr,3.0)-fed_last)
        Z=np.random.normal(0,1,N_SIMUL)
        prices=(prices+kappa*(theta-prices)+mac*prices+sig_adj*prices*Z)
        prices=np.clip(prices,0.01,None) 
        paths[:,t]=prices
    return paths,mu_p,sig_p,theta,kappa,sig_adj

def sim_fx(s):
    mu,sig,last=fit_fx(s)
    Z=np.random.normal(0,1,(N_SIMUL,HORIZON))
    return last*np.exp(np.cumsum((mu-0.5*sig**2)+sig*Z,axis=1)),mu,sig

def sim_cpi(last_cpi):
    paths=np.zeros((N_SIMUL,HORIZON)); c=last_cpi
    for t in range(HORIZON):
        c*=(1+CPI_GROWTH_PROJ)
        paths[:,t]=c
    return paths

def calc_pot(d,p,fx,cpi,cpi0):
    mkt_u=d*p; mkt_b=mkt_u*fx; mkt_r=mkt_u*(cpi0/cpi)
    rows=[]
    for t in range(HORIZON):
        for arr,lbl in [(mkt_u[:,t],"usd_nominal"),(mkt_b[:,t],"brl_nominal"),(mkt_r[:,t],"usd_real_2024")]:
            rows.append({"year":PROJ_YEAR+t+1,"metric":lbl,
                         "mean":np.mean(arr),"median":np.median(arr),
                         "p10":np.percentile(arr,10),"p90":np.percentile(arr,90),"std":np.std(arr)})
    return pd.DataFrame(rows)

def get_top_potentials(mc):
    pots = []
    for lbl, r in mc.items():
        mkt = r["potencial"]
        val = mkt[(mkt.year == PROJ_YEARS[-1]) & (mkt.metric == "usd_nominal")]["median"].values[0]
        pots.append((lbl, val))
    pots.sort(key=lambda x: x[1], reverse=True)
    return [p[0] for p in pots]

# ── HELPERS VISUAIS L.E.K. ────────────────────────────────────────────────────
def _lek_ax(ax,title,xlabel=None,ylabel=None,subtitle=None):
    full=title+(f"\n{subtitle}" if subtitle else "")
    ax.set_title(full,fontsize=9,fontweight="bold",color=LEK_NAVY,pad=6,loc="left")
    ax.set_facecolor(LEK_BG)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(LEK_LIGHT_GRAY); ax.spines["bottom"].set_color(LEK_LIGHT_GRAY)
    ax.grid(True,color=LEK_LIGHT_GRAY,linewidth=0.6,axis="y"); ax.set_axisbelow(True)
    if xlabel: ax.set_xlabel(xlabel,fontsize=8,color=LEK_GRAY)
    if ylabel: ax.set_ylabel(ylabel,fontsize=8,color=LEK_GRAY)

def _lek_fig(fig,title,subtitle=""):
    fig.patch.set_facecolor(LEK_BG)
    bar=fig.add_axes([0,0.975,1,0.015]); bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])
    fig.suptitle(title,fontsize=13,fontweight="bold",color=LEK_NAVY,y=0.968,x=0.02,ha="left")
    if subtitle: fig.text(0.02,0.952,subtitle,fontsize=8,color=LEK_GRAY,ha="left",va="top")

def _lek_leg(ax):
    return ax.legend(fontsize=7,framealpha=0.9,facecolor=LEK_WHITE,edgecolor=LEK_LIGHT_GRAY,labelcolor=LEK_NAVY)

def _wm(fig):
    fig.text(0.98,0.01,"Análise Estratégica 2026",fontsize=7,color=LEK_GRAY,ha="right",va="bottom",fontstyle="italic")

def fan(ax,hist_s,paths,color):
    proj=PROJ_YEARS
    if len(hist_s) > 0:
        ax.plot(list(hist_s.index),hist_s.values,color=COLOR_HIST,lw=2.2,label="Histórico",zorder=3)
        ultimo_ano = hist_s.index[-1]
        ultimo_valor = hist_s.values[-1]
    else:
        ultimo_ano = PROJ_YEAR
        ultimo_valor = np.median(paths[:,0])

    n_amostras = min(150, paths.shape[0])
    amostra_idx = np.random.choice(paths.shape[0], n_amostras, replace=False)
    x_spaghetti = [ultimo_ano] + proj
    
    for idx in amostra_idx:
        y_spaghetti = [ultimo_valor] + list(paths[idx, :])
        ax.plot(x_spaghetti, y_spaghetti, color=color, alpha=0.15, lw=0.8, zorder=1)
        
    ax.plot(proj,np.median(paths,axis=0),color=LEK_NAVY,lw=2.5,ls="--",label="Mediana",zorder=4)
    ax.axvline(PROJ_YEAR,color=LEK_GRAY,lw=0.8,ls=":",alpha=0.7)

# ── FIGURAS ───────────────────────────────────────────────────────────────────
def fig1_demand(mc,save):
    crops=list(mc.keys())
    fig,axes=plt.subplots(2,3,figsize=(19,11))
    _lek_fig(fig,"Monte Carlo – Demanda Mundial por Cultura",
             f"{N_SIMUL:,} simulações  |  GBM Correlacionado (Cholesky)  |  Horizonte {PROJ_YEARS[0]}–{PROJ_YEARS[-1]}")
    for i,lbl in enumerate(crops):
        ax=axes[i//3][i%3]; r=mc[lbl]
        fan(ax,r["demand_hist"],r["demand_paths"],LEK_PALETTE[i%len(LEK_PALETTE)])
        _lek_ax(ax,lbl,"Ano","1.000 MT",subtitle=f"μ={r['mu_demand']*100:+.1f}%  σ={r['sigma_demand']*100:.1f}%")
        if i==0: _lek_leg(ax)
    for j in range(len(crops), 6): axes[j//3][j%3].set_visible(False)
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.94]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig)

def fig2_price(mc,save):
    crops=list(mc.keys())
    fig,axes=plt.subplots(2,3,figsize=(19,11))
    _lek_fig(fig,"Monte Carlo – Preço por Cultura (Ornstein-Uhlenbeck + Macro)",
             "KAPPA e THETA calibrados via regressão linear dinâmica AR(1)")
    for i,lbl in enumerate(crops):
        ax=axes[i//3][i%3]; r=mc[lbl]
        fan(ax,r["price_hist"],r["price_paths"],LEK_PALETTE[i%len(LEK_PALETTE)])
        ax.axhline(r["theta_price"],color=LEK_GRAY,lw=1.2,ls=":",alpha=0.8,label=f"θ={r['theta_price']:.2f}")
        _lek_ax(ax,lbl,"Ano",r["price_unit"],subtitle=f"θ={r['theta_price']:.2f}  κ={r['kappa']:.2f}  σ_adj={r['sigma_adj']:.3f}")
        if i==0: _lek_leg(ax)
    for j in range(len(crops), 6): axes[j//3][j%3].set_visible(False)
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.94]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig)

def fig3_fx_cpi(mc,df_base,save):
    fx_hist=get_macro_series(df_base,"usd_brl")[lambda s:s.index>=HIST_START]
    cpi_hist=get_macro_series(df_base,"us_cpi_index")[lambda s:s.index>=HIST_START]
    fx_paths=mc[list(mc.keys())[0]]["fx_paths"]
    fig,axes=plt.subplots(1,2,figsize=(15,6))
    _lek_fig(fig,"Projeção de Variáveis Macroeconômicas (Proxies)")
    fan(axes[0],fx_hist,fx_paths,COLOR_MACRO)
    _lek_ax(axes[0],"Câmbio USD/BRL","Ano","R$/US$"); _lek_leg(axes[0])
    last_cpi=float(cpi_hist.iloc[-1])
    proj=[last_cpi*(1+CPI_GROWTH_PROJ)**(t+1) for t in range(HORIZON)]
    axes[1].plot(list(cpi_hist.index),cpi_hist.values,color=COLOR_HIST,lw=2.2,label="Proxy Histórico")
    axes[1].plot(PROJ_YEARS,proj,color=LEK_GREEN,lw=2.5,ls="--",label=f"Proj. {CPI_GROWTH_PROJ*100:.1f}%a.a.")
    axes[1].axvline(PROJ_YEAR,color=LEK_GRAY,lw=0.8,ls=":",alpha=0.7)
    _lek_ax(axes[1],"US CPI Index","Ano","Índice"); _lek_leg(axes[1])
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.93]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig)

def fig4_dashboard(df_base,mc,ord_crops,save):
    fig=plt.figure(figsize=(22,12)); fig.patch.set_facecolor(LEK_BG)
    gs=gridspec.GridSpec(2,3,figure=fig,hspace=0.40,wspace=0.30,top=0.88,bottom=0.08,left=0.06,right=0.97)
    bar=fig.add_axes([0,0.955,1,0.015]); bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])
    fig.suptitle("Dashboard de Potencial de Mercado", fontsize=14,fontweight="bold",color=LEK_NAVY,y=0.975,x=0.02,ha="left")

    ax=fig.add_subplot(gs[0,:2]); meds,p10s,p90s=[],[],[]
    for lbl in ord_crops:
        mkt=mc[lbl]["potencial"]; row=mkt[(mkt.year==PROJ_YEARS[-1])&(mkt.metric=="usd_nominal")]
        meds.append(row["median"].values[0]); p10s.append(row["p10"].values[0]); p90s.append(row["p90"].values[0])
    x=np.arange(len(ord_crops))
    bcolors=[LEK_GREEN if i<2 else LEK_LIGHT_GRAY for i in range(len(ord_crops))]
    ax.bar(x,meds,color=bcolors,edgecolor=LEK_BG,width=0.6)
    ax.errorbar(x,meds,yerr=[np.array(meds)-np.array(p10s),np.array(p90s)-np.array(meds)], fmt="none",color=LEK_DARK_GREEN,capsize=6,lw=1.8)
    ax.set_xticks(x); ax.set_xticklabels(ord_crops,rotation=15,ha="right",fontsize=9)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v,_:f"{v/1e6:.1f}M" if v>=1e6 else f"{v:,.0f}"))
    _lek_ax(ax,f"Potencial de Mercado Global – {PROJ_YEARS[-1]} (USD Nominal)",subtitle="Mediana ± P10–P90")

    ax=fig.add_subplot(gs[0,2]); meds_brl=[]
    for lbl in ord_crops:
        mkt=mc[lbl]["potencial"]; row=mkt[(mkt.year==PROJ_YEARS[-1])&(mkt.metric=="brl_nominal")]
        meds_brl.append(row["median"].values[0])
    bc2=[LEK_DARK_GREEN if i<2 else "#90A4AE" for i in range(len(ord_crops))]
    ax.barh(ord_crops,meds_brl,color=bc2,edgecolor=LEK_BG,height=0.6); ax.invert_yaxis()
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v,_:f"{v/1e6:.1f}M"))
    _lek_ax(ax,f"Potencial BRL {PROJ_YEARS[-1]}")

    ax=fig.add_subplot(gs[1,0]); ax.axis("off")
    tdata=[[lbl[:14],f"{mc[lbl]['mu_demand']*100:+.1f}%",f"{mc[lbl]['sigma_demand']*100:.1f}%",f"{mc[lbl]['kappa']:.2f}"] for lbl in ord_crops]
    t=ax.table(cellText=tdata,colLabels=["Cultura","μ Dem","σ Dem","κ"],loc="center",cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1,1.8)
    for (r_i,c_i),cell in t.get_celld().items():
        cell.set_edgecolor(LEK_LIGHT_GRAY)
        if r_i==0: cell.set_facecolor(LEK_DARK_GREEN); cell.set_text_props(color=LEK_WHITE,fontweight="bold")
        elif r_i%2==0: cell.set_facecolor(LEK_LIGHT_GREEN)
    ax.set_title("Parâmetros Estocásticos Calibrados",fontsize=10,color=LEK_NAVY,fontweight="bold",pad=12,loc="left")

    for col_i,(col_key,title) in enumerate([("market_share_producao_pct","Market Share – Produção (%)"),("market_share_exportacao_pct","Market Share – Exportação (%)")]):
        ax=fig.add_subplot(gs[1,col_i+1])
        for i,lbl in enumerate(ord_crops):
            # Correção 1: Busca flexível que lê a palavra 'Café' corretamente da base
            pat = "Caf" if lbl == "Cafe" else lbl
            s=get_crop_series(df_base, pat, col_key)
            
            # Correção 2: REMOVIDA qualquer multiplicação de porcentagem. 
            # O código confia cegamente que o valor inserido na base é o valor percentual real.
            if not s.empty: ax.plot(s.index,s.values,lw=2,label=lbl[:8],color=LEK_PALETTE[i%len(LEK_PALETTE)],marker="o",markersize=4)
        _lek_leg(ax); _lek_ax(ax,title,"Ano","%")

    _wm(fig); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig)

def fig5_top2(mc,top2,save):
    fig,axes=plt.subplots(2,2,figsize=(15,11))
    _lek_fig(fig,f"Distribuições Probabilísticas – Top 2 por Potencial USD", f"{top2[0]}  e  {top2[1]}  |  ano {PROJ_YEARS[-1]}")
    for col,lbl in enumerate(top2):
        r=mc[lbl]; color=LEK_PALETTE[col%len(LEK_PALETTE)]
        for row_i,(arr,title,c) in enumerate([
            (r["demand_paths"][:,-1]*r["price_paths"][:,-1],f"{lbl} – USD Nominal",color),
            (r["demand_paths"][:,-1]*r["price_paths"][:,-1]*r["fx_paths"][:,-1],f"{lbl} – BRL Nominal",LEK_DARK_GREEN)]):
            ax=axes[row_i][col]
            mu_v,p10_v,p90_v=np.mean(arr),np.percentile(arr,10),np.percentile(arr,90)
            ax.hist(arr,bins=80,color=c,edgecolor=LEK_BG,alpha=0.85)
            ax.axvline(mu_v,color=LEK_DARK_GREEN if row_i==0 else LEK_GREEN,lw=2.5,label=f"Média: {mu_v:,.0f}")
            ax.axvline(p10_v,color=LEK_GRAY,lw=1.5,ls="--",label=f"P10: {p10_v:,.0f}")
            ax.axvline(p90_v,color=LEK_GRAY,lw=1.5,ls="--",label=f"P90: {p90_v:,.0f}")
            _lek_leg(ax); _lek_ax(ax,title)
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.93]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig)

def fig6_sensitivity(mc,top2,save):
    fig,axes=plt.subplots(1,2,figsize=(15,6))
    _lek_fig(fig,"Análise de Sensibilidade Macro – Top 2 Culturas")
    fed_range=np.linspace(1.0,6.5,25)
    for i,lbl in enumerate(top2):
        r = mc[lbl]
        prices=[r["theta_price"]*np.exp(FED_PRICE_IMPACT*(f-r["price_hist"].iloc[-1])*HORIZON) for f in fed_range]
        axes[0].plot(fed_range,prices,lw=2.5,label=lbl,color=LEK_PALETTE[i%len(LEK_PALETTE)])
    axes[0].axvline(FED_FUNDS_PROJ[PROJ_YEARS[-1]],color=LEK_GRAY,lw=1.5,ls="--",label=f"Fed proj.{PROJ_YEARS[-1]}: {FED_FUNDS_PROJ[PROJ_YEARS[-1]]}%")
    _lek_leg(axes[0]); _lek_ax(axes[0],"Preço de Equilíbrio (θ) vs. Fed Funds","Fed Funds (%)","Preço")
    
    vix_range=np.linspace(8,50,25)
    for i,lbl in enumerate(top2):
        r = mc[lbl]
        spread=(r["sigma_price"]+VIX_SENSITIVITY.get(lbl, 0.010)*vix_range)*np.sqrt(HORIZON)*200
        axes[1].plot(vix_range,spread,lw=2.5,label=lbl,color=LEK_PALETTE[i%len(LEK_PALETTE)])
    axes[1].axvline(VIX_PROJ,color=LEK_GRAY,lw=1.5,ls="--",label=f"VIX proj.: {VIX_PROJ}")
    _lek_leg(axes[1]); _lek_ax(axes[1],"Spread P90–P10 do Preço vs. VIX","VIX Médio","Spread % (~±1σ)")
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.93]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig)

# ── PIPELINE ──────────────────────────────────────────────────────────────────
def open_image(p):
    try:
        if sys.platform.startswith("darwin"): subprocess.Popen(["open",str(p)])
        elif sys.platform.startswith("win"):  os.startfile(str(p))
        else: subprocess.Popen(["xdg-open",str(p)])
    except: pass

def run():
    print("="*70); print("  Modelagem Estocástica Avançada 2026 (Cholesky + O-U Dinâmico)"); print("="*70)
    df=load_base()
    
    fx_hist=get_macro_series(df,"usd_brl")[lambda s:s.index>=HIST_START]
    cpi_hist=get_macro_series(df,"us_cpi_index")[lambda s:s.index>=HIST_START]
    vix_hist=get_macro_series(df,"vix_avg")[lambda s:s.index>=HIST_START]
    fed_hist=get_macro_series(df,"us_fed_funds_rate_pct")[lambda s:s.index>=HIST_START]
    
    vix_last=float(vix_hist.iloc[-1]); fed_last=float(fed_hist.iloc[-1]); cpi_last=float(cpi_hist.iloc[-1])
    fx_paths,mu_fx,sig_fx=sim_fx(fx_hist); cpi_paths=sim_cpi(cpi_last)
    
    CSEARCH={"Milho":"Milho","Cafe":"Caf","Cana":"Cana","Citros":"Citros","Trigo":"Trigo"}
    mc={}
    
    df_demand_hist = pd.DataFrame()
    valid_crops = []
    
    # Monta a matriz histórica limpa a partir da base completada
    for lbl, pat in CSEARCH.items():
        dh = get_crop_series(df, pat, "consumo_mundial_1000mt")
        if not dh.empty and len(dh) >= 3:
            df_demand_hist[lbl] = dh
            valid_crops.append(lbl)
            
    # Cholesky para correlações de Demanda
    ret_demand = df_demand_hist.pct_change().dropna()
    corr_matrix = ret_demand.corr().fillna(0).values
    corr_matrix += np.eye(len(corr_matrix)) * 1e-6 
    L_cholesky = np.linalg.cholesky(corr_matrix)
    
    Z_uncorr = np.random.normal(0, 1, (len(valid_crops), N_SIMUL * HORIZON))
    Z_corr_flat = L_cholesky.dot(Z_uncorr)
    Z_corr = Z_corr_flat.reshape(len(valid_crops), N_SIMUL, HORIZON)
    
    # Roda as simulações para todas as culturas limpas
    for idx, lbl in enumerate(valid_crops):
        pat = CSEARCH[lbl]
        dh = df_demand_hist[lbl]
        ph = get_crop_series(df, pat, "preco_usd_unidade")
        pr = get_crop_series(df, pat, "preco_real_usd2024_unidade")
        try: pu=df[df["cultura"].str.contains(pat,case=False,na=False)]["unidade_preco"].iloc[0]
        except: pu="USD/t"
            
        d_p, mu_d, sig_d = sim_demand_correlated(dh, Z_corr[idx])
        p_p, mu_p, sig_p, theta, kappa_dyn, sig_adj = sim_price(ph, lbl, fed_last)
        
        pot=calc_pot(d_p, p_p, fx_paths, cpi_paths, cpi_last)
        
        mc[lbl]={"demand_paths":d_p,"price_paths":p_p,"fx_paths":fx_paths,"cpi_paths":cpi_paths,
                 "demand_hist":dh,"price_hist":ph,"price_hist_real":pr,
                 "mu_demand":mu_d,"sigma_demand":sig_d,"mu_price":mu_p,"sigma_price":sig_p,
                 "sigma_adj":sig_adj,"theta_price":theta,"kappa":kappa_dyn,"price_unit":pu,"potencial":pot}

    ord_crops = get_top_potentials(mc)
    top2 = ord_crops[:2] if len(ord_crops) >= 2 else ord_crops
    
    figs={"fig1":OUTPUT_DIR/"fig1_demand_mc.png","fig2":OUTPUT_DIR/"fig2_price_mc.png",
          "fig3":OUTPUT_DIR/"fig3_fx_cpi.png","fig4":OUTPUT_DIR/"fig4_dashboard.png",
          "fig5":OUTPUT_DIR/"fig5_top2_distributions.png","fig6":OUTPUT_DIR/"fig6_macro_sensitivity.png"}
          
    fig1_demand(mc,figs["fig1"]); fig2_price(mc,figs["fig2"]); fig3_fx_cpi(mc,df,figs["fig3"])
    fig4_dashboard(df,mc,ord_crops,figs["fig4"])
    if len(top2) >= 2:
        fig5_top2(mc,top2,figs["fig5"]); fig6_sensitivity(mc,top2,figs["fig6"])
        
    rows=[]
    for lbl,r in mc.items():
        for t in range(HORIZON):
            dp,pp,fp=r["demand_paths"][:,t],r["price_paths"][:,t],r["fx_paths"][:,t]
            mu=dp*pp; mb=mu*fp
            rows.append({"Cultura":lbl,"Ano":PROJ_YEARS[t],
                         "Dem_P10":np.percentile(dp,10),"Dem_Med":np.median(dp),"Dem_P90":np.percentile(dp,90),
                         "Preco_P10":np.percentile(pp,10),"Preco_Med":np.median(pp),"Preco_P90":np.percentile(pp,90),
                         "FX_Med":np.median(fp),"Pot_USD_Med":np.median(mu),"Pot_BRL_Med":np.median(mb)})
    pd.DataFrame(rows).to_csv(OUTPUT_DIR/"projecoes_mc.csv",index=False,encoding="utf-8-sig",float_format="%.3f")
    
    summary={"top_crops_by_potential":ord_crops,
             "premissas":{"HIST_START":HIST_START,"N_SIMUL":N_SIMUL,"HORIZON":HORIZON}}
             
    with open(OUTPUT_DIR/"resumo_mc.json","w",encoding="utf-8") as f: json.dump(summary,f,ensure_ascii=False,indent=2)
    
    for p in figs.values(): open_image(p)
    return mc, summary

if __name__=="__main__":
    mc, summary=run()

  Modelagem Estocástica Avançada 2026 (Cholesky + O-U Dinâmico)


In [23]:
"""
=============================================================================
MONTE_CARLO_LEK2026.PY  –  Modelo Estocástico com Cholesky e KAPPA Dinâmico
=============================================================================
Lê:    Base LEK (1).xlsx (Sheet1)
Gera:  6 figuras PNG  +  projecoes_mc.csv  +  resumo_mc.json
=============================================================================
"""
import warnings; warnings.filterwarnings("ignore")
import os, sys, subprocess, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from pathlib import Path

BASE_PATH  = Path("Base LEK.xlsx")
OUTPUT_DIR = Path("resultados_lek_modelo_novo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42)

# ── IDENTIDADE VISUAL L.E.K. ─────────────────────────────────────────────────
LEK_GREEN       = "#00A651"
LEK_DARK_GREEN  = "#004D2C"
LEK_LIGHT_GREEN = "#C8E6C9"
LEK_NAVY        = "#1A2744"
LEK_GRAY        = "#6E7B8B"
LEK_LIGHT_GRAY  = "#E8ECEF"
LEK_WHITE       = "#FFFFFF"
LEK_BG          = "#F8F9FA"

# Nova paleta degradê especificada para os gráficos dinâmicos (ordem alfabética)
CUSTOM_COLORS = ["#073700", "#236E03", "#57A50E", '#2fd800', '#38ff00']
LEK_PALETTE     = CUSTOM_COLORS 

COLOR_HIST      = LEK_DARK_GREEN
COLOR_MACRO     = "#FF6F00"

plt.rcParams.update({
    "font.family":"DejaVu Sans","font.size":9,
    "axes.titlesize":10,"axes.labelsize":9,
    "xtick.labelsize":8,"ytick.labelsize":8,"legend.fontsize":8,
    "figure.facecolor":LEK_BG,"axes.facecolor":LEK_BG,
    "axes.edgecolor":LEK_LIGHT_GRAY,"axes.grid":True,
    "grid.color":LEK_LIGHT_GRAY,"grid.linewidth":0.6,
    "axes.spines.top":False,"axes.spines.right":False,
    "text.color":LEK_NAVY,"axes.labelcolor":LEK_NAVY,
    "xtick.color":LEK_GRAY,"ytick.color":LEK_GRAY,
})

PROJ_YEAR  = 2025 # Último ano consolidado da base
HORIZON    = 5
PROJ_YEARS = list(range(PROJ_YEAR+1, PROJ_YEAR+HORIZON+1))

# ── PREMISSAS EXTERNAS ───────────────────────────────────────────────────────
N_SIMUL    = 10_000
HIST_START = 2018 
FALLBACK_MU=0.02; FALLBACK_SIGMA=0.05
VIX_SENSITIVITY={"Milho":0.010,"Cafe":0.012,"Cana":0.009,"Citros":0.015,"Trigo":0.010}
FED_PRICE_IMPACT=-0.015
FED_FUNDS_PROJ={2026:3.50,2027:3.00,2028:3.00,2029:3.00,2030:3.00}
CPI_GROWTH_PROJ=0.025
VIX_PROJ=18.0

# ── BASE ──────────────────────────────────────────────────────────────────────
def load_base():
    df = pd.read_excel(BASE_PATH, sheet_name=0)
    df['ano_int'] = df['Ano'].astype(str).str.split('/').str[0].apply(pd.to_numeric, errors='coerce')
    
    # ── CORREÇÃO DA CANA ──
    df.loc[df['Commodities'] == 'Cana', 'Demanda (Global)(1000 t)'] = df.loc[df['Commodities'] == 'Cana', 'Produção (mil t)']
    
    rename_dict = {
        'Commodities': 'cultura',
        'Preço de venda da tonelada - U$D/t': 'preco_usd_unidade',
        'Demanda (Global)(1000 t)': 'consumo_mundial_1000mt',
        'MarketShareProdução (%)': 'market_share_producao_pct',
        'MarketShareExportação (%)': 'market_share_exportacao_pct',
        'Produção / HE': 'yield_brasil_mt_ha'
    }
    df = df.rename(columns=rename_dict)
    df['unidade_preco'] = "USD/t"
    
    if 'preco_real_usd2024_unidade' not in df.columns:
        df['preco_real_usd2024_unidade'] = df['preco_usd_unidade']
    return df

def get_crop_series(df, pat, col):
    mask = df["cultura"].str.contains(pat, case=False, na=False)
    series = pd.to_numeric(df[mask].sort_values("ano_int").set_index("ano_int")[col], errors="coerce").dropna()
    
    # TRATAMENTO DE PORCENTAGEM: Se a coluna for de Market Share, 
    # garantimos que ela não seja dividida, apenas se precisarmos disso cru.
    # No caso atual, o usuário informou que 0.1 significa 0.1%.
    # Como as premissas não exigem o cálculo do Market Share dentro do Monte Carlo
    # em si (apenas preço e demanda global), a série será exportada para o matplotlib
    # da exata forma que está na base (ex: 30,37 ou 0,7)
    return series

def get_macro_series(df, col):
    anos = sorted(df['ano_int'].dropna().unique())
    if not anos: anos = list(range(HIST_START, PROJ_YEAR + 1))
    
    mock_data = {
        "usd_brl": np.linspace(3.5, 5.5, len(anos)), 
        "us_cpi_index": np.linspace(250, 320, len(anos)),
        "vix_avg": np.full(len(anos), 18.0),
        "us_fed_funds_rate_pct": np.full(len(anos), 3.0)
    }
    
    if col in mock_data:
        s = pd.Series(mock_data[col], index=anos)
        return s.loc[s.index >= HIST_START]
    return pd.Series(dtype=float)

# ── CALIBRAÇÃO (COM KAPPA DINÂMICO) ───────────────────────────────────────────
def fit_gbm(s):
    s=s[s.index>=HIST_START].dropna()
    if len(s)<3: return FALLBACK_MU,FALLBACK_SIGMA
    lr=np.log(s/s.shift(1)).dropna()
    return float(lr.mean()),float(lr.std())

def fit_ou_dynamic(ps):
    s = ps[ps.index >= HIST_START].dropna()
    if len(s) < 4: 
        return FALLBACK_MU, FALLBACK_SIGMA, float(s.iloc[-1]) if len(s)>0 else 100, 0.1, float(s.iloc[-1]) if len(s)>0 else 100
    
    y = s.diff().dropna().values
    X = s.shift(1).dropna().values
    b, a = np.polyfit(X, y, 1)
    
    kappa = -b
    if kappa <= 0:
        kappa = 0.05
        theta = float(s.mean())
    else:
        theta = a / kappa
        
    sigma = np.std(y - (a + b * X)) / float(s.mean())
    return float(s.pct_change().mean()), sigma, theta, max(kappa, 0.05), float(s.iloc[-1])

def fit_fx(s):
    s=s[s.index>=HIST_START].dropna()
    lr=np.log(s/s.shift(1)).dropna()
    return float(lr.mean()),float(lr.std()),float(s.iloc[-1])

# ── SIMULAÇÕES ────────────────────────────────────────────────────────────────
def sim_demand_correlated(s, Z_corr_slice):
    mu,sig=fit_gbm(s); last=float(s.dropna().iloc[-1])
    return last*np.exp(np.cumsum((mu-0.5*sig**2)+sig*Z_corr_slice,axis=1)),mu,sig

def sim_price(ps,lbl,fed_last):
    mu_p,sig_p,theta,kappa,last_p=fit_ou_dynamic(ps)
    sig_adj=sig_p+VIX_SENSITIVITY.get(lbl, 0.010)*VIX_PROJ
    prices=np.full(N_SIMUL,last_p)
    paths=np.zeros((N_SIMUL,HORIZON))
    for t in range(HORIZON):
        yr=PROJ_YEAR+t+1
        mac=FED_PRICE_IMPACT*(FED_FUNDS_PROJ.get(yr,3.0)-fed_last)
        Z=np.random.normal(0,1,N_SIMUL)
        prices=(prices+kappa*(theta-prices)+mac*prices+sig_adj*prices*Z)
        prices=np.clip(prices,0.01,None) 
        paths[:,t]=prices
    return paths,mu_p,sig_p,theta,kappa,sig_adj

def sim_fx(s):
    mu,sig,last=fit_fx(s)
    Z=np.random.normal(0,1,(N_SIMUL,HORIZON))
    return last*np.exp(np.cumsum((mu-0.5*sig**2)+sig*Z,axis=1)),mu,sig

def sim_cpi(last_cpi):
    paths=np.zeros((N_SIMUL,HORIZON)); c=last_cpi
    for t in range(HORIZON):
        c*=(1+CPI_GROWTH_PROJ)
        paths[:,t]=c
    return paths

def calc_pot(d,p,fx,cpi,cpi0):
    mkt_u=d*p; mkt_b=mkt_u*fx; mkt_r=mkt_u*(cpi0/cpi)
    rows=[]
    for t in range(HORIZON):
        for arr,lbl in [(mkt_u[:,t],"usd_nominal"),(mkt_b[:,t],"brl_nominal"),(mkt_r[:,t],"usd_real_2024")]:
            rows.append({"year":PROJ_YEAR+t+1,"metric":lbl,
                         "mean":np.mean(arr),"median":np.median(arr),
                         "p10":np.percentile(arr,10),"p90":np.percentile(arr,90),"std":np.std(arr)})
    return pd.DataFrame(rows)

def get_top_potentials(mc):
    pots = []
    for lbl, r in mc.items():
        mkt = r["potencial"]
        val = mkt[(mkt.year == PROJ_YEARS[-1]) & (mkt.metric == "usd_nominal")]["median"].values[0]
        pots.append((lbl, val))
    pots.sort(key=lambda x: x[1], reverse=True)
    return [p[0] for p in pots]

# ── HELPERS VISUAIS L.E.K. ────────────────────────────────────────────────────
def _lek_ax(ax,title,xlabel=None,ylabel=None,subtitle=None):
    full=title+(f"\n{subtitle}" if subtitle else "")
    ax.set_title(full,fontsize=9,fontweight="bold",color=LEK_NAVY,pad=6,loc="left")
    ax.set_facecolor(LEK_BG)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(LEK_LIGHT_GRAY); ax.spines["bottom"].set_color(LEK_LIGHT_GRAY)
    ax.grid(True,color=LEK_LIGHT_GRAY,linewidth=0.6,axis="y"); ax.set_axisbelow(True)
    if xlabel: ax.set_xlabel(xlabel,fontsize=8,color=LEK_GRAY)
    if ylabel: ax.set_ylabel(ylabel,fontsize=8,color=LEK_GRAY)

def _lek_fig(fig,title,subtitle=""):
    fig.patch.set_facecolor(LEK_BG)
    bar=fig.add_axes([0,0.975,1,0.015]); bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])
    fig.suptitle(title,fontsize=13,fontweight="bold",color=LEK_NAVY,y=0.968,x=0.02,ha="left")
    if subtitle: fig.text(0.02,0.952,subtitle,fontsize=8,color=LEK_GRAY,ha="left",va="top")

def _lek_leg(ax):
    return ax.legend(fontsize=7,framealpha=0.9,facecolor=LEK_WHITE,edgecolor=LEK_LIGHT_GRAY,labelcolor=LEK_NAVY)

def _wm(fig):
    fig.text(0.98,0.01,"Análise Estratégica 2026",fontsize=7,color=LEK_GRAY,ha="right",va="bottom",fontstyle="italic")

def fan(ax,hist_s,paths,color):
    proj=PROJ_YEARS
    if len(hist_s) > 0:
        ax.plot(list(hist_s.index),hist_s.values,color=COLOR_HIST,lw=2.2,label="Histórico",zorder=3)
        ultimo_ano = hist_s.index[-1]
        ultimo_valor = hist_s.values[-1]
    else:
        ultimo_ano = PROJ_YEAR
        ultimo_valor = np.median(paths[:,0])

    n_amostras = min(150, paths.shape[0])
    amostra_idx = np.random.choice(paths.shape[0], n_amostras, replace=False)
    x_spaghetti = [ultimo_ano] + proj
    
    for idx in amostra_idx:
        y_spaghetti = [ultimo_valor] + list(paths[idx, :])
        ax.plot(x_spaghetti, y_spaghetti, color=color, alpha=0.15, lw=0.8, zorder=1)
        
    ax.plot(proj,np.median(paths,axis=0),color=LEK_NAVY,lw=2.5,ls="--",label="Mediana",zorder=4)
    ax.axvline(PROJ_YEAR,color=LEK_GRAY,lw=0.8,ls=":",alpha=0.7)

# ── FIGURAS ───────────────────────────────────────────────────────────────────
def fig1_demand(mc,save):
    crops=list(mc.keys())
    fig,axes=plt.subplots(2,3,figsize=(19,11))
    _lek_fig(fig,"Monte Carlo – Demanda Mundial por Cultura",
             f"{N_SIMUL:,} simulações  |  GBM Correlacionado (Cholesky)  |  Horizonte {PROJ_YEARS[0]}–{PROJ_YEARS[-1]}")
    for i,lbl in enumerate(crops):
        ax=axes[i//3][i%3]; r=mc[lbl]
        fan(ax,r["demand_hist"],r["demand_paths"],LEK_PALETTE[i%len(LEK_PALETTE)])
        _lek_ax(ax,lbl,"Ano","1.000 MT",subtitle=f"μ={r['mu_demand']*100:+.1f}%  σ={r['sigma_demand']*100:.1f}%")
        if i==0: _lek_leg(ax)
    for j in range(len(crops), 6): axes[j//3][j%3].set_visible(False)
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.94]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig)

def fig2_price(mc,save):
    crops=list(mc.keys())
    fig,axes=plt.subplots(2,3,figsize=(19,11))
    _lek_fig(fig,"Monte Carlo – Preço por Cultura (Ornstein-Uhlenbeck + Macro)",
             "KAPPA e THETA calibrados via regressão linear dinâmica AR(1)")
    for i,lbl in enumerate(crops):
        ax=axes[i//3][i%3]; r=mc[lbl]
        fan(ax,r["price_hist"],r["price_paths"],LEK_PALETTE[i%len(LEK_PALETTE)])
        ax.axhline(r["theta_price"],color=LEK_GRAY,lw=1.2,ls=":",alpha=0.8,label=f"θ={r['theta_price']:.2f}")
        _lek_ax(ax,lbl,"Ano",r["price_unit"],subtitle=f"θ={r['theta_price']:.2f}  κ={r['kappa']:.2f}  σ_adj={r['sigma_adj']:.3f}")
        if i==0: _lek_leg(ax)
    for j in range(len(crops), 6): axes[j//3][j%3].set_visible(False)
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.94]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig)

def fig3_fx_cpi(mc,df_base,save):
    fx_hist=get_macro_series(df_base,"usd_brl")[lambda s:s.index>=HIST_START]
    cpi_hist=get_macro_series(df_base,"us_cpi_index")[lambda s:s.index>=HIST_START]
    fx_paths=mc[list(mc.keys())[0]]["fx_paths"]
    fig,axes=plt.subplots(1,2,figsize=(15,6))
    _lek_fig(fig,"Projeção de Variáveis Macroeconômicas (Proxies)")
    fan(axes[0],fx_hist,fx_paths,COLOR_MACRO)
    _lek_ax(axes[0],"Câmbio USD/BRL","Ano","R$/US$"); _lek_leg(axes[0])
    last_cpi=float(cpi_hist.iloc[-1])
    proj=[last_cpi*(1+CPI_GROWTH_PROJ)**(t+1) for t in range(HORIZON)]
    axes[1].plot(list(cpi_hist.index),cpi_hist.values,color=COLOR_HIST,lw=2.2,label="Proxy Histórico")
    axes[1].plot(PROJ_YEARS,proj,color=LEK_GREEN,lw=2.5,ls="--",label=f"Proj. {CPI_GROWTH_PROJ*100:.1f}%a.a.")
    axes[1].axvline(PROJ_YEAR,color=LEK_GRAY,lw=0.8,ls=":",alpha=0.7)
    _lek_ax(axes[1],"US CPI Index","Ano","Índice"); _lek_leg(axes[1])
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.93]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig)

def fig4_dashboard(df_base,mc,ord_crops,save):
    fig=plt.figure(figsize=(22,12)); fig.patch.set_facecolor(LEK_BG)
    gs=gridspec.GridSpec(2,3,figure=fig,hspace=0.40,wspace=0.30,top=0.88,bottom=0.08,left=0.06,right=0.97)
    bar=fig.add_axes([0,0.955,1,0.015]); bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])
    fig.suptitle("Dashboard de Potencial de Mercado", fontsize=14,fontweight="bold",color=LEK_NAVY,y=0.975,x=0.02,ha="left")

    ax=fig.add_subplot(gs[0,:2]); meds,p10s,p90s=[],[],[]
    for lbl in ord_crops:
        mkt=mc[lbl]["potencial"]; row=mkt[(mkt.year==PROJ_YEARS[-1])&(mkt.metric=="usd_nominal")]
        meds.append(row["median"].values[0]); p10s.append(row["p10"].values[0]); p90s.append(row["p90"].values[0])
    x=np.arange(len(ord_crops))
    bcolors=[LEK_GREEN if i<2 else LEK_LIGHT_GRAY for i in range(len(ord_crops))]
    ax.bar(x,meds,color=bcolors,edgecolor=LEK_BG,width=0.6)
    ax.errorbar(x,meds,yerr=[np.array(meds)-np.array(p10s),np.array(p90s)-np.array(meds)], fmt="none",color=LEK_DARK_GREEN,capsize=6,lw=1.8)
    ax.set_xticks(x); ax.set_xticklabels(ord_crops,rotation=15,ha="right",fontsize=9)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v,_:f"{v/1e6:.1f}M" if v>=1e6 else f"{v:,.0f}"))
    _lek_ax(ax,f"Potencial de Mercado Global – {PROJ_YEARS[-1]} (USD Nominal)",subtitle="Mediana ± P10–P90")

    ax=fig.add_subplot(gs[0,2]); meds_brl=[]
    for lbl in ord_crops:
        mkt=mc[lbl]["potencial"]; row=mkt[(mkt.year==PROJ_YEARS[-1])&(mkt.metric=="brl_nominal")]
        meds_brl.append(row["median"].values[0])
    bc2=[LEK_DARK_GREEN if i<2 else "#90A4AE" for i in range(len(ord_crops))]
    ax.barh(ord_crops,meds_brl,color=bc2,edgecolor=LEK_BG,height=0.6); ax.invert_yaxis()
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v,_:f"{v/1e6:.1f}M"))
    _lek_ax(ax,f"Potencial BRL {PROJ_YEARS[-1]}")

    ax=fig.add_subplot(gs[1,0]); ax.axis("off")
    tdata=[[lbl[:14],f"{mc[lbl]['mu_demand']*100:+.1f}%",f"{mc[lbl]['sigma_demand']*100:.1f}%",f"{mc[lbl]['kappa']:.2f}"] for lbl in ord_crops]
    t=ax.table(cellText=tdata,colLabels=["Cultura","μ Dem","σ Dem","κ"],loc="center",cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(9); t.scale(1,1.8)
    for (r_i,c_i),cell in t.get_celld().items():
        cell.set_edgecolor(LEK_LIGHT_GRAY)
        if r_i==0: cell.set_facecolor(LEK_DARK_GREEN); cell.set_text_props(color=LEK_WHITE,fontweight="bold")
        elif r_i%2==0: cell.set_facecolor(LEK_LIGHT_GREEN)
    ax.set_title("Parâmetros Estocásticos Calibrados",fontsize=10,color=LEK_NAVY,fontweight="bold",pad=12,loc="left")

    for col_i,(col_key,title) in enumerate([("market_share_producao_pct","Market Share – Produção (%)"),("market_share_exportacao_pct","Market Share – Exportação (%)")]):
        ax=fig.add_subplot(gs[1,col_i+1])
        for i,lbl in enumerate(ord_crops):
            pat = "Caf" if lbl == "Cafe" else lbl
            s=get_crop_series(df_base, pat, col_key)
            
            # Aqui eu removi qualquer tipo de multiplicação condicional ou regra de decimal.
            # O matplotlib simplesmente lerá e desenhará o número que estiver no seu Excel.
            if not s.empty: ax.plot(s.index,s.values,lw=2,label=lbl[:8],color=LEK_PALETTE[i%len(LEK_PALETTE)],marker="o",markersize=4)
        _lek_leg(ax); _lek_ax(ax,title,"Ano","%")

    _wm(fig); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig)

def fig5_top2(mc,top2,save):
    fig,axes=plt.subplots(2,2,figsize=(15,11))
    _lek_fig(fig,f"Distribuições Probabilísticas – Top 2 por Potencial USD", f"{top2[0]}  e  {top2[1]}  |  ano {PROJ_YEARS[-1]}")
    for col,lbl in enumerate(top2):
        r=mc[lbl]; color=LEK_PALETTE[col%len(LEK_PALETTE)]
        for row_i,(arr,title,c) in enumerate([
            (r["demand_paths"][:,-1]*r["price_paths"][:,-1],f"{lbl} – USD Nominal",color),
            (r["demand_paths"][:,-1]*r["price_paths"][:,-1]*r["fx_paths"][:,-1],f"{lbl} – BRL Nominal",LEK_DARK_GREEN)]):
            ax=axes[row_i][col]
            mu_v,p10_v,p90_v=np.mean(arr),np.percentile(arr,10),np.percentile(arr,90)
            ax.hist(arr,bins=80,color=c,edgecolor=LEK_BG,alpha=0.85)
            ax.axvline(mu_v,color=LEK_DARK_GREEN if row_i==0 else LEK_GREEN,lw=2.5,label=f"Média: {mu_v:,.0f}")
            ax.axvline(p10_v,color=LEK_GRAY,lw=1.5,ls="--",label=f"P10: {p10_v:,.0f}")
            ax.axvline(p90_v,color=LEK_GRAY,lw=1.5,ls="--",label=f"P90: {p90_v:,.0f}")
            _lek_leg(ax); _lek_ax(ax,title)
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.93]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig)

def fig6_sensitivity(mc,top2,save):
    fig,axes=plt.subplots(1,2,figsize=(15,6))
    _lek_fig(fig,"Análise de Sensibilidade Macro – Top 2 Culturas")
    fed_range=np.linspace(1.0,6.5,25)
    for i,lbl in enumerate(top2):
        r = mc[lbl]
        prices=[r["theta_price"]*np.exp(FED_PRICE_IMPACT*(f-r["price_hist"].iloc[-1])*HORIZON) for f in fed_range]
        axes[0].plot(fed_range,prices,lw=2.5,label=lbl,color=LEK_PALETTE[i%len(LEK_PALETTE)])
    axes[0].axvline(FED_FUNDS_PROJ[PROJ_YEARS[-1]],color=LEK_GRAY,lw=1.5,ls="--",label=f"Fed proj.{PROJ_YEARS[-1]}: {FED_FUNDS_PROJ[PROJ_YEARS[-1]]}%")
    _lek_leg(axes[0]); _lek_ax(axes[0],"Preço de Equilíbrio (θ) vs. Fed Funds","Fed Funds (%)","Preço")
    
    vix_range=np.linspace(8,50,25)
    for i,lbl in enumerate(top2):
        r = mc[lbl]
        spread=(r["sigma_price"]+VIX_SENSITIVITY.get(lbl, 0.010)*vix_range)*np.sqrt(HORIZON)*200
        axes[1].plot(vix_range,spread,lw=2.5,label=lbl,color=LEK_PALETTE[i%len(LEK_PALETTE)])
    axes[1].axvline(VIX_PROJ,color=LEK_GRAY,lw=1.5,ls="--",label=f"VIX proj.: {VIX_PROJ}")
    _lek_leg(axes[1]); _lek_ax(axes[1],"Spread P90–P10 do Preço vs. VIX","VIX Médio","Spread % (~±1σ)")
    _wm(fig); plt.tight_layout(rect=[0,0.02,1,0.93]); fig.savefig(save,dpi=150,bbox_inches="tight"); plt.close(fig)

# ── PIPELINE ──────────────────────────────────────────────────────────────────
def open_image(p):
    try:
        if sys.platform.startswith("darwin"): subprocess.Popen(["open",str(p)])
        elif sys.platform.startswith("win"):  os.startfile(str(p))
        else: subprocess.Popen(["xdg-open",str(p)])
    except: pass

def run():
    print("="*70); print("  Modelagem Estocástica Avançada 2026 (Cholesky + O-U Dinâmico)"); print("="*70)
    df=load_base()
    
    fx_hist=get_macro_series(df,"usd_brl")[lambda s:s.index>=HIST_START]
    cpi_hist=get_macro_series(df,"us_cpi_index")[lambda s:s.index>=HIST_START]
    vix_hist=get_macro_series(df,"vix_avg")[lambda s:s.index>=HIST_START]
    fed_hist=get_macro_series(df,"us_fed_funds_rate_pct")[lambda s:s.index>=HIST_START]
    
    vix_last=float(vix_hist.iloc[-1]); fed_last=float(fed_hist.iloc[-1]); cpi_last=float(cpi_hist.iloc[-1])
    fx_paths,mu_fx,sig_fx=sim_fx(fx_hist); cpi_paths=sim_cpi(cpi_last)
    
    CSEARCH={"Milho":"Milho","Cafe":"Caf","Cana":"Cana","Citros":"Citros","Trigo":"Trigo"}
    mc={}
    
    df_demand_hist = pd.DataFrame()
    valid_crops = []
    
    # Monta a matriz histórica limpa a partir da base completada
    for lbl, pat in CSEARCH.items():
        dh = get_crop_series(df, pat, "consumo_mundial_1000mt")
        if not dh.empty and len(dh) >= 3:
            df_demand_hist[lbl] = dh
            valid_crops.append(lbl)
            
    # Cholesky para correlações de Demanda
    ret_demand = df_demand_hist.pct_change().dropna()
    corr_matrix = ret_demand.corr().fillna(0).values
    corr_matrix += np.eye(len(corr_matrix)) * 1e-6 
    L_cholesky = np.linalg.cholesky(corr_matrix)
    
    Z_uncorr = np.random.normal(0, 1, (len(valid_crops), N_SIMUL * HORIZON))
    Z_corr_flat = L_cholesky.dot(Z_uncorr)
    Z_corr = Z_corr_flat.reshape(len(valid_crops), N_SIMUL, HORIZON)
    
    # Roda as simulações para todas as culturas limpas
    for idx, lbl in enumerate(valid_crops):
        pat = CSEARCH[lbl]
        dh = df_demand_hist[lbl]
        ph = get_crop_series(df, pat, "preco_usd_unidade")
        pr = get_crop_series(df, pat, "preco_real_usd2024_unidade")
        try: pu=df[df["cultura"].str.contains(pat,case=False,na=False)]["unidade_preco"].iloc[0]
        except: pu="USD/t"
            
        d_p, mu_d, sig_d = sim_demand_correlated(dh, Z_corr[idx])
        p_p, mu_p, sig_p, theta, kappa_dyn, sig_adj = sim_price(ph, lbl, fed_last)
        
        pot=calc_pot(d_p, p_p, fx_paths, cpi_paths, cpi_last)
        
        mc[lbl]={"demand_paths":d_p,"price_paths":p_p,"fx_paths":fx_paths,"cpi_paths":cpi_paths,
                 "demand_hist":dh,"price_hist":ph,"price_hist_real":pr,
                 "mu_demand":mu_d,"sigma_demand":sig_d,"mu_price":mu_p,"sigma_price":sig_p,
                 "sigma_adj":sig_adj,"theta_price":theta,"kappa":kappa_dyn,"price_unit":pu,"potencial":pot}

    ord_crops = get_top_potentials(mc)
    top2 = ord_crops[:2] if len(ord_crops) >= 2 else ord_crops
    
    figs={"fig1":OUTPUT_DIR/"fig1_demand_mc.png","fig2":OUTPUT_DIR/"fig2_price_mc.png",
          "fig3":OUTPUT_DIR/"fig3_fx_cpi.png","fig4":OUTPUT_DIR/"fig4_dashboard.png",
          "fig5":OUTPUT_DIR/"fig5_top2_distributions.png","fig6":OUTPUT_DIR/"fig6_macro_sensitivity.png"}
          
    fig1_demand(mc,figs["fig1"]); fig2_price(mc,figs["fig2"]); fig3_fx_cpi(mc,df,figs["fig3"])
    fig4_dashboard(df,mc,ord_crops,figs["fig4"])
    if len(top2) >= 2:
        fig5_top2(mc,top2,figs["fig5"]); fig6_sensitivity(mc,top2,figs["fig6"])
        
    rows=[]
    for lbl,r in mc.items():
        for t in range(HORIZON):
            dp,pp,fp=r["demand_paths"][:,t],r["price_paths"][:,t],r["fx_paths"][:,t]
            mu=dp*pp; mb=mu*fp
            rows.append({"Cultura":lbl,"Ano":PROJ_YEARS[t],
                         "Dem_P10":np.percentile(dp,10),"Dem_Med":np.median(dp),"Dem_P90":np.percentile(dp,90),
                         "Preco_P10":np.percentile(pp,10),"Preco_Med":np.median(pp),"Preco_P90":np.percentile(pp,90),
                         "FX_Med":np.median(fp),"Pot_USD_Med":np.median(mu),"Pot_BRL_Med":np.median(mb)})
    pd.DataFrame(rows).to_csv(OUTPUT_DIR/"projecoes_mc.csv",index=False,encoding="utf-8-sig",float_format="%.3f")
    
    summary={"top_crops_by_potential":ord_crops,
             "premissas":{"HIST_START":HIST_START,"N_SIMUL":N_SIMUL,"HORIZON":HORIZON}}
             
    with open(OUTPUT_DIR/"resumo_mc.json","w",encoding="utf-8") as f: json.dump(summary,f,ensure_ascii=False,indent=2)
    
    for p in figs.values(): open_image(p)
    return mc, summary

if __name__=="__main__":
    mc, summary=run()

  Modelagem Estocástica Avançada 2026 (Cholesky + O-U Dinâmico)


In [27]:
import warnings; warnings.filterwarnings("ignore")
import os, sys, subprocess, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from pathlib import Path

BASE_PATH  = Path("Base LEK.xlsx")
OUTPUT_DIR = Path("resultados_lek_modelo_novo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42)

# ── IDENTIDADE VISUAL L.E.K. ─────────────────────────────────────────────────
LEK_GREEN       = "#00A651"
LEK_DARK_GREEN  = "#004D2C"
LEK_LIGHT_GREEN = "#C8E6C9"
LEK_NAVY        = "#1A2744"
LEK_GRAY        = "#6E7B8B"
LEK_LIGHT_GRAY  = "#E8ECEF"
LEK_WHITE       = "#FFFFFF"
LEK_BG          = "#F8F9FA"

# Cores especificadas pelo usuário por cultura
CROP_COLORS = {
    "Cafe": "#073700",
    "Cana": "#236E03",
    "Citros": "#57A50E",
    "Milho": "#2fd800",
    "Trigo": "#38ff00"
}

def get_crop_color(lbl):
    for k, v in CROP_COLORS.items():
        if k.lower() in lbl.lower(): return v
    return LEK_GREEN

COLOR_HIST      = LEK_DARK_GREEN
COLOR_MACRO     = "#FF6F00"

plt.rcParams.update({
    "font.family":"DejaVu Sans","font.size":9,
    "axes.titlesize":10,"axes.labelsize":9,
    "xtick.labelsize":8,"ytick.labelsize":8,"legend.fontsize":8,
    "figure.facecolor":LEK_BG,"axes.facecolor":LEK_BG,
    "axes.edgecolor":LEK_LIGHT_GRAY,"axes.grid":True,
    "grid.color":LEK_LIGHT_GRAY,"grid.linewidth":0.6,
    "axes.spines.top":False,"axes.spines.right":False,
    "text.color":LEK_NAVY,"axes.labelcolor":LEK_NAVY,
    "xtick.color":LEK_GRAY,"ytick.color":LEK_GRAY,
})

PROJ_YEAR  = 2025 # Último ano consolidado da base
HORIZON    = 5
PROJ_YEARS = list(range(PROJ_YEAR+1, PROJ_YEAR+HORIZON+1))

# ── PREMISSAS EXTERNAS ───────────────────────────────────────────────────────
N_SIMUL    = 10_000
HIST_START = 2018 
FALLBACK_MU=0.02; FALLBACK_SIGMA=0.05
VIX_SENSITIVITY={"Milho":0.010,"Cafe":0.012,"Cana":0.009,"Citros":0.015,"Trigo":0.010}
FED_PRICE_IMPACT=-0.015
FED_FUNDS_PROJ={2026:3.50,2027:3.00,2028:3.00,2029:3.00,2030:3.00}
CPI_GROWTH_PROJ=0.025
VIX_PROJ=18.0

# ── BASE ──────────────────────────────────────────────────────────────────────
def load_base():
    df = pd.read_excel(BASE_PATH, sheet_name=0)
    # Limpeza básica nos nomes das colunas
    df.columns = df.columns.str.strip()
    
    df['ano_int'] = df['Ano'].astype(str).str.split('/').str[0].apply(pd.to_numeric, errors='coerce')
    
    # Renomeia coluna alvo de busca para padronização interna
    if 'Commodities' in df.columns:
        df = df.rename(columns={'Commodities': 'cultura'})
    elif 'Commodity' in df.columns:
        df = df.rename(columns={'Commodity': 'cultura'})
        
    df['unidade_preco'] = "USD/t"
    return df

def get_series_by_aliases(df, pat, aliases):
    # Busca a cultura na base
    mask = df["cultura"].str.contains(pat, case=False, na=False)
    sub = df[mask]
    if sub.empty: return pd.Series(dtype=float)
    
    # Normaliza colunas do DataFrame (tudo minúsculo sem espaços)
    df_cols_clean = {str(c).strip().lower().replace(" ", ""): c for c in sub.columns}
    
    for alias in aliases:
        alias_clean = str(alias).strip().lower().replace(" ", "")
        if alias_clean in df_cols_clean:
            real_col = df_cols_clean[alias_clean]
            # Extrai e converte para numérico
            s = pd.to_numeric(sub.sort_values("ano_int").set_index("ano_int")[real_col], errors="coerce").dropna()
            if not s.empty:
                return s
                
    return pd.Series(dtype=float)

def get_macro_series(df, col):
    anos = sorted(df['ano_int'].dropna().unique())
    if not anos: anos = list(range(HIST_START, PROJ_YEAR + 1))
    
    mock_data = {
        "usd_brl": np.linspace(3.5, 5.5, len(anos)), 
        "us_cpi_index": np.linspace(250, 320, len(anos)),
        "vix_avg": np.full(len(anos), 18.0),
        "us_fed_funds_rate_pct": np.full(len(anos), 3.0)
    }
    
    if col in mock_data:
        s = pd.Series(mock_data[col], index=anos)
        return s.loc[s.index >= HIST_START]
    return pd.Series(dtype=float)

# ── CALIBRAÇÃO ───────────────────────────────────────────────────────────────
def fit_gbm(s):
    s=s[s.index>=HIST_START].dropna()
    if len(s)<3: return FALLBACK_MU,FALLBACK_SIGMA
    lr=np.log(s/s.shift(1)).dropna()
    return float(lr.mean()),float(lr.std())

def fit_ou_dynamic(ps):
    s = ps[ps.index >= HIST_START].dropna()
    if len(s) < 4: 
        return FALLBACK_MU, FALLBACK_SIGMA, float(s.iloc[-1]) if len(s)>0 else 100, 0.1, float(s.iloc[-1]) if len(s)>0 else 100
    
    y = s.diff().dropna().values
    X = s.shift(1).dropna().values
    b, a = np.polyfit(X, y, 1)
    
    kappa = -b
    if kappa <= 0:
        kappa = 0.05
        theta = float(s.mean())
    else:
        theta = a / kappa
        
    sigma = np.std(y - (a + b * X)) / float(s.mean())
    return float(s.pct_change().mean()), sigma, theta, max(kappa, 0.05), float(s.iloc[-1])

def fit_fx(s):
    s=s[s.index>=HIST_START].dropna()
    lr=np.log(s/s.shift(1)).dropna()
    return float(lr.mean()),float(lr.std()),float(s.iloc[-1])

# ── SIMULAÇÕES ────────────────────────────────────────────────────────────────
def sim_gbm_independent(s):
    s=s[s.index>=HIST_START].dropna()
    if len(s)<3: 
        return np.zeros((N_SIMUL,HORIZON)), FALLBACK_MU, FALLBACK_SIGMA
    last=float(s.iloc[-1])
    mu,sig=fit_gbm(s)
    Z=np.random.normal(0,1,(N_SIMUL,HORIZON))
    paths = last*np.exp(np.cumsum((mu-0.5*sig**2)+sig*Z,axis=1))
    return paths, mu, sig

def sim_price(ps,lbl,fed_last):
    mu_p,sig_p,theta,kappa,last_p=fit_ou_dynamic(ps)
    sig_adj=sig_p+VIX_SENSITIVITY.get(lbl, 0.010)*VIX_PROJ
    prices=np.full(N_SIMUL,last_p)
    paths=np.zeros((N_SIMUL,HORIZON))
    for t in range(HORIZON):
        yr=PROJ_YEAR+t+1
        mac=FED_PRICE_IMPACT*(FED_FUNDS_PROJ.get(yr,3.0)-fed_last)
        Z=np.random.normal(0,1,N_SIMUL)
        prices=(prices+kappa*(theta-prices)+mac*prices+sig_adj*prices*Z)
        prices=np.clip(prices,0.01,None) 
        paths[:,t]=prices
    return paths,mu_p,sig_p,theta,kappa,sig_adj

def sim_fx(s):
    mu,sig,last=fit_fx(s)
    Z=np.random.normal(0,1,(N_SIMUL,HORIZON))
    return last*np.exp(np.cumsum((mu-0.5*sig**2)+sig*Z,axis=1)),mu,sig

# ── HELPERS VISUAIS L.E.K. ────────────────────────────────────────────────────
def _lek_ax(ax,title,xlabel=None,ylabel=None,subtitle=None):
    full=title+(f"\n{subtitle}" if subtitle else "")
    ax.set_title(full,fontsize=9,fontweight="bold",color=LEK_NAVY,pad=6,loc="left")
    ax.set_facecolor(LEK_BG)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(LEK_LIGHT_GRAY); ax.spines["bottom"].set_color(LEK_LIGHT_GRAY)
    ax.grid(True,color=LEK_LIGHT_GRAY,linewidth=0.6,axis="y"); ax.set_axisbelow(True)
    if xlabel: ax.set_xlabel(xlabel,fontsize=8,color=LEK_GRAY)
    if ylabel: ax.set_ylabel(ylabel,fontsize=8,color=LEK_GRAY)

def _lek_fig(fig,title,subtitle=""):
    fig.patch.set_facecolor(LEK_BG)
    bar=fig.add_axes([0,0.975,1,0.015]); bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])
    fig.suptitle(title,fontsize=13,fontweight="bold",color=LEK_NAVY,y=0.968,x=0.02,ha="left")
    if subtitle: fig.text(0.02,0.952,subtitle,fontsize=8,color=LEK_GRAY,ha="left",va="top")

def _lek_leg(ax):
    return ax.legend(fontsize=7,framealpha=0.9,facecolor=LEK_WHITE,edgecolor=LEK_LIGHT_GRAY,labelcolor=LEK_NAVY)

def fan(ax,hist_s,paths,color):
    proj=PROJ_YEARS
    # Foco a partir de 2024
    hist_plot = hist_s[hist_s.index >= 2024]
    
    if len(hist_plot) > 0:
        ax.plot(list(hist_plot.index),hist_plot.values,color=COLOR_HIST,lw=2.2,label="Histórico (>=2024)",zorder=3)
        ultimo_ano = hist_plot.index[-1]
        ultimo_valor = hist_plot.values[-1]
    elif len(hist_s) > 0:
        ax.plot(list(hist_s.index),hist_s.values,color=COLOR_HIST,lw=2.2,label="Histórico",zorder=3)
        ultimo_ano = hist_s.index[-1]
        ultimo_valor = hist_s.values[-1]
    else:
        ultimo_ano = PROJ_YEAR
        ultimo_valor = np.median(paths[:,0]) if paths.size > 0 else 0

    n_amostras = min(150, paths.shape[0])
    # Se houver caminhos válidos (não zerados)
    if n_amostras > 0 and not np.all(paths == 0):
        amostra_idx = np.random.choice(paths.shape[0], n_amostras, replace=False)
        x_spaghetti = [ultimo_ano] + proj
        
        for idx in amostra_idx:
            y_spaghetti = [ultimo_valor] + list(paths[idx, :])
            ax.plot(x_spaghetti, y_spaghetti, color=color, alpha=0.15, lw=0.8, zorder=1)
            
        ax.plot(proj,np.median(paths,axis=0),color=LEK_NAVY,lw=2.5,ls="--",label="Mediana",zorder=4)
    else:
        # Array zerado = dados não encontrados. Coloca um texto de aviso
        ax.text(0.5, 0.5, 'Dados não encontrados\nna base', horizontalalignment='center',
                verticalalignment='center', transform=ax.transAxes, color='red')
    
    ax.axvline(PROJ_YEAR,color=LEK_GRAY,lw=0.8,ls=":",alpha=0.7)
    
    # Limita o eixo X para que os gráficos comecem no mínimo em 2024
    ax.set_xlim(left=2024)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

# ── FIGURAS POR CULTURA (UM PNG PARA CADA) ────────────────────────────────────
def plot_crop_dashboard(lbl, r, save_path):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    _lek_fig(fig, f"Dashboard de Projeções (2024+) : {lbl.upper()}", f"{N_SIMUL:,} simulações | Preços Macro-Ajustados")
    c = get_crop_color(lbl)
    
    # 1. Demanda Global
    fan(axes[0,0], r['demanda_global_hist'], r['demanda_global_paths'], c)
    _lek_ax(axes[0,0], "Demanda Global", "Ano", "1.000 MT")
    _lek_leg(axes[0,0])
    
    # 2. Demanda Interna
    fan(axes[0,1], r['demanda_interna_hist'], r['demanda_interna_paths'], c)
    _lek_ax(axes[0,1], "Demanda Interna (Brasil)", "Ano", "1.000 MT")
    
    # 3. Produção Global
    fan(axes[0,2], r['producao_global_hist'], r['producao_global_paths'], c)
    _lek_ax(axes[0,2], "Produção Global", "Ano", "1.000 MT")
    
    # 4. Produção Interna
    fan(axes[1,0], r['producao_interna_hist'], r['producao_interna_paths'], c)
    _lek_ax(axes[1,0], "Produção Interna (Brasil)", "Ano", "1.000 MT")
    
    # 5. Preço Global (USD)
    fan(axes[1,1], r['price_hist'], r['price_paths'], c)
    _lek_ax(axes[1,1], "Preço Global (USD)", "Ano", r.get('price_unit', 'USD/t'))
    
    # 6. Preço Global (BRL) - Derived
    price_brl_paths = r['price_paths'] * r['fx_paths']
    
    if len(r['price_hist']) > 0 and len(r['fx_hist']) > 0:
        common_idx = r['price_hist'].index.intersection(r['fx_hist'].index)
        hist_brl = r['price_hist'][common_idx] * r['fx_hist'][common_idx]
    else:
        hist_brl = pd.Series(dtype=float)
        
    fan(axes[1,2], hist_brl.dropna(), price_brl_paths, c)
    _lek_ax(axes[1,2], "Preço Global (BRL)", "Ano", "R$/t")
    
    plt.tight_layout(rect=[0,0.02,1,0.94])
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)

# ── PIPELINE ──────────────────────────────────────────────────────────────────
def run():
    print("="*70); print("  Modelagem Estocástica Avançada 2026 (Produção/Demanda/Preço)"); print("="*70)
    
    try:
        df = load_base()
    except FileNotFoundError:
        print("Arquivo Base LEK.xlsx não encontrado.")
        return
        
    fx_hist=get_macro_series(df,"usd_brl")[lambda s:s.index>=HIST_START]
    fed_hist=get_macro_series(df,"us_fed_funds_rate_pct")[lambda s:s.index>=HIST_START]
    fed_last=float(fed_hist.iloc[-1]) if len(fed_hist)>0 else 3.0
    
    fx_paths,mu_fx,sig_fx=sim_fx(fx_hist)
    
    CSEARCH={"Milho":"Milho","Cafe":"Caf","Cana":"Cana","Citros":"Citros","Trigo":"Trigo"}
    mc={}
    
    # Aliases hiper-flexíveis. A função get_series_by_aliases limpa espaços e capitalização
    aliases_dem_glob = ['Demanda(Global)(1000t)', 'DemandaGlobal']
    aliases_dem_int  = ['Demanda(Interna)(1000t)', 'DemandaInterna']
    aliases_prod_int = ['Produção(milt)', 'ProduçãoInterna', 'ProduçãoBrasil']
    aliases_prod_glob = ['Produção(Global)(1000t)', 'ProduçãoGlobal']
    aliases_price    = ['Preçodevendadatonelada-U$D/t', 'preco_usd_unidade']
    aliases_mkt_share= ['MarketShareProdução(%)']
    
    for lbl, pat in CSEARCH.items():
        # Captura os dados históricos
        dh_glob = get_series_by_aliases(df, pat, aliases_dem_glob)
        dh_int  = get_series_by_aliases(df, pat, aliases_dem_int)
        ph_int  = get_series_by_aliases(df, pat, aliases_prod_int)
        price_h = get_series_by_aliases(df, pat, aliases_price)
        
        # Especial para Cana: a demanda global costuma ser espelhada na produção na sua base original
        if lbl == "Cana" and dh_glob.empty:
            dh_glob = ph_int
            
        # A Produção Global não existe explicitamente na sua tabela, então a derivamos a partir
        # da Produção Interna e do Market Share de Produção (%).
        ph_glob = get_series_by_aliases(df, pat, aliases_prod_glob)
        if ph_glob.empty and not ph_int.empty:
            mkt_share = get_series_by_aliases(df, pat, aliases_mkt_share)
            if not mkt_share.empty:
                # Se o Brasil produz 100.000 ton e tem 10% de mkt share, o mundo produz 1.000.000 ton
                ph_glob = ph_int / (mkt_share / 100.0)
            else:
                # Falha segura: igual a demanda global
                ph_glob = dh_glob

        try: pu=df[df["cultura"].str.contains(pat,case=False,na=False)]["unidade_preco"].iloc[0]
        except: pu="USD/t"
            
        # Simulações (GBM Independente)
        dp_glob, _, _ = sim_gbm_independent(dh_glob)
        dp_int,  _, _ = sim_gbm_independent(dh_int)
        pp_glob, _, _ = sim_gbm_independent(ph_glob)
        pp_int,  _, _ = sim_gbm_independent(ph_int)
        
        # Preço (OU-Dinâmico)
        price_p, mu_p, sig_p, theta, kappa_dyn, sig_adj = sim_price(price_h, lbl, fed_last)
        
        mc[lbl]={
            "demanda_global_hist": dh_glob, "demanda_global_paths": dp_glob,
            "demanda_interna_hist": dh_int, "demanda_interna_paths": dp_int,
            "producao_global_hist": ph_glob, "producao_global_paths": pp_glob,
            "producao_interna_hist": ph_int, "producao_interna_paths": pp_int,
            "price_hist": price_h, "price_paths": price_p,
            "fx_hist": fx_hist, "fx_paths": fx_paths,
            "price_unit": pu
        }

        # Salvar o PNG da cultura
        save_path = OUTPUT_DIR / f"dashboard_2024_{lbl}.png"
        plot_crop_dashboard(lbl, mc[lbl], save_path)
        print(f"Gráfico gerado: {save_path.name}")
        
    print(f"\nConcluído! Gráficos salvos em: {OUTPUT_DIR}")

if __name__=="__main__":
    run()

  Modelagem Estocástica Avançada 2026 (Produção/Demanda/Preço)
Gráfico gerado: dashboard_2024_Milho.png
Gráfico gerado: dashboard_2024_Cafe.png
Gráfico gerado: dashboard_2024_Cana.png
Gráfico gerado: dashboard_2024_Citros.png
Gráfico gerado: dashboard_2024_Trigo.png

Concluído! Gráficos salvos em: resultados_lek_modelo_novo


In [28]:
import warnings; warnings.filterwarnings("ignore")
import os, sys, subprocess, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from pathlib import Path

BASE_PATH  = Path("Base LEK.xlsx")
OUTPUT_DIR = Path("resultados_lek_modelo_novo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42)

# ── IDENTIDADE VISUAL L.E.K. ─────────────────────────────────────────────────
LEK_GREEN       = "#00A651"
LEK_DARK_GREEN  = "#004D2C"
LEK_LIGHT_GREEN = "#C8E6C9"
LEK_NAVY        = "#1A2744"
LEK_GRAY        = "#6E7B8B"
LEK_LIGHT_GRAY  = "#E8ECEF"
LEK_WHITE       = "#FFFFFF"
LEK_BG          = "#F8F9FA"

CROP_COLORS = {
    "Cafe":   "#073700",
    "Cana":   "#236E03",
    "Citros": "#57A50E",
    "Milho":  "#2fd800",
    "Trigo":  "#38ff00"
}

def get_crop_color(lbl):
    for k, v in CROP_COLORS.items():
        if k.lower() in lbl.lower(): return v
    return LEK_GREEN

COLOR_HIST  = LEK_DARK_GREEN
COLOR_MACRO = "#FF6F00"

plt.rcParams.update({
    "font.family":"DejaVu Sans","font.size":9,
    "axes.titlesize":10,"axes.labelsize":9,
    "xtick.labelsize":8,"ytick.labelsize":8,"legend.fontsize":8,
    "figure.facecolor":LEK_WHITE,"axes.facecolor":LEK_WHITE,
    "axes.edgecolor":LEK_NAVY,"axes.linewidth":1.2,
    "axes.grid":False,
    "axes.spines.top":False,"axes.spines.right":False,
    "axes.spines.left":True,"axes.spines.bottom":True,
    "text.color":LEK_NAVY,"axes.labelcolor":LEK_NAVY,
    "xtick.color":LEK_NAVY,"ytick.color":LEK_NAVY,
    "xtick.direction":"out","ytick.direction":"out",
})

PROJ_YEAR  = 2025
HORIZON    = 5
PROJ_YEARS = list(range(PROJ_YEAR+1, PROJ_YEAR+HORIZON+1))
HIST_START_PLOT = 2024   # ← todos os gráficos começam AQUI

N_SIMUL       = 10_000
HIST_START    = 2018
FALLBACK_MU   = 0.02
FALLBACK_SIGMA= 0.05
VIX_SENSITIVITY = {"Milho":0.010,"Cafe":0.012,"Cana":0.009,"Citros":0.015,"Trigo":0.010}
FED_PRICE_IMPACT = -0.015
FED_FUNDS_PROJ   = {2026:3.50,2027:3.00,2028:3.00,2029:3.00,2030:3.00}
VIX_PROJ = 18.0

# ── BASE ──────────────────────────────────────────────────────────────────────
def load_base():
    df = pd.read_excel(BASE_PATH, sheet_name=0)
    df.columns = df.columns.str.strip()
    df['ano_int'] = df['Ano'].astype(str).str.split('/').str[0].apply(pd.to_numeric, errors='coerce')
    if 'Commodities' in df.columns:
        df = df.rename(columns={'Commodities': 'cultura'})
    elif 'Commodity' in df.columns:
        df = df.rename(columns={'Commodity': 'cultura'})
    df['unidade_preco'] = "USD/t"
    return df

def get_series_by_aliases(df, pat, aliases):
    mask = df["cultura"].str.contains(pat, case=False, na=False)
    sub = df[mask]
    if sub.empty: return pd.Series(dtype=float)
    df_cols_clean = {str(c).strip().lower().replace(" ", ""): c for c in sub.columns}
    for alias in aliases:
        alias_clean = str(alias).strip().lower().replace(" ", "")
        if alias_clean in df_cols_clean:
            real_col = df_cols_clean[alias_clean]
            s = pd.to_numeric(sub.sort_values("ano_int").set_index("ano_int")[real_col], errors="coerce").dropna()
            if not s.empty:
                return s
    return pd.Series(dtype=float)

def get_macro_series(df, col):
    anos = sorted(df['ano_int'].dropna().unique())
    if not anos: anos = list(range(HIST_START, PROJ_YEAR + 1))
    mock_data = {
        "usd_brl":              np.linspace(3.5, 5.5, len(anos)),
        "us_cpi_index":         np.linspace(250, 320, len(anos)),
        "vix_avg":              np.full(len(anos), 18.0),
        "us_fed_funds_rate_pct":np.full(len(anos), 3.0)
    }
    if col in mock_data:
        s = pd.Series(mock_data[col], index=anos)
        return s.loc[s.index >= HIST_START]
    return pd.Series(dtype=float)

# ── CALIBRAÇÃO ───────────────────────────────────────────────────────────────
def fit_gbm(s):
    s = s[s.index >= HIST_START].dropna()
    if len(s) < 3: return FALLBACK_MU, FALLBACK_SIGMA
    lr = np.log(s / s.shift(1)).dropna()
    return float(lr.mean()), float(lr.std())

def fit_ou_dynamic(ps):
    s = ps[ps.index >= HIST_START].dropna()
    if len(s) < 4:
        return FALLBACK_MU, FALLBACK_SIGMA, float(s.iloc[-1]) if len(s)>0 else 100, 0.1, float(s.iloc[-1]) if len(s)>0 else 100
    y = s.diff().dropna().values
    X = s.shift(1).dropna().values
    b, a = np.polyfit(X, y, 1)
    kappa = -b
    if kappa <= 0:
        kappa = 0.05
        theta = float(s.mean())
    else:
        theta = a / kappa
    sigma = np.std(y - (a + b * X)) / float(s.mean())
    return float(s.pct_change().mean()), sigma, theta, max(kappa, 0.05), float(s.iloc[-1])

def fit_fx(s):
    s = s[s.index >= HIST_START].dropna()
    lr = np.log(s / s.shift(1)).dropna()
    return float(lr.mean()), float(lr.std()), float(s.iloc[-1])

# ── SIMULAÇÕES ────────────────────────────────────────────────────────────────
def sim_gbm_independent(s):
    s = s[s.index >= HIST_START].dropna()
    if len(s) < 3:
        return np.zeros((N_SIMUL, HORIZON)), FALLBACK_MU, FALLBACK_SIGMA
    last = float(s.iloc[-1])
    mu, sig = fit_gbm(s)
    Z = np.random.normal(0, 1, (N_SIMUL, HORIZON))
    paths = last * np.exp(np.cumsum((mu - 0.5*sig**2) + sig*Z, axis=1))
    return paths, mu, sig

def sim_price(ps, lbl, fed_last):
    mu_p, sig_p, theta, kappa, last_p = fit_ou_dynamic(ps)
    sig_adj = sig_p + VIX_SENSITIVITY.get(lbl, 0.010) * VIX_PROJ
    prices = np.full(N_SIMUL, last_p)
    paths = np.zeros((N_SIMUL, HORIZON))
    for t in range(HORIZON):
        yr = PROJ_YEAR + t + 1
        mac = FED_PRICE_IMPACT * (FED_FUNDS_PROJ.get(yr, 3.0) - fed_last)
        Z = np.random.normal(0, 1, N_SIMUL)
        prices = (prices + kappa*(theta - prices) + mac*prices + sig_adj*prices*Z)
        prices = np.clip(prices, 0.01, None)
        paths[:, t] = prices
    return paths, mu_p, sig_p, theta, kappa, sig_adj

def sim_fx(s):
    mu, sig, last = fit_fx(s)
    Z = np.random.normal(0, 1, (N_SIMUL, HORIZON))
    return last * np.exp(np.cumsum((mu - 0.5*sig**2) + sig*Z, axis=1)), mu, sig

# ── HELPERS VISUAIS ───────────────────────────────────────────────────────────
def _lek_ax(ax, title, xlabel=None, ylabel=None, subtitle=None):
    full = title + (f"\n{subtitle}" if subtitle else "")
    ax.set_title(full, fontsize=9, fontweight="bold", color=LEK_NAVY, pad=6, loc="left")
    ax.set_facecolor(LEK_WHITE)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(LEK_NAVY); ax.spines["bottom"].set_color(LEK_NAVY)
    ax.spines["left"].set_linewidth(1.2); ax.spines["bottom"].set_linewidth(1.2)
    ax.grid(False)
    if xlabel: ax.set_xlabel(xlabel, fontsize=8, color=LEK_GRAY)
    if ylabel: ax.set_ylabel(ylabel, fontsize=8, color=LEK_GRAY)

def _lek_fig(fig, title, subtitle=""):
    fig.patch.set_facecolor(LEK_WHITE)
    # Barra verde no topo
    bar = fig.add_axes([0, 0.982, 1, 0.018])
    bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])
    # Título principal logo abaixo da barra
    fig.suptitle(title, fontsize=13, fontweight="bold", color=LEK_NAVY,
                 y=0.966, x=0.02, ha="left")
    # Subtítulo com espaço suficiente abaixo do título (fonte menor, mais abaixo)
    if subtitle:
        fig.text(0.02, 0.938, subtitle, fontsize=8, color=LEK_GRAY, ha="left", va="top")

def _lek_leg(ax):
    return ax.legend(fontsize=7, framealpha=0.9, facecolor=LEK_WHITE,
                     edgecolor=LEK_LIGHT_GRAY, labelcolor=LEK_NAVY)

# ── FAN CHART (sempre de 2024 em diante) ─────────────────────────────────────
def fan(ax, hist_s, paths, color):
    proj = PROJ_YEARS

    # Filtra histórico para começar em HIST_START_PLOT (2024)
    hist_plot = hist_s[hist_s.index >= HIST_START_PLOT] if len(hist_s) > 0 else hist_s

    if len(hist_plot) > 0:
        ax.plot(list(hist_plot.index), hist_plot.values,
                color=COLOR_HIST, lw=2.2, label="Histórico (2024+)", zorder=3)
        ultimo_ano   = hist_plot.index[-1]
        ultimo_valor = hist_plot.values[-1]
    elif len(hist_s) > 0:
        # Se não há dados em 2024+, usa último ponto disponível como âncora
        ultimo_ano   = hist_s.index[-1]
        ultimo_valor = hist_s.values[-1]
    else:
        ultimo_ano   = PROJ_YEAR
        ultimo_valor = np.median(paths[:, 0]) if paths.size > 0 else 0

    n_amostras = min(150, paths.shape[0])
    if n_amostras > 0 and not np.all(paths == 0):
        amostra_idx = np.random.choice(paths.shape[0], n_amostras, replace=False)
        x_spaghetti = [ultimo_ano] + proj
        for idx in amostra_idx:
            y_spaghetti = [ultimo_valor] + list(paths[idx, :])
            ax.plot(x_spaghetti, y_spaghetti, color=color, alpha=0.15, lw=0.8, zorder=1)
        ax.plot(proj, np.median(paths, axis=0),
                color=LEK_NAVY, lw=2.5, ls="--", label="Mediana", zorder=4)
    else:
        ax.text(0.5, 0.5, 'Dados não encontrados\nna base',
                ha='center', va='center', transform=ax.transAxes, color='red')

    ax.axvline(PROJ_YEAR, color=LEK_GRAY, lw=0.8, ls=":", alpha=0.7)

    # Eixo X sempre começa em 2024
    ax.set_xlim(left=HIST_START_PLOT)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))


# ── GOOGLE SLIDES 16:9 ────────────────────────────────────────────────────────
# 33,87 × 19,05 cm @ 150 dpi → ~1994 × 1122 px
SLIDE_W_IN = 33.87 / 2.54   # 13.33"
SLIDE_H_IN = 19.05 / 2.54   # 7.50"
SLIDE_DPI  = 150

# Margem reservada para o header L.E.K. (barra verde + título + subtítulo)
HEADER_FRAC = 0.13   # 13 % do topo


def _plot_grade_5(culturas_mc, titulo, subtitulo, ylabel, get_hist, get_paths, save_path):
    """
    Gera um PNG 16:9 com 5 subplots lado a lado (1 linha × 5 colunas),
    um por cultura, e salva em save_path.

    Parâmetros
    ----------
    culturas_mc : dict  {lbl: r}
    titulo      : str   título principal (header L.E.K.)
    subtitulo   : str   subtítulo abaixo do título
    ylabel      : str   rótulo do eixo Y (igual para todos os subplots)
    get_hist    : callable(lbl, r) → pd.Series   histórico a plotar
    get_paths   : callable(lbl, r) → np.ndarray  paths simulados
    save_path   : Path
    """
    n = len(culturas_mc)   # 5
    fig, axes = plt.subplots(
        1, n,
        figsize=(SLIDE_W_IN, SLIDE_H_IN),
        sharey=False
    )
    fig.patch.set_facecolor(LEK_WHITE)

    # ── Header L.E.K. ────────────────────────────────────────────────────────
    bar = fig.add_axes([0, 1 - 0.018, 1, 0.018])
    bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])

    fig.suptitle(titulo, fontsize=14, fontweight="bold", color=LEK_NAVY,
                 y=0.975, x=0.02, ha="left")
    if subtitulo:
        fig.text(0.02, 0.945, subtitulo, fontsize=8, color=LEK_GRAY,
                 ha="left", va="top")

    # ── Subplots ──────────────────────────────────────────────────────────────
    for ax, (lbl, r) in zip(axes, culturas_mc.items()):
        c = get_crop_color(lbl)
        hist  = get_hist(lbl, r)
        paths = get_paths(lbl, r)

        fan(ax, hist, paths, c)

        # Título do subplot = nome da cultura + parâmetros relevantes
        extra = ""
        if "mu_dem_glob" in r and get_hist == _hist_dem_glob:
            extra = f"\nµ={r['mu_dem_glob']*100:+.1f}%  σ={r['sig_dem_glob']*100:.1f}%"
        elif "theta" in r and get_hist != _hist_dem_glob:
            extra = f"\nθ={r['theta']:.1f}  κ={r['kappa']:.2f}"

        ax.set_title(lbl + extra, fontsize=8, fontweight="bold",
                     color=LEK_NAVY, loc="left", pad=4)
        ax.set_facecolor(LEK_WHITE)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color(LEK_LIGHT_GRAY)
        ax.spines["bottom"].set_color(LEK_LIGHT_GRAY)
        ax.grid(False)
        ax.set_xlabel("Ano", fontsize=7, color=LEK_GRAY)
        ax.tick_params(axis="both", labelsize=7)
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=4))

        # ylabel só no primeiro subplot
        if ax is axes[0]:
            ax.set_ylabel(ylabel, fontsize=7, color=LEK_GRAY)

        # Legenda compacta só no último subplot
        if ax is axes[-1]:
            ax.legend(fontsize=6, framealpha=0.9, facecolor=LEK_WHITE,
                      edgecolor=LEK_LIGHT_GRAY, labelcolor=LEK_NAVY,
                      loc="upper left")

    plt.tight_layout(rect=[0, 0.02, 1, 1 - HEADER_FRAC])
    fig.savefig(save_path, dpi=SLIDE_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"  Salvo: {save_path.name}")


# helpers para get_hist / get_paths nas grades
def _hist_dem_glob(lbl, r): return r["demanda_global_hist"]
def _paths_dem_glob(lbl, r): return r["demanda_global_paths"]

def _hist_price_usd(lbl, r): return r["price_hist"]
def _paths_price_usd(lbl, r): return r["price_paths"]

def _hist_price_brl(lbl, r):
    ph = r["price_hist"]; fx = r["fx_hist"]
    if len(ph) > 0 and len(fx) > 0:
        idx = ph.index.intersection(fx.index)
        return (ph[idx] * fx[idx]).dropna()
    return pd.Series(dtype=float)

def _paths_price_brl(lbl, r):
    return r["price_paths"] * r["fx_paths"]


# ── PIPELINE PRINCIPAL ────────────────────────────────────────────────────────
def run():
    print("="*70)
    print("  Modelagem Estocástica – PNGs Individuais por Cultura (2024+)")
    print("="*70)

    try:
        df = load_base()
    except FileNotFoundError:
        print("ERRO: Arquivo 'Base LEK.xlsx' não encontrado.")
        return

    fx_hist  = get_macro_series(df, "usd_brl")[lambda s: s.index >= HIST_START]
    fed_hist = get_macro_series(df, "us_fed_funds_rate_pct")[lambda s: s.index >= HIST_START]
    fed_last = float(fed_hist.iloc[-1]) if len(fed_hist) > 0 else 3.0

    fx_paths, _, _ = sim_fx(fx_hist)

    CSEARCH = {
        "Milho":  "Milho",
        "Cafe":   "Caf",
        "Cana":   "Cana",
        "Citros": "Citros",
        "Trigo":  "Trigo",
    }

    mc = {}

    aliases_dem_glob  = ['Demanda(Global)(1000t)', 'DemandaGlobal']
    aliases_dem_int   = ['Demanda(Interna)(1000t)', 'DemandaInterna']
    aliases_prod_int  = ['Produção(milt)', 'ProduçãoInterna', 'ProduçãoBrasil']
    aliases_prod_glob = ['Produção(Global)(1000t)', 'ProduçãoGlobal']
    aliases_price     = ['Preçodevendadatonelada-U$D/t', 'preco_usd_unidade']
    aliases_mkt_share = ['MarketShareProdução(%)']

    for lbl, pat in CSEARCH.items():
        print(f"\n[ {lbl} ]")

        dh_glob = get_series_by_aliases(df, pat, aliases_dem_glob)
        dh_int  = get_series_by_aliases(df, pat, aliases_dem_int)
        ph_int  = get_series_by_aliases(df, pat, aliases_prod_int)
        price_h = get_series_by_aliases(df, pat, aliases_price)

        if lbl == "Cana" and dh_glob.empty:
            dh_glob = ph_int

        ph_glob = get_series_by_aliases(df, pat, aliases_prod_glob)
        if ph_glob.empty and not ph_int.empty:
            mkt_share = get_series_by_aliases(df, pat, aliases_mkt_share)
            ph_glob = ph_int / (mkt_share / 100.0) if not mkt_share.empty else dh_glob

        try:
            pu = df[df["cultura"].str.contains(pat, case=False, na=False)]["unidade_preco"].iloc[0]
        except Exception:
            pu = "USD/t"

        dp_glob, mu_dg, sig_dg = sim_gbm_independent(dh_glob)
        dp_int,  _,     _      = sim_gbm_independent(dh_int)
        pp_glob, _,     _      = sim_gbm_independent(ph_glob)
        pp_int,  _,     _      = sim_gbm_independent(ph_int)
        price_p, mu_p, sig_p, theta, kappa, sig_adj = sim_price(price_h, lbl, fed_last)

        r = {
            "demanda_global_hist":  dh_glob,
            "demanda_global_paths": dp_glob,
            "demanda_interna_hist": dh_int,
            "demanda_interna_paths": dp_int,
            "producao_global_hist": ph_glob,
            "producao_global_paths": pp_glob,
            "producao_interna_hist": ph_int,
            "producao_interna_paths": pp_int,
            "price_hist":   price_h,
            "price_paths":  price_p,
            "fx_hist":  fx_hist,
            "fx_paths": fx_paths,
            "price_unit": pu,
            # parâmetros calibrados (para títulos)
            "mu_dem_glob":  mu_dg,
            "sig_dem_glob": sig_dg,
            "theta":    theta,
            "kappa":    kappa,
            "sig_adj":  sig_adj,
        }

        # ── acumula resultados para gerar as grades depois ───────────────────
        mc[lbl] = r

    # ── Gera os 3 PNGs de grade (1 por tipo, 5 culturas cada) ────────────────
    _plot_grade_5(
        mc,
        titulo    = "Monte Carlo – Demanda Global por Cultura",
        subtitulo = f"{N_SIMUL:,} simulações | GBM Correlacionado | Horizonte {PROJ_YEARS[0]}–{PROJ_YEARS[-1]}",
        ylabel    = "1.000 MT",
        get_hist  = _hist_dem_glob,
        get_paths = _paths_dem_glob,
        save_path = OUTPUT_DIR / "grade_demanda_global.png",
    )

    _plot_grade_5(
        mc,
        titulo    = "Monte Carlo – Preço Global (USD) por Cultura",
        subtitulo = f"Ornstein-Uhlenbeck + Macro | Horizonte {PROJ_YEARS[0]}–{PROJ_YEARS[-1]}",
        ylabel    = "USD/t",
        get_hist  = _hist_price_usd,
        get_paths = _paths_price_usd,
        save_path = OUTPUT_DIR / "grade_preco_usd.png",
    )

    _plot_grade_5(
        mc,
        titulo    = "Monte Carlo – Preço Global (BRL) por Cultura",
        subtitulo = f"Preço USD × câmbio USD/BRL simulado | Horizonte {PROJ_YEARS[0]}–{PROJ_YEARS[-1]}",
        ylabel    = "R$/t",
        get_hist  = _hist_price_brl,
        get_paths = _paths_price_brl,
        save_path = OUTPUT_DIR / "grade_preco_brl.png",
    )

    print(f"\n✓ Concluído! 3 PNGs 16:9 salvos em: {OUTPUT_DIR}/")


if __name__ == "__main__":
    run()

  Modelagem Estocástica – PNGs Individuais por Cultura (2024+)

[ Milho ]

[ Cafe ]

[ Cana ]

[ Citros ]

[ Trigo ]
  Salvo: grade_demanda_global.png
  Salvo: grade_preco_usd.png
  Salvo: grade_preco_brl.png

✓ Concluído! 3 PNGs 16:9 salvos em: resultados_lek_modelo_novo/


In [29]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

BASE_PATH  = Path("Base LEK.xlsx")
OUTPUT_DIR = Path("resultados_lek_modelo_novo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42)

# ── IDENTIDADE VISUAL L.E.K. ──────────────────────────────────────────────────
LEK_GREEN      = "#00A651"
LEK_DARK_GREEN = "#004D2C"
LEK_NAVY       = "#1A2744"
LEK_GRAY       = "#6E7B8B"
LEK_WHITE      = "#FFFFFF"

CROP_COLORS = {
    "Cafe":   "#073700",
    "Cana":   "#236E03",
    "Citros": "#57A50E",
    "Milho":  "#2fd800",
    "Trigo":  "#38ff00",
}

def get_crop_color(lbl):
    for k, v in CROP_COLORS.items():
        if k.lower() in lbl.lower():
            return v
    return LEK_GREEN

COLOR_HIST = LEK_DARK_GREEN

plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "font.size":          9,
    "axes.titlesize":     10,
    "axes.labelsize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "legend.fontsize":    8,
    "figure.facecolor":   LEK_WHITE,
    "axes.facecolor":     LEK_WHITE,
    "axes.edgecolor":     LEK_NAVY,
    "axes.linewidth":     1.2,
    "axes.grid":          False,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.spines.left":   True,
    "axes.spines.bottom": True,
    "text.color":         LEK_NAVY,
    "axes.labelcolor":    LEK_NAVY,
    "xtick.color":        LEK_NAVY,
    "ytick.color":        LEK_NAVY,
    "xtick.direction":    "out",
    "ytick.direction":    "out",
})

# ── CONSTANTES ────────────────────────────────────────────────────────────────
PROJ_YEAR       = 2025
HORIZON         = 5
PROJ_YEARS      = list(range(PROJ_YEAR + 1, PROJ_YEAR + HORIZON + 1))
HIST_START_PLOT = 2024   # eixo X começa aqui
HIST_START      = 2018

N_SIMUL          = 10_000
FALLBACK_MU      = 0.02
FALLBACK_SIGMA   = 0.05
VIX_SENSITIVITY  = {"Milho":0.010,"Cafe":0.012,"Cana":0.009,"Citros":0.015,"Trigo":0.010}
FED_PRICE_IMPACT = -0.015
FED_FUNDS_PROJ   = {2026:3.50,2027:3.00,2028:3.00,2029:3.00,2030:3.00}
VIX_PROJ         = 18.0

# Google Slides 16:9
SLIDE_W_IN = 33.87 / 2.54   # 13.33"
SLIDE_H_IN = 19.05 / 2.54   #  7.50"
SLIDE_DPI  = 150

CSEARCH = {
    "Milho":  "Milho",
    "Cafe":   "Caf",
    "Cana":   "Cana",
    "Citros": "Citros",
    "Trigo":  "Trigo",
}

# ── LEITURA DA BASE ───────────────────────────────────────────────────────────
def load_base():
    df = pd.read_excel(BASE_PATH, sheet_name=0)
    df.columns = df.columns.str.strip()
    df["ano_int"] = (df["Ano"].astype(str).str.split("/").str[0]
                     .apply(pd.to_numeric, errors="coerce"))
    for col in ("Commodities", "Commodity"):
        if col in df.columns:
            df = df.rename(columns={col: "cultura"})
            break
    df["unidade_preco"] = "USD/t"
    return df

def get_series_by_aliases(df, pat, aliases):
    mask = df["cultura"].str.contains(pat, case=False, na=False)
    sub  = df[mask]
    if sub.empty:
        return pd.Series(dtype=float)
    cols_clean = {str(c).strip().lower().replace(" ", ""): c for c in sub.columns}
    for alias in aliases:
        key = str(alias).strip().lower().replace(" ", "")
        if key in cols_clean:
            s = pd.to_numeric(
                sub.sort_values("ano_int").set_index("ano_int")[cols_clean[key]],
                errors="coerce"
            ).dropna()
            if not s.empty:
                return s
    return pd.Series(dtype=float)

def get_macro_series(df, col):
    anos = sorted(df["ano_int"].dropna().unique())
    if not anos:
        anos = list(range(HIST_START, PROJ_YEAR + 1))
    mock = {
        "usd_brl":               np.linspace(3.5, 5.5, len(anos)),
        "us_cpi_index":          np.linspace(250, 320, len(anos)),
        "vix_avg":               np.full(len(anos), 18.0),
        "us_fed_funds_rate_pct": np.full(len(anos), 3.0),
    }
    if col in mock:
        s = pd.Series(mock[col], index=anos)
        return s.loc[s.index >= HIST_START]
    return pd.Series(dtype=float)

# ── CALIBRAÇÃO ────────────────────────────────────────────────────────────────
def fit_gbm(s):
    s = s[s.index >= HIST_START].dropna()
    if len(s) < 3:
        return FALLBACK_MU, FALLBACK_SIGMA
    lr = np.log(s / s.shift(1)).dropna()
    return float(lr.mean()), float(lr.std())

def fit_ou_dynamic(ps):
    s = ps[ps.index >= HIST_START].dropna()
    if len(s) < 4:
        v = float(s.iloc[-1]) if len(s) > 0 else 100
        return FALLBACK_MU, FALLBACK_SIGMA, v, 0.1, v
    y = s.diff().dropna().values
    X = s.shift(1).dropna().values
    b, a = np.polyfit(X, y, 1)
    kappa = -b
    if kappa <= 0:
        kappa, theta = 0.05, float(s.mean())
    else:
        theta = a / kappa
    sigma = np.std(y - (a + b * X)) / float(s.mean())
    return float(s.pct_change().mean()), sigma, theta, max(kappa, 0.05), float(s.iloc[-1])

def fit_fx(s):
    s  = s[s.index >= HIST_START].dropna()
    lr = np.log(s / s.shift(1)).dropna()
    return float(lr.mean()), float(lr.std()), float(s.iloc[-1])

# ── SIMULAÇÕES ────────────────────────────────────────────────────────────────
def sim_gbm_independent(s):
    s = s[s.index >= HIST_START].dropna()
    if len(s) < 3:
        return np.zeros((N_SIMUL, HORIZON)), FALLBACK_MU, FALLBACK_SIGMA
    last   = float(s.iloc[-1])
    mu, sg = fit_gbm(s)
    Z      = np.random.normal(0, 1, (N_SIMUL, HORIZON))
    paths  = last * np.exp(np.cumsum((mu - 0.5*sg**2) + sg*Z, axis=1))
    return paths, mu, sg

def sim_price(ps, lbl, fed_last):
    mu_p, sig_p, theta, kappa, last_p = fit_ou_dynamic(ps)
    sig_adj = sig_p + VIX_SENSITIVITY.get(lbl, 0.010) * VIX_PROJ
    prices  = np.full(N_SIMUL, last_p)
    paths   = np.zeros((N_SIMUL, HORIZON))
    for t in range(HORIZON):
        yr  = PROJ_YEAR + t + 1
        mac = FED_PRICE_IMPACT * (FED_FUNDS_PROJ.get(yr, 3.0) - fed_last)
        Z   = np.random.normal(0, 1, N_SIMUL)
        prices = prices + kappa*(theta - prices) + mac*prices + sig_adj*prices*Z
        prices = np.clip(prices, 0.01, None)
        paths[:, t] = prices
    return paths, mu_p, sig_p, theta, kappa, sig_adj

def sim_fx(s):
    mu, sg, last = fit_fx(s)
    Z = np.random.normal(0, 1, (N_SIMUL, HORIZON))
    return last * np.exp(np.cumsum((mu - 0.5*sg**2) + sg*Z, axis=1)), mu, sg

# ── FAN CHART ─────────────────────────────────────────────────────────────────
def fan(ax, hist_s, paths, color):
    hist_plot = hist_s[hist_s.index >= HIST_START_PLOT] if len(hist_s) > 0 else hist_s

    if len(hist_plot) > 0:
        ax.plot(list(hist_plot.index), hist_plot.values,
                color=COLOR_HIST, lw=2.2, label="Histórico", zorder=3)
        ultimo_ano, ultimo_valor = hist_plot.index[-1], hist_plot.values[-1]
    elif len(hist_s) > 0:
        ultimo_ano, ultimo_valor = hist_s.index[-1], hist_s.values[-1]
    else:
        ultimo_ano  = PROJ_YEAR
        ultimo_valor = np.median(paths[:, 0]) if paths.size > 0 else 0

    if not np.all(paths == 0):
        idx = np.random.choice(paths.shape[0], min(150, paths.shape[0]), replace=False)
        x_sp = [ultimo_ano] + PROJ_YEARS
        for i in idx:
            ax.plot(x_sp, [ultimo_valor] + list(paths[i]), color=color,
                    alpha=0.15, lw=0.8, zorder=1)
        ax.plot(PROJ_YEARS, np.median(paths, axis=0),
                color=LEK_NAVY, lw=2.5, ls="--", label="Mediana", zorder=4)
    else:
        ax.text(0.5, 0.5, "Dados não\nencontrados",
                ha="center", va="center", transform=ax.transAxes, color="red")

    ax.axvline(PROJ_YEAR, color=LEK_GRAY, lw=0.8, ls=":", alpha=0.7)
    ax.set_xlim(left=HIST_START_PLOT)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=4))

# ── ESTILO DE CADA SUBPLOT ────────────────────────────────────────────────────
def _style_ax(ax, titulo_subplot, ylabel=None, show_legend=False):
    ax.set_title(titulo_subplot, fontsize=8, fontweight="bold",
                 color=LEK_NAVY, loc="left", pad=4)
    ax.set_facecolor(LEK_WHITE)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(LEK_NAVY)
    ax.spines["left"].set_linewidth(1.1)
    ax.spines["bottom"].set_color(LEK_NAVY)
    ax.spines["bottom"].set_linewidth(1.1)
    ax.grid(False)
    ax.set_xlabel("Ano", fontsize=7, color=LEK_GRAY)
    ax.tick_params(axis="both", labelsize=7, colors=LEK_NAVY, length=3)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=7, color=LEK_GRAY)
    if show_legend:
        ax.legend(fontsize=6, framealpha=0.95, facecolor=LEK_WHITE,
                  edgecolor=LEK_GRAY, labelcolor=LEK_NAVY, loc="upper left")

# ── HEADER L.E.K. NA FIGURA ───────────────────────────────────────────────────
def _header(fig, titulo, subtitulo=""):
    fig.patch.set_facecolor(LEK_WHITE)
    bar = fig.add_axes([0, 0.982, 1, 0.018])
    bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])
    fig.suptitle(titulo, fontsize=13, fontweight="bold", color=LEK_NAVY,
                 y=0.966, x=0.02, ha="left")
    if subtitulo:
        fig.text(0.02, 0.940, subtitulo, fontsize=8, color=LEK_GRAY,
                 ha="left", va="top")

# ── FIGURA COM 5 SUBPLOTS (layout igual às imagens originais: 3 em cima, 2 embaixo)
def plot_5_culturas(mc, titulo, subtitulo, ylabel_fn, get_hist, get_paths,
                    param_fn, save_path):
    """
    mc        : dict {lbl: r}
    ylabel_fn : callable(lbl, r) -> str
    param_fn  : callable(lbl, r) -> str   parâmetros para o subtítulo do subplot
    get_hist  : callable(lbl, r) -> Series
    get_paths : callable(lbl, r) -> ndarray
    """
    lbls = list(mc.keys())   # 5 culturas

    # layout 3 + 2  (igual às figuras originais)
    fig = plt.figure(figsize=(SLIDE_W_IN, SLIDE_H_IN))
    fig.patch.set_facecolor(LEK_WHITE)

    # GridSpec: 2 linhas, 6 colunas
    # linha 0: 3 subplots (cols 0-1, 2-3, 4-5)
    # linha 1: 2 subplots centrados (cols 1-2, 3-4)
    from matplotlib.gridspec import GridSpec
    gs = GridSpec(2, 6, figure=fig,
                  left=0.06, right=0.98,
                  top=0.86, bottom=0.07,
                  hspace=0.45, wspace=0.35)

    slices_top = [gs[0, 0:2], gs[0, 2:4], gs[0, 4:6]]
    slices_bot = [gs[1, 1:3], gs[1, 3:5]]

    all_slices = slices_top + slices_bot
    axes = [fig.add_subplot(s) for s in all_slices]

    _header(fig, titulo, subtitulo)

    for ax, lbl in zip(axes, lbls):
        r     = mc[lbl]
        color = get_crop_color(lbl)
        hist  = get_hist(lbl, r)
        paths = get_paths(lbl, r)

        fan(ax, hist, paths, color)

        params = param_fn(lbl, r)
        titulo_sub = f"{lbl}\n{params}" if params else lbl
        ylabel = ylabel_fn(lbl, r) if ax is axes[0] or ax is axes[3] else None
        show_leg = (ax is axes[0])   # legenda só no primeiro

        _style_ax(ax, titulo_sub, ylabel=ylabel, show_legend=show_leg)

    fig.savefig(save_path, dpi=SLIDE_DPI, bbox_inches="tight",
                facecolor=LEK_WHITE)
    plt.close(fig)
    print(f"  Salvo: {save_path.name}")

# ── HELPERS get_hist / get_paths / ylabel / params ────────────────────────────
def _gh_dem(lbl, r):    return r["demanda_global_hist"]
def _gp_dem(lbl, r):    return r["demanda_global_paths"]
def _yl_dem(lbl, r):    return "1.000 MT"
def _pm_dem(lbl, r):    return f"µ={r['mu_dem_glob']*100:+.1f}%  σ={r['sig_dem_glob']*100:.1f}%"

def _gh_usd(lbl, r):    return r["price_hist"]
def _gp_usd(lbl, r):    return r["price_paths"]
def _yl_usd(lbl, r):    return r.get("price_unit", "USD/t")
def _pm_usd(lbl, r):    return f"θ={r['theta']:.2f}  κ={r['kappa']:.2f}  σ_adj={r['sig_adj']:.3f}"

def _gh_brl(lbl, r):
    ph, fx = r["price_hist"], r["fx_hist"]
    if len(ph) > 0 and len(fx) > 0:
        idx = ph.index.intersection(fx.index)
        return (ph[idx] * fx[idx]).dropna()
    return pd.Series(dtype=float)

def _gp_brl(lbl, r):    return r["price_paths"] * r["fx_paths"]
def _yl_brl(lbl, r):    return "R$/t"
def _pm_brl(lbl, r):    return f"θ={r['theta']:.2f}  κ={r['kappa']:.2f}  σ_adj={r['sig_adj']:.3f}"

# ── PIPELINE ──────────────────────────────────────────────────────────────────
def run():
    print("="*70)
    print("  Modelagem Estocástica L.E.K. — 3 PNGs 16:9 (2024+)")
    print("="*70)

    try:
        df = load_base()
    except FileNotFoundError:
        print("ERRO: 'Base LEK.xlsx' não encontrado.")
        return

    fx_hist  = get_macro_series(df, "usd_brl")[lambda s: s.index >= HIST_START]
    fed_hist = get_macro_series(df, "us_fed_funds_rate_pct")[lambda s: s.index >= HIST_START]
    fed_last = float(fed_hist.iloc[-1]) if len(fed_hist) > 0 else 3.0
    fx_paths, _, _ = sim_fx(fx_hist)

    aliases_dem_glob  = ["Demanda(Global)(1000t)", "DemandaGlobal"]
    aliases_dem_int   = ["Demanda(Interna)(1000t)", "DemandaInterna"]
    aliases_prod_int  = ["Produção(milt)", "ProduçãoInterna", "ProduçãoBrasil"]
    aliases_prod_glob = ["Produção(Global)(1000t)", "ProduçãoGlobal"]
    aliases_price     = ["Preçodevendadatonelada-U$D/t", "preco_usd_unidade"]
    aliases_mkt_share = ["MarketShareProdução(%)"]

    mc = {}

    for lbl, pat in CSEARCH.items():
        print(f"\n[ {lbl} ]")

        dh_glob = get_series_by_aliases(df, pat, aliases_dem_glob)
        dh_int  = get_series_by_aliases(df, pat, aliases_dem_int)
        ph_int  = get_series_by_aliases(df, pat, aliases_prod_int)
        price_h = get_series_by_aliases(df, pat, aliases_price)

        if lbl == "Cana" and dh_glob.empty:
            dh_glob = ph_int

        ph_glob = get_series_by_aliases(df, pat, aliases_prod_glob)
        if ph_glob.empty and not ph_int.empty:
            mkt = get_series_by_aliases(df, pat, aliases_mkt_share)
            ph_glob = ph_int / (mkt / 100.0) if not mkt.empty else dh_glob

        try:
            pu = df[df["cultura"].str.contains(pat, case=False, na=False)
                   ]["unidade_preco"].iloc[0]
        except Exception:
            pu = "USD/t"

        dp_glob, mu_dg, sg_dg = sim_gbm_independent(dh_glob)
        dp_int,  _,     _     = sim_gbm_independent(dh_int)
        pp_glob, _,     _     = sim_gbm_independent(ph_glob)
        pp_int,  _,     _     = sim_gbm_independent(ph_int)
        price_p, mu_p, sg_p, theta, kappa, sig_adj = sim_price(price_h, lbl, fed_last)

        mc[lbl] = {
            "demanda_global_hist":   dh_glob,
            "demanda_global_paths":  dp_glob,
            "demanda_interna_hist":  dh_int,
            "demanda_interna_paths": dp_int,
            "producao_global_hist":  ph_glob,
            "producao_global_paths": pp_glob,
            "producao_interna_hist": ph_int,
            "producao_interna_paths":pp_int,
            "price_hist":   price_h,
            "price_paths":  price_p,
            "fx_hist":      fx_hist,
            "fx_paths":     fx_paths,
            "price_unit":   pu,
            "mu_dem_glob":  mu_dg,
            "sig_dem_glob": sg_dg,
            "theta":        theta,
            "kappa":        kappa,
            "sig_adj":      sig_adj,
        }

    # ── 3 PNGs finais ─────────────────────────────────────────────────────────
    plot_5_culturas(
        mc,
        titulo    = "Monte Carlo – Demanda Mundial por Cultura",
        subtitulo = f"{N_SIMUL:,} simulações | GBM | Horizonte {PROJ_YEARS[0]}–{PROJ_YEARS[-1]}",
        ylabel_fn = _yl_dem,
        get_hist  = _gh_dem,
        get_paths = _gp_dem,
        param_fn  = _pm_dem,
        save_path = OUTPUT_DIR / "fig_demanda_global.png",
    )

    plot_5_culturas(
        mc,
        titulo    = "Monte Carlo – Preço por Cultura (Ornstein-Uhlenbeck + Macro)",
        subtitulo = "KAPPA e THETA calibrados via regressão linear dinâmica AR(1)",
        ylabel_fn = _yl_usd,
        get_hist  = _gh_usd,
        get_paths = _gp_usd,
        param_fn  = _pm_usd,
        save_path = OUTPUT_DIR / "fig_preco_usd.png",
    )

    plot_5_culturas(
        mc,
        titulo    = "Monte Carlo – Preço (BRL) por Cultura",
        subtitulo = f"Preço USD × câmbio USD/BRL simulado | Horizonte {PROJ_YEARS[0]}–{PROJ_YEARS[-1]}",
        ylabel_fn = _yl_brl,
        get_hist  = _gh_brl,
        get_paths = _gp_brl,
        param_fn  = _pm_brl,
        save_path = OUTPUT_DIR / "fig_preco_brl.png",
    )

    print(f"\n✓ Concluído! 3 PNGs 16:9 em: {OUTPUT_DIR}/")


if __name__ == "__main__":
    run()

  Modelagem Estocástica L.E.K. — 3 PNGs 16:9 (2024+)

[ Milho ]

[ Cafe ]

[ Cana ]

[ Citros ]

[ Trigo ]
  Salvo: fig_demanda_global.png
  Salvo: fig_preco_usd.png
  Salvo: fig_preco_brl.png

✓ Concluído! 3 PNGs 16:9 em: resultados_lek_modelo_novo/


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, sys, subprocess, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from pathlib import Path

BASE_PATH  = Path("Base LEK.xlsx")
OUTPUT_DIR = Path("resultados_lek_modelo_novo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(42)

# ── IDENTIDADE VISUAL L.E.K. ─────────────────────────────────────────────────
LEK_GREEN       = "#00A651"
LEK_DARK_GREEN  = "#004D2C"
LEK_LIGHT_GREEN = "#C8E6C9"
LEK_NAVY        = "#1A2744"
LEK_GRAY        = "#6E7B8B"
LEK_LIGHT_GRAY  = "#E8ECEF"
LEK_WHITE       = "#FFFFFF"
LEK_BG          = "#F8F9FA"

CROP_COLORS = {
    "Cafe":   "#073700",
    "Cana":   "#236E03",
    "Citros": "#57A50E",
    "Milho":  "#2fd800",
    "Trigo":  "#38ff00"
}

def get_crop_color(lbl):
    for k, v in CROP_COLORS.items():
        if k.lower() in lbl.lower(): return v
    return LEK_GREEN

COLOR_HIST  = LEK_DARK_GREEN
COLOR_MACRO = "#FF6F00"

plt.rcParams.update({
    "font.family":"DejaVu Sans","font.size":9,
    "axes.titlesize":10,"axes.labelsize":9,
    "xtick.labelsize":8,"ytick.labelsize":8,"legend.fontsize":8,
    "figure.facecolor":LEK_BG,"axes.facecolor":LEK_BG,
    "axes.edgecolor":LEK_LIGHT_GRAY,"axes.grid":True,
    "grid.color":LEK_LIGHT_GRAY,"grid.linewidth":0.6,
    "axes.spines.top":False,"axes.spines.right":False,
    "text.color":LEK_NAVY,"axes.labelcolor":LEK_NAVY,
    "xtick.color":LEK_GRAY,"ytick.color":LEK_GRAY,
})

PROJ_YEAR  = 2025
HORIZON    = 5
PROJ_YEARS = list(range(PROJ_YEAR+1, PROJ_YEAR+HORIZON+1))
HIST_START_PLOT = 2024   # ← todos os gráficos começam AQUI

N_SIMUL       = 10_000
HIST_START    = 2018
FALLBACK_MU   = 0.02
FALLBACK_SIGMA= 0.05
VIX_SENSITIVITY = {"Milho":0.010,"Cafe":0.012,"Cana":0.009,"Citros":0.015,"Trigo":0.010}
FED_PRICE_IMPACT = -0.015
FED_FUNDS_PROJ   = {2026:3.50,2027:3.00,2028:3.00,2029:3.00,2030:3.00}
VIX_PROJ = 18.0

# ── BASE ──────────────────────────────────────────────────────────────────────
def load_base():
    df = pd.read_excel(BASE_PATH, sheet_name=0)
    df.columns = df.columns.str.strip()
    df['ano_int'] = df['Ano'].astype(str).str.split('/').str[0].apply(pd.to_numeric, errors='coerce')
    if 'Commodities' in df.columns:
        df = df.rename(columns={'Commodities': 'cultura'})
    elif 'Commodity' in df.columns:
        df = df.rename(columns={'Commodity': 'cultura'})
    df['unidade_preco'] = "USD/t"
    return df

def get_series_by_aliases(df, pat, aliases):
    mask = df["cultura"].str.contains(pat, case=False, na=False)
    sub = df[mask]
    if sub.empty: return pd.Series(dtype=float)
    df_cols_clean = {str(c).strip().lower().replace(" ", ""): c for c in sub.columns}
    for alias in aliases:
        alias_clean = str(alias).strip().lower().replace(" ", "")
        if alias_clean in df_cols_clean:
            real_col = df_cols_clean[alias_clean]
            s = pd.to_numeric(sub.sort_values("ano_int").set_index("ano_int")[real_col], errors="coerce").dropna()
            if not s.empty:
                return s
    return pd.Series(dtype=float)

def get_macro_series(df, col):
    anos = sorted(df['ano_int'].dropna().unique())
    if not anos: anos = list(range(HIST_START, PROJ_YEAR + 1))
    mock_data = {
        "usd_brl":              np.linspace(3.5, 5.5, len(anos)),
        "us_cpi_index":         np.linspace(250, 320, len(anos)),
        "vix_avg":              np.full(len(anos), 18.0),
        "us_fed_funds_rate_pct":np.full(len(anos), 3.0)
    }
    if col in mock_data:
        s = pd.Series(mock_data[col], index=anos)
        return s.loc[s.index >= HIST_START]
    return pd.Series(dtype=float)

# ── CALIBRAÇÃO ───────────────────────────────────────────────────────────────
def fit_gbm(s):
    s = s[s.index >= HIST_START].dropna()
    if len(s) < 3: return FALLBACK_MU, FALLBACK_SIGMA
    lr = np.log(s / s.shift(1)).dropna()
    return float(lr.mean()), float(lr.std())

def fit_ou_dynamic(ps):
    s = ps[ps.index >= HIST_START].dropna()
    if len(s) < 4:
        return FALLBACK_MU, FALLBACK_SIGMA, float(s.iloc[-1]) if len(s)>0 else 100, 0.1, float(s.iloc[-1]) if len(s)>0 else 100
    y = s.diff().dropna().values
    X = s.shift(1).dropna().values
    b, a = np.polyfit(X, y, 1)
    kappa = -b
    if kappa <= 0:
        kappa = 0.05
        theta = float(s.mean())
    else:
        theta = a / kappa
    sigma = np.std(y - (a + b * X)) / float(s.mean())
    return float(s.pct_change().mean()), sigma, theta, max(kappa, 0.05), float(s.iloc[-1])

def fit_fx(s):
    s = s[s.index >= HIST_START].dropna()
    lr = np.log(s / s.shift(1)).dropna()
    return float(lr.mean()), float(lr.std()), float(s.iloc[-1])

# ── SIMULAÇÕES ────────────────────────────────────────────────────────────────
def sim_gbm_independent(s):
    s = s[s.index >= HIST_START].dropna()
    if len(s) < 3:
        return np.zeros((N_SIMUL, HORIZON)), FALLBACK_MU, FALLBACK_SIGMA
    last = float(s.iloc[-1])
    mu, sig = fit_gbm(s)
    Z = np.random.normal(0, 1, (N_SIMUL, HORIZON))
    paths = last * np.exp(np.cumsum((mu - 0.5*sig**2) + sig*Z, axis=1))
    return paths, mu, sig

def sim_price(ps, lbl, fed_last):
    mu_p, sig_p, theta, kappa, last_p = fit_ou_dynamic(ps)
    sig_adj = sig_p + VIX_SENSITIVITY.get(lbl, 0.010) * VIX_PROJ
    prices = np.full(N_SIMUL, last_p)
    paths = np.zeros((N_SIMUL, HORIZON))
    for t in range(HORIZON):
        yr = PROJ_YEAR + t + 1
        mac = FED_PRICE_IMPACT * (FED_FUNDS_PROJ.get(yr, 3.0) - fed_last)
        Z = np.random.normal(0, 1, N_SIMUL)
        prices = (prices + kappa*(theta - prices) + mac*prices + sig_adj*prices*Z)
        prices = np.clip(prices, 0.01, None)
        paths[:, t] = prices
    return paths, mu_p, sig_p, theta, kappa, sig_adj

def sim_fx(s):
    mu, sig, last = fit_fx(s)
    Z = np.random.normal(0, 1, (N_SIMUL, HORIZON))
    return last * np.exp(np.cumsum((mu - 0.5*sig**2) + sig*Z, axis=1)), mu, sig

# ── HELPERS VISUAIS ───────────────────────────────────────────────────────────
def _lek_ax(ax, title, xlabel=None, ylabel=None, subtitle=None):
    full = title + (f"\n{subtitle}" if subtitle else "")
    ax.set_title(full, fontsize=9, fontweight="bold", color=LEK_NAVY, pad=6, loc="left")
    ax.set_facecolor(LEK_BG)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(LEK_LIGHT_GRAY); ax.spines["bottom"].set_color(LEK_LIGHT_GRAY)
    ax.grid(True, color=LEK_LIGHT_GRAY, linewidth=0.6, axis="y"); ax.set_axisbelow(True)
    if xlabel: ax.set_xlabel(xlabel, fontsize=8, color=LEK_GRAY)
    if ylabel: ax.set_ylabel(ylabel, fontsize=8, color=LEK_GRAY)

def _lek_fig(fig, title, subtitle=""):
    fig.patch.set_facecolor(LEK_BG)
    # Barra verde no topo
    bar = fig.add_axes([0, 0.982, 1, 0.018])
    bar.set_facecolor(LEK_GREEN)
    for sp in bar.spines.values(): sp.set_visible(False)
    bar.set_xticks([]); bar.set_yticks([])
    # Título principal logo abaixo da barra
    fig.suptitle(title, fontsize=13, fontweight="bold", color=LEK_NAVY,
                 y=0.966, x=0.02, ha="left")
    # Subtítulo com espaço suficiente abaixo do título (fonte menor, mais abaixo)
    if subtitle:
        fig.text(0.02, 0.938, subtitle, fontsize=8, color=LEK_GRAY, ha="left", va="top")

def _lek_leg(ax):
    return ax.legend(fontsize=7, framealpha=0.9, facecolor=LEK_WHITE,
                     edgecolor=LEK_LIGHT_GRAY, labelcolor=LEK_NAVY)

# ── FAN CHART (sempre de 2024 em diante) ─────────────────────────────────────
def fan(ax, hist_s, paths, color):
    proj = PROJ_YEARS

    # Filtra histórico para começar em HIST_START_PLOT (2024)
    hist_plot = hist_s[hist_s.index >= HIST_START_PLOT] if len(hist_s) > 0 else hist_s

    if len(hist_plot) > 0:
        ax.plot(list(hist_plot.index), hist_plot.values,
                color=COLOR_HIST, lw=2.2, label="Histórico (2024+)", zorder=3)
        ultimo_ano   = hist_plot.index[-1]
        ultimo_valor = hist_plot.values[-1]
    elif len(hist_s) > 0:
        # Se não há dados em 2024+, usa último ponto disponível como âncora
        ultimo_ano   = hist_s.index[-1]
        ultimo_valor = hist_s.values[-1]
    else:
        ultimo_ano   = PROJ_YEAR
        ultimo_valor = np.median(paths[:, 0]) if paths.size > 0 else 0

    n_amostras = min(150, paths.shape[0])
    if n_amostras > 0 and not np.all(paths == 0):
        amostra_idx = np.random.choice(paths.shape[0], n_amostras, replace=False)
        x_spaghetti = [ultimo_ano] + proj
        for idx in amostra_idx:
            y_spaghetti = [ultimo_valor] + list(paths[idx, :])
            ax.plot(x_spaghetti, y_spaghetti, color=color, alpha=0.15, lw=0.8, zorder=1)
        ax.plot(proj, np.median(paths, axis=0),
                color=LEK_NAVY, lw=2.5, ls="--", label="Mediana", zorder=4)
    else:
        ax.text(0.5, 0.5, 'Dados não encontrados\nna base',
                ha='center', va='center', transform=ax.transAxes, color='red')

    ax.axvline(PROJ_YEAR, color=LEK_GRAY, lw=0.8, ls=":", alpha=0.7)

    # Eixo X sempre começa em 2024
    ax.set_xlim(left=HIST_START_PLOT)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))


# ── PLOT INDIVIDUAL: DEMANDA GLOBAL ──────────────────────────────────────────
def plot_demanda_global(lbl, r, save_path):
    """Salva um PNG só com o gráfico de Demanda Global para a cultura."""
    fig, ax = plt.subplots(figsize=(10, 6))
    _lek_fig(fig,
             f"Monte Carlo – Demanda Global : {lbl.upper()}",
             f"{N_SIMUL:,} simulações | GBM | Horizonte {PROJ_YEARS[0]}–{PROJ_YEARS[-1]}")

    c = get_crop_color(lbl)
    fan(ax, r['demanda_global_hist'], r['demanda_global_paths'], c)
    _lek_ax(ax, f"Demanda Global – {lbl}", "Ano", "1.000 MT",
            subtitle=f"µ={r['mu_dem_glob']*100:+.1f}%  σ={r['sig_dem_glob']*100:.1f}%")
    _lek_leg(ax)

    plt.tight_layout(rect=[0, 0.02, 1, 0.90])
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Salvo: {save_path.name}")


# ── PLOT INDIVIDUAL: PREÇO (USD) ──────────────────────────────────────────────
def plot_preco_usd(lbl, r, save_path):
    """Salva um PNG só com o gráfico de Preço Global USD para a cultura."""
    fig, ax = plt.subplots(figsize=(10, 6))
    _lek_fig(fig,
             f"Monte Carlo – Preço Global (USD) : {lbl.upper()}",
             f"Ornstein-Uhlenbeck + Macro | θ={r['theta']:.2f}  κ={r['kappa']:.2f}  σ_adj={r['sig_adj']:.3f}")

    c = get_crop_color(lbl)
    fan(ax, r['price_hist'], r['price_paths'], c)
    _lek_ax(ax, f"Preço Global – {lbl}", "Ano",
            r.get('price_unit', 'USD/t'),
            subtitle=f"θ={r['theta']:.2f}  κ={r['kappa']:.2f}  σ_adj={r['sig_adj']:.3f}")
    _lek_leg(ax)

    plt.tight_layout(rect=[0, 0.02, 1, 0.90])
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Salvo: {save_path.name}")


# ── PLOT INDIVIDUAL: PREÇO (BRL) ──────────────────────────────────────────────
def plot_preco_brl(lbl, r, save_path):
    """Salva um PNG só com o gráfico de Preço Global BRL para a cultura."""
    price_brl_paths = r['price_paths'] * r['fx_paths']

    if len(r['price_hist']) > 0 and len(r['fx_hist']) > 0:
        common_idx = r['price_hist'].index.intersection(r['fx_hist'].index)
        hist_brl = r['price_hist'][common_idx] * r['fx_hist'][common_idx]
    else:
        hist_brl = pd.Series(dtype=float)

    fig, ax = plt.subplots(figsize=(10, 6))
    _lek_fig(fig,
             f"Monte Carlo – Preço Global (BRL) : {lbl.upper()}",
             f"Preço USD × câmbio USD/BRL simulado | Horizonte {PROJ_YEARS[0]}–{PROJ_YEARS[-1]}")

    c = get_crop_color(lbl)
    fan(ax, hist_brl.dropna(), price_brl_paths, c)
    _lek_ax(ax, f"Preço Global BRL – {lbl}", "Ano", "R$/t")
    _lek_leg(ax)

    plt.tight_layout(rect=[0, 0.02, 1, 0.90])
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Salvo: {save_path.name}")


# ── PIPELINE PRINCIPAL ────────────────────────────────────────────────────────
def run():
    print("="*70)
    print("  Modelagem Estocástica – PNGs Individuais por Cultura (2024+)")
    print("="*70)

    try:
        df = load_base()
    except FileNotFoundError:
        print("ERRO: Arquivo 'Base LEK.xlsx' não encontrado.")
        return

    fx_hist  = get_macro_series(df, "usd_brl")[lambda s: s.index >= HIST_START]
    fed_hist = get_macro_series(df, "us_fed_funds_rate_pct")[lambda s: s.index >= HIST_START]
    fed_last = float(fed_hist.iloc[-1]) if len(fed_hist) > 0 else 3.0

    fx_paths, _, _ = sim_fx(fx_hist)

    CSEARCH = {
        "Milho":  "Milho",
        "Cafe":   "Caf",
        "Cana":   "Cana",
        "Citros": "Citros",
        "Trigo":  "Trigo",
    }

    aliases_dem_glob  = ['Demanda(Global)(1000t)', 'DemandaGlobal']
    aliases_dem_int   = ['Demanda(Interna)(1000t)', 'DemandaInterna']
    aliases_prod_int  = ['Produção(milt)', 'ProduçãoInterna', 'ProduçãoBrasil']
    aliases_prod_glob = ['Produção(Global)(1000t)', 'ProduçãoGlobal']
    aliases_price     = ['Preçodevendadatonelada-U$D/t', 'preco_usd_unidade']
    aliases_mkt_share = ['MarketShareProdução(%)']

    for lbl, pat in CSEARCH.items():
        print(f"\n[ {lbl} ]")

        dh_glob = get_series_by_aliases(df, pat, aliases_dem_glob)
        dh_int  = get_series_by_aliases(df, pat, aliases_dem_int)
        ph_int  = get_series_by_aliases(df, pat, aliases_prod_int)
        price_h = get_series_by_aliases(df, pat, aliases_price)

        if lbl == "Cana" and dh_glob.empty:
            dh_glob = ph_int

        ph_glob = get_series_by_aliases(df, pat, aliases_prod_glob)
        if ph_glob.empty and not ph_int.empty:
            mkt_share = get_series_by_aliases(df, pat, aliases_mkt_share)
            ph_glob = ph_int / (mkt_share / 100.0) if not mkt_share.empty else dh_glob

        try:
            pu = df[df["cultura"].str.contains(pat, case=False, na=False)]["unidade_preco"].iloc[0]
        except Exception:
            pu = "USD/t"

        dp_glob, mu_dg, sig_dg = sim_gbm_independent(dh_glob)
        dp_int,  _,     _      = sim_gbm_independent(dh_int)
        pp_glob, _,     _      = sim_gbm_independent(ph_glob)
        pp_int,  _,     _      = sim_gbm_independent(ph_int)
        price_p, mu_p, sig_p, theta, kappa, sig_adj = sim_price(price_h, lbl, fed_last)

        r = {
            "demanda_global_hist":  dh_glob,
            "demanda_global_paths": dp_glob,
            "demanda_interna_hist": dh_int,
            "demanda_interna_paths": dp_int,
            "producao_global_hist": ph_glob,
            "producao_global_paths": pp_glob,
            "producao_interna_hist": ph_int,
            "producao_interna_paths": pp_int,
            "price_hist":   price_h,
            "price_paths":  price_p,
            "fx_hist":  fx_hist,
            "fx_paths": fx_paths,
            "price_unit": pu,
            # parâmetros calibrados (para títulos)
            "mu_dem_glob":  mu_dg,
            "sig_dem_glob": sig_dg,
            "theta":    theta,
            "kappa":    kappa,
            "sig_adj":  sig_adj,
        }

        # ── Salva cada gráfico em PNG próprio ────────────────────────────────
        plot_demanda_global(
            lbl, r,
            OUTPUT_DIR / f"demanda_global_{lbl}.png"
        )

        plot_preco_usd(
            lbl, r,
            OUTPUT_DIR / f"preco_usd_{lbl}.png"
        )

        plot_preco_brl(
            lbl, r,
            OUTPUT_DIR / f"preco_brl_{lbl}.png"
        )

    print(f"\n✓ Concluído! Todos os PNGs salvos em: {OUTPUT_DIR}/")


if __name__ == "__main__":
    run()